In [1]:
from pathlib import Path
import zipfile

PROJECT_DIR = Path(r"E:\Governance-Policy\Cogent Economics & Finance")
ZIP_PATH = PROJECT_DIR / "wb3ndt69gf-3.zip"
OUTPUT_DIR = PROJECT_DIR / "revision_outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Project folder:", PROJECT_DIR.exists())
print("Dataset ZIP:", ZIP_PATH.exists())
print("Output folder:", OUTPUT_DIR.exists())

print("\nFILES INSIDE ZIP:")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for f in z.infolist():
        print(f"{f.filename:70s} {f.file_size / 1024**2:10.2f} MB")

Project folder: True
Dataset ZIP: True
Output folder: True

FILES INSIDE ZIP:
Labels.do                                                                    0.01 MB
P2P_Macro_Codes.do                                                           0.00 MB
P2P_Macro_Data.zip                                                         548.43 MB
P2P_Macro_Data.dta                                                        2881.70 MB


In [2]:
# ============================================================
# CELL 2 — Inspect original Stata definitions
# Reviewer-analysis audit only
# ============================================================

import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    for filename in ["Labels.do", "P2P_Macro_Codes.do"]:

        print("\n" + "=" * 100)
        print(filename)
        print("=" * 100)

        raw = z.read(filename)

        text = None
        for encoding in ["utf-8", "cp1252", "latin-1"]:
            try:
                text = raw.decode(encoding)
                break
            except UnicodeDecodeError:
                continue

        print(text)


Labels.do
label variable	acc_now_delinq	"The number of accounts on which the borrower is now delinquent."
label variable	acc_open_past_24mths	"Number of trades opened in past 24 months."
label variable	addr_state	"The state provided by the borrower in the loan application"
label variable	all_util	"Balance to credit limit on all trades"
label variable	annual_inc	"The self-reported annual income provided by the borrower during registration."
label variable	annual_inc_joint	"The combined self-reported annual income provided by the co-borrowers during registration"
label variable	application_type	"Indicates whether the loan is an individual application or a joint application with two co-borrowers"
label variable	avg_cur_bal	"Average current balance of all accounts"
label variable	bc_open_to_buy	"Total open to buy on revolving bankcards."
label variable	bc_util	"Ratio of total current balance to high credit/credit limit for all bankcard accounts."
label variable	chargeoff_within_12_mths	"N

In [5]:
# ============================================================
# CELL 3C — Get actual column names safely
# Reads ONLY ONE ROW from the 2.81 GB Stata file
# ============================================================

import pandas as pd

print("Reading one row only...")

reader = pd.read_stata(
    DTA_PATH,
    convert_categoricals=False,
    iterator=True
)

one_row = reader.read(1)

variables = one_row.columns.tolist()

print("Successfully opened dataset.")
print("Number of variables:", len(variables))

print("\n" + "=" * 100)
print("ALL VARIABLE NAMES")
print("=" * 100)

for i, v in enumerate(variables, 1):
    print(f"{i:3d}. {v}")

# Reviewer-relevant variables
keywords = [
    "loan_status",
    "badloan",
    "issue",
    "last_pymnt",
    "next_pymnt",
    "term",
    "unemployment",
    "state",
    "addr_state",
    "baseline",
    "default",
    "charge",
    "paid",
    "recover",
    "out_prncp",
    "total_rec_prncp",
    "hardship",
    "settlement"
]

relevant = [
    v for v in variables
    if any(k.lower() in v.lower() for k in keywords)
]

print("\n" + "=" * 100)
print("REVIEWER-RELEVANT VARIABLES FOUND")
print("=" * 100)

for v in relevant:
    print(v)

reader.close()

del one_row

Reading one row only...
Successfully opened dataset.
Number of variables: 195

ALL VARIABLE NAMES
  1. loan_amnt
  2. funded_amnt
  3. funded_amnt_inv
  4. term
  5. int_rate
  6. installment
  7. grade
  8. sub_grade
  9. emp_title
 10. emp_length
 11. home_ownership
 12. annual_inc
 13. verification_status
 14. issue_d
 15. loan_status
 16. pymnt_plan
 17. url
 18. desc
 19. purpose
 20. title
 21. zip_code
 22. addr_state
 23. dti
 24. delinq_2yrs
 25. earliest_cr_line
 26. inq_last_6mths
 27. mths_since_last_delinq
 28. mths_since_last_record
 29. open_acc
 30. pub_rec
 31. revol_bal
 32. revol_util
 33. total_acc
 34. initial_list_status
 35. out_prncp
 36. out_prncp_inv
 37. total_pymnt
 38. total_pymnt_inv
 39. total_rec_prncp
 40. total_rec_int
 41. total_rec_late_fee
 42. recoveries
 43. collection_recovery_fee
 44. last_pymnt_d
 45. last_pymnt_amnt
 46. next_pymnt_d
 47. last_credit_pull_d
 48. collections_12_mths_ex_med
 49. mths_since_last_major_derog
 50. policy_code
 51. 

C:\Users\sufi\AppData\Local\Temp\ipykernel_15092\1163112196.py:64: FutureWarning: The StataReader.close() method is not part of the public API and will be removed in a future version without notice. Using StataReader as a context manager is the only supported method.
  reader.close()


In [6]:
# ============================================================
# CELL 4 — Outcome/status audit
# Reviewer comments:
#   - clarify loan maturity
#   - clarify default classification
#   - assess right censoring
#   - identify early repayment / prepayment
#
# Reads ONLY reviewer-relevant columns in chunks.
# ============================================================

import pandas as pd
import numpy as np

USECOLS = [
    "loan_status",
    "issue_d",
    "last_pymnt_d",
    "term",
    "badloan",
    "out_prncp"
]

CHUNK_SIZE = 200_000

status_counts = {}
badloan_counts = {}
status_badloan_counts = {}

n_total = 0
missing_issue = 0
missing_last_payment = 0
missing_term = 0

issue_min = None
issue_max = None

print("Scanning reviewer-relevant outcome fields...")
print("No modelling is being performed in this cell.\n")

with pd.read_stata(
    DTA_PATH,
    columns=USECOLS,
    convert_categoricals=False,
    chunksize=CHUNK_SIZE
) as reader:

    for chunk_no, df in enumerate(reader, start=1):

        n_total += len(df)

        # ---------------------------------------------
        # Loan status counts
        # ---------------------------------------------
        sc = (
            df["loan_status"]
            .fillna("<MISSING>")
            .astype(str)
            .value_counts(dropna=False)
        )

        for k, v in sc.items():
            status_counts[k] = status_counts.get(k, 0) + int(v)

        # ---------------------------------------------
        # Existing badloan counts
        # ---------------------------------------------
        bc = (
            df["badloan"]
            .fillna(-999)
            .value_counts(dropna=False)
        )

        for k, v in bc.items():
            badloan_counts[k] = badloan_counts.get(k, 0) + int(v)

        # ---------------------------------------------
        # Cross-tab: loan_status × existing badloan
        # ---------------------------------------------
        tmp = pd.crosstab(
            df["loan_status"].fillna("<MISSING>").astype(str),
            df["badloan"].fillna(-999)
        )

        for status in tmp.index:
            for bad in tmp.columns:
                key = (status, bad)
                status_badloan_counts[key] = (
                    status_badloan_counts.get(key, 0)
                    + int(tmp.loc[status, bad])
                )

        # ---------------------------------------------
        # Missingness
        # ---------------------------------------------
        missing_issue += int(df["issue_d"].isna().sum())
        missing_last_payment += int(df["last_pymnt_d"].isna().sum())
        missing_term += int(df["term"].isna().sum())

        # ---------------------------------------------
        # Origination-date range
        # ---------------------------------------------
        issue_parsed = pd.to_datetime(
            df["issue_d"],
            errors="coerce"
        )

        cmin = issue_parsed.min()
        cmax = issue_parsed.max()

        if pd.notna(cmin):
            issue_min = cmin if issue_min is None else min(issue_min, cmin)

        if pd.notna(cmax):
            issue_max = cmax if issue_max is None else max(issue_max, cmax)

        print(
            f"Chunk {chunk_no:02d} | "
            f"rows processed: {n_total:,}"
        )

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 100)
print("A. TOTAL OBSERVATIONS")
print("=" * 100)
print(f"{n_total:,}")

print("\n" + "=" * 100)
print("B. LOAN STATUS COUNTS")
print("=" * 100)

status_table = (
    pd.Series(status_counts, name="n")
    .sort_values(ascending=False)
    .to_frame()
)

status_table["percent"] = (
    100 * status_table["n"] / n_total
)

print(status_table.to_string())

print("\n" + "=" * 100)
print("C. EXISTING badloan COUNTS")
print("=" * 100)

badloan_table = (
    pd.Series(badloan_counts, name="n")
    .sort_index()
    .to_frame()
)

badloan_table["percent"] = (
    100 * badloan_table["n"] / n_total
)

print(badloan_table.to_string())

print("\n" + "=" * 100)
print("D. loan_status × EXISTING badloan")
print("=" * 100)

cross_rows = []

for (status, bad), n in status_badloan_counts.items():
    cross_rows.append({
        "loan_status": status,
        "badloan": bad,
        "n": n
    })

cross_table = pd.DataFrame(cross_rows)

cross_table = cross_table.sort_values(
    ["loan_status", "badloan"]
)

print(cross_table.to_string(index=False))

print("\n" + "=" * 100)
print("E. DATE / FOLLOW-UP FIELD AVAILABILITY")
print("=" * 100)

print("Origination range:", issue_min, "to", issue_max)

print(
    f"Missing issue_d: "
    f"{missing_issue:,} "
    f"({100*missing_issue/n_total:.4f}%)"
)

print(
    f"Missing last_pymnt_d: "
    f"{missing_last_payment:,} "
    f"({100*missing_last_payment/n_total:.4f}%)"
)

print(
    f"Missing term: "
    f"{missing_term:,} "
    f"({100*missing_term/n_total:.4f}%)"
)

# Save audit tables
status_table.to_csv(
    OUTPUT_DIR / "01_loan_status_counts.csv"
)

badloan_table.to_csv(
    OUTPUT_DIR / "02_badloan_counts.csv"
)

cross_table.to_csv(
    OUTPUT_DIR / "03_status_by_badloan.csv",
    index=False
)

print("\nAudit tables saved in:")
print(OUTPUT_DIR)

Scanning reviewer-relevant outcome fields...
No modelling is being performed in this cell.



C:\Users\sufi\AppData\Local\Temp\ipykernel_15092\571830959.py:103: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  issue_parsed = pd.to_datetime(


Chunk 01 | rows processed: 200,000


C:\Users\sufi\AppData\Local\Temp\ipykernel_15092\571830959.py:103: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  issue_parsed = pd.to_datetime(


Chunk 02 | rows processed: 400,000


C:\Users\sufi\AppData\Local\Temp\ipykernel_15092\571830959.py:103: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  issue_parsed = pd.to_datetime(


... (18 outputs left out)

Chunk 12 | rows processed: 2,400,000


C:\Users\sufi\AppData\Local\Temp\ipykernel_15092\571830959.py:103: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  issue_parsed = pd.to_datetime(


Chunk 13 | rows processed: 2,600,000
Chunk 14 | rows processed: 2,703,430

A. TOTAL OBSERVATIONS
2,703,430

B. LOAN STATUS COUNTS
                          n    percent
Current             1704854  63.062628
Fully Paid           765339  28.309925
Charged Off          177884   6.579937
Late (31-120 days)    31106   1.150612
In Grace Period       16615   0.614590
Late (16-30 days)      7510   0.277795
Default                 122   0.004513

C. EXISTING badloan COUNTS
           n    percent
0.0  2486808  91.987142
1.0   216622   8.012858

D. loan_status × EXISTING badloan
       loan_status  badloan       n
       Charged Off      0.0       0
       Charged Off      1.0  177884
           Current      0.0 1704854
           Current      1.0       0
           Default      0.0       0
           Default      1.0     122
        Fully Paid      0.0  765339
        Fully Paid      1.0       0
   In Grace Period      0.0   16615
   In Grace Period      1.0       0
 Late (16-30 days)      0.0

C:\Users\sufi\AppData\Local\Temp\ipykernel_15092\571830959.py:103: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  issue_parsed = pd.to_datetime(


In [7]:
# ============================================================
# CELL 5 — Inspect actual date representations
# DO NOT perform survival analysis yet
# ============================================================

import pandas as pd

DATE_COLS = [
    "issue_d",
    "issue_date",
    "last_pymnt_d",
    "last_date",
    "term",
    "loan_status",
    "next_pymnt_d"
]

sample = pd.read_stata(
    DTA_PATH,
    columns=DATE_COLS,
    convert_categoricals=False,
    nrows=30
)

print("=" * 100)
print("A. DATA TYPES")
print("=" * 100)

print(sample.dtypes)

print("\n" + "=" * 100)
print("B. FIRST 30 RAW RECORDS")
print("=" * 100)

print(sample.to_string(index=False))

print("\n" + "=" * 100)
print("C. UNIQUE SAMPLE VALUES")
print("=" * 100)

for col in [
    "issue_d",
    "issue_date",
    "last_pymnt_d",
    "last_date",
    "term",
    "next_pymnt_d"
]:
    print(f"\n--- {col} ---")
    print(sample[col].unique()[:30])

TypeError: read_stata() got an unexpected keyword argument 'nrows'

In [8]:
# ============================================================
# CELL 5B — Inspect actual date representations safely
# Reads ONLY 30 rows using the verified iterator method
# ============================================================

import pandas as pd

DATE_COLS = [
    "issue_d",
    "issue_date",
    "last_pymnt_d",
    "last_date",
    "term",
    "loan_status",
    "next_pymnt_d"
]

print("Reading 30 rows only...\n")

with pd.read_stata(
    DTA_PATH,
    columns=DATE_COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    sample = reader.read(30)

print("=" * 100)
print("A. DATA TYPES")
print("=" * 100)

print(sample.dtypes)

print("\n" + "=" * 100)
print("B. FIRST 30 RAW RECORDS")
print("=" * 100)

print(sample.to_string(index=False))

print("\n" + "=" * 100)
print("C. UNIQUE SAMPLE VALUES")
print("=" * 100)

for col in [
    "issue_d",
    "issue_date",
    "last_pymnt_d",
    "last_date",
    "term",
    "next_pymnt_d"
]:
    print(f"\n--- {col} ---")
    print(sample[col].unique()[:30])

print("\n" + "=" * 100)
print("D. PYTHON VALUE TYPES")
print("=" * 100)

for col in [
    "issue_d",
    "issue_date",
    "last_pymnt_d",
    "last_date",
    "term",
    "next_pymnt_d"
]:
    non_missing = sample[col].dropna()

    if len(non_missing) > 0:
        value = non_missing.iloc[0]
        print(
            f"{col:15s} | "
            f"value = {repr(value)} | "
            f"type = {type(value)}"
        )
    else:
        print(f"{col:15s} | all values missing in sample")

Reading 30 rows only...

A. DATA TYPES
issue_d                 object
issue_date      datetime64[ns]
last_pymnt_d            object
last_date       datetime64[ns]
term                    object
loan_status             object
next_pymnt_d            object
dtype: object

B. FIRST 30 RAW RECORDS
issue_d issue_date last_pymnt_d  last_date       term loan_status next_pymnt_d
 Aug-12 2012-08-01       Sep-15 2015-09-01  36 months  Fully Paid             
 Aug-12 2012-08-01       Sep-15 2015-09-01  36 months  Fully Paid             
 Sep-12 2012-09-01       Sep-15 2015-09-01  36 months  Fully Paid             
 Sep-12 2012-09-01       Jan-14 2014-01-01  36 months Charged Off             
 Sep-12 2012-09-01       Jun-14 2014-06-01  36 months Charged Off             
 Sep-12 2012-09-01       Sep-15 2015-09-01  36 months  Fully Paid             
 Sep-12 2012-09-01       Dec-14 2014-12-01  36 months Charged Off             
 Sep-12 2012-09-01       Jul-14 2014-07-01  36 months  Fully Paid        

In [9]:
# ============================================================
# CELL 6 — Establish observation horizon and status-date structure
#
# Purpose:
#   1. Determine latest dates actually represented in dataset
#   2. Examine last-payment timing by loan status
#   3. Examine next-payment dates for active/non-terminal loans
#   4. Check whether constructed last_date differs from raw
#      last_pymnt_d because of the original Stata replacement
#
# No survival modelling yet.
# ============================================================

import pandas as pd
import numpy as np

USECOLS = [
    "issue_date",
    "last_pymnt_d",
    "last_date",
    "next_pymnt_d",
    "loan_status",
    "term"
]

CHUNK_SIZE = 200_000

summary = {}

for status in [
    "Fully Paid",
    "Charged Off",
    "Default",
    "Current",
    "Late (16-30 days)",
    "Late (31-120 days)",
    "In Grace Period"
]:
    summary[status] = {
        "n": 0,
        "issue_min": None,
        "issue_max": None,
        "last_min": None,
        "last_max": None,
        "raw_last_blank": 0,
        "next_nonblank": 0,
        "next_min": None,
        "next_max": None
    }

overall_issue_min = None
overall_issue_max = None
overall_last_min = None
overall_last_max = None
overall_next_min = None
overall_next_max = None

raw_last_blank_total = 0

print("Scanning date structure...\n")

with pd.read_stata(
    DTA_PATH,
    columns=USECOLS,
    convert_categoricals=False,
    chunksize=CHUNK_SIZE
) as reader:

    for chunk_no, df in enumerate(reader, start=1):

        # ----------------------------------------------------
        # Raw string cleaning
        # ----------------------------------------------------
        raw_last = (
            df["last_pymnt_d"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        raw_next = (
            df["next_pymnt_d"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        raw_last_blank = raw_last.eq("")
        raw_next_nonblank = raw_next.ne("")

        raw_last_blank_total += int(raw_last_blank.sum())

        # ----------------------------------------------------
        # Parse next payment date explicitly
        # Example: May-17
        # ----------------------------------------------------
        next_dt = pd.to_datetime(
            raw_next.where(raw_next_nonblank),
            format="%b-%y",
            errors="coerce"
        )

        # ----------------------------------------------------
        # Overall date ranges
        # ----------------------------------------------------
        for series, name in [
            (df["issue_date"], "issue"),
            (df["last_date"], "last"),
            (next_dt, "next")
        ]:
            cmin = series.min()
            cmax = series.max()

            if pd.notna(cmin):
                if name == "issue":
                    overall_issue_min = (
                        cmin if overall_issue_min is None
                        else min(overall_issue_min, cmin)
                    )
                elif name == "last":
                    overall_last_min = (
                        cmin if overall_last_min is None
                        else min(overall_last_min, cmin)
                    )
                else:
                    overall_next_min = (
                        cmin if overall_next_min is None
                        else min(overall_next_min, cmin)
                    )

            if pd.notna(cmax):
                if name == "issue":
                    overall_issue_max = (
                        cmax if overall_issue_max is None
                        else max(overall_issue_max, cmax)
                    )
                elif name == "last":
                    overall_last_max = (
                        cmax if overall_last_max is None
                        else max(overall_last_max, cmax)
                    )
                else:
                    overall_next_max = (
                        cmax if overall_next_max is None
                        else max(overall_next_max, cmax)
                    )

        # ----------------------------------------------------
        # Status-specific audit
        # ----------------------------------------------------
        for status in summary:

            mask = df["loan_status"].astype(str).eq(status)

            if not mask.any():
                continue

            s = summary[status]

            s["n"] += int(mask.sum())
            s["raw_last_blank"] += int(raw_last_blank[mask].sum())
            s["next_nonblank"] += int(raw_next_nonblank[mask].sum())

            issue_vals = df.loc[mask, "issue_date"]
            last_vals = df.loc[mask, "last_date"]
            next_vals = next_dt[mask]

            for vals, min_key, max_key in [
                (issue_vals, "issue_min", "issue_max"),
                (last_vals, "last_min", "last_max"),
                (next_vals, "next_min", "next_max")
            ]:

                vmin = vals.min()
                vmax = vals.max()

                if pd.notna(vmin):
                    s[min_key] = (
                        vmin if s[min_key] is None
                        else min(s[min_key], vmin)
                    )

                if pd.notna(vmax):
                    s[max_key] = (
                        vmax if s[max_key] is None
                        else max(s[max_key], vmax)
                    )

        print(f"Chunk {chunk_no:02d} processed")

# ============================================================
# Display results
# ============================================================

print("\n" + "=" * 100)
print("A. OVERALL DATE RANGES")
print("=" * 100)

print("Issue-date range:       ", overall_issue_min, "to", overall_issue_max)
print("Constructed last_date:  ", overall_last_min, "to", overall_last_max)
print("Next-payment range:     ", overall_next_min, "to", overall_next_max)
print("Blank raw last_pymnt_d: ", f"{raw_last_blank_total:,}")

print("\n" + "=" * 100)
print("B. STATUS-SPECIFIC DATE STRUCTURE")
print("=" * 100)

rows = []

for status, s in summary.items():

    rows.append({
        "loan_status": status,
        "n": s["n"],
        "issue_min": s["issue_min"],
        "issue_max": s["issue_max"],
        "last_min": s["last_min"],
        "last_max": s["last_max"],
        "raw_last_blank": s["raw_last_blank"],
        "next_nonblank": s["next_nonblank"],
        "next_min": s["next_min"],
        "next_max": s["next_max"]
    })

date_audit = pd.DataFrame(rows)

print(date_audit.to_string(index=False))

print("\n" + "=" * 100)
print("C. PROPORTIONS BY STATUS")
print("=" * 100)

date_audit["raw_last_blank_pct"] = np.where(
    date_audit["n"] > 0,
    100 * date_audit["raw_last_blank"] / date_audit["n"],
    np.nan
)

date_audit["next_nonblank_pct"] = np.where(
    date_audit["n"] > 0,
    100 * date_audit["next_nonblank"] / date_audit["n"],
    np.nan
)

print(
    date_audit[
        [
            "loan_status",
            "n",
            "raw_last_blank_pct",
            "next_nonblank_pct"
        ]
    ].to_string(index=False)
)

date_audit.to_csv(
    OUTPUT_DIR / "04_status_date_audit.csv",
    index=False
)

print("\nSaved:")
print(OUTPUT_DIR / "04_status_date_audit.csv")

Scanning date structure...

Chunk 01 processed
Chunk 02 processed
Chunk 03 processed
Chunk 04 processed
Chunk 05 processed
Chunk 06 processed
Chunk 07 processed
Chunk 08 processed
Chunk 09 processed
Chunk 10 processed
Chunk 11 processed
Chunk 12 processed
Chunk 13 processed
Chunk 14 processed

A. OVERALL DATE RANGES
Issue-date range:        2012-08-01 00:00:00 to 2019-12-01 00:00:00
Constructed last_date:   2012-08-01 00:00:00 to 2020-07-01 00:00:00
Next-payment range:      2017-04-01 00:00:00 to 2020-10-01 00:00:00
Blank raw last_pymnt_d:  3,167

B. STATUS-SPECIFIC DATE STRUCTURE
       loan_status       n  issue_min  issue_max   last_min   last_max  raw_last_blank  next_nonblank   next_min   next_max
        Fully Paid  765339 2012-08-01 2019-12-01 2012-09-01 2020-06-01               0              0        NaT        NaT
       Charged Off  177884 2012-08-01 2019-12-01 2012-08-01 2020-04-01            2740              0        NaT        NaT
           Default     122 2013-06-01 20

In [10]:
# ============================================================
# CELL 7 — Reconstruct manuscript outcome-maturity cohort
#
# Goal:
#   Test whether the stated maturity rule reproduces:
#       Train:      875,616
#       Validation:  27,374
#       Test:        41,145
#
# Manuscript chronological split:
#   Train: Aug 2012 - Jun 2018
#   Validation: Jul 2018 - Mar 2019
#   Test: Apr 2019 - Dec 2019
#
# Diagnostic maturity rule:
#   terminal resolution OR enough follow-up for contractual term
#
# IMPORTANT:
#   This is a reconstruction/audit, NOT yet the final
#   survival/competing-risk outcome definition.
# ============================================================

import pandas as pd
import numpy as np

USECOLS = [
    "issue_date",
    "loan_status",
    "term",
    "badloan"
]

CHUNK_SIZE = 200_000

# ------------------------------------------------------------
# Candidate administrative cutoff dates
#
# We do NOT assume which one was used.
# We test plausible month-level cutoffs against the published
# mature-cohort counts.
# ------------------------------------------------------------

cutoffs = pd.date_range(
    "2020-04-01",
    "2020-10-01",
    freq="MS"
)

results = []

print("Testing maturity reconstruction...\n")

with pd.read_stata(
    DTA_PATH,
    columns=USECOLS,
    convert_categoricals=False,
    chunksize=CHUNK_SIZE
) as reader:

    for chunk_no, df in enumerate(reader, start=1):

        issue = pd.to_datetime(
            df["issue_date"],
            errors="coerce"
        )

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        # Extract contractual term as numeric months
        term_months = pd.to_numeric(
            df["term"]
            .fillna("")
            .astype(str)
            .str.extract(r"(\d+)", expand=False),
            errors="coerce"
        )

        # ----------------------------------------------------
        # Chronological manuscript split
        # ----------------------------------------------------

        split = np.select(
            [
                (issue >= pd.Timestamp("2012-08-01")) &
                (issue <= pd.Timestamp("2018-06-01")),

                (issue >= pd.Timestamp("2018-07-01")) &
                (issue <= pd.Timestamp("2019-03-01")),

                (issue >= pd.Timestamp("2019-04-01")) &
                (issue <= pd.Timestamp("2019-12-01"))
            ],
            [
                "Train",
                "Validation",
                "Test"
            ],
            default="Outside"
        )

        # ----------------------------------------------------
        # Terminal resolution candidate
        #
        # Fully Paid = competing terminal resolution
        # Charged Off = terminal adverse resolution
        #
        # Default is deliberately NOT called terminal here
        # because Cell 6 showed all Default loans have a
        # next scheduled payment date.
        # ----------------------------------------------------

        terminal = status.isin([
            "Fully Paid",
            "Charged Off"
        ])

        # ----------------------------------------------------
        # Evaluate candidate administrative cutoffs
        # ----------------------------------------------------

        for cutoff in cutoffs:

            # Whole months elapsed from issue month to cutoff
            followup_months = (
                (cutoff.year - issue.dt.year) * 12
                + (cutoff.month - issue.dt.month)
            )

            contractual_followup_complete = (
                followup_months >= term_months
            )

            mature = terminal | contractual_followup_complete

            temp = pd.DataFrame({
                "split": split,
                "mature": mature,
                "badloan": df["badloan"]
            })

            for split_name in ["Train", "Validation", "Test"]:

                smask = temp["split"].eq(split_name)

                n_total = int(smask.sum())
                n_mature = int(
                    (smask & temp["mature"]).sum()
                )

                mature_bad = temp.loc[
                    smask & temp["mature"],
                    "badloan"
                ]

                n_bad = int(
                    (mature_bad == 1).sum()
                )

                # Accumulate across chunks
                key = (cutoff, split_name)

                existing = next(
                    (
                        r for r in results
                        if r["key"] == key
                    ),
                    None
                )

                if existing is None:
                    results.append({
                        "key": key,
                        "cutoff": cutoff,
                        "split": split_name,
                        "total": n_total,
                        "mature": n_mature,
                        "badloan_mature": n_bad
                    })
                else:
                    existing["total"] += n_total
                    existing["mature"] += n_mature
                    existing["badloan_mature"] += n_bad

        print(f"Chunk {chunk_no:02d} processed")

# ------------------------------------------------------------
# Final table
# ------------------------------------------------------------

result_df = pd.DataFrame(results).drop(
    columns="key"
)

result_df["maturity_pct"] = (
    100 * result_df["mature"] / result_df["total"]
)

result_df["badloan_rate_mature_pct"] = np.where(
    result_df["mature"] > 0,
    100 * result_df["badloan_mature"] / result_df["mature"],
    np.nan
)

print("\n" + "=" * 110)
print("MATURITY RECONSTRUCTION RESULTS")
print("=" * 110)

print(
    result_df.to_string(
        index=False,
        formatters={
            "maturity_pct": "{:.3f}".format,
            "badloan_rate_mature_pct": "{:.3f}".format
        }
    )
)

# ------------------------------------------------------------
# Compare directly with manuscript counts
# ------------------------------------------------------------

targets = {
    "Train": 875_616,
    "Validation": 27_374,
    "Test": 41_145
}

result_df["manuscript_target"] = (
    result_df["split"].map(targets)
)

result_df["difference_from_target"] = (
    result_df["mature"]
    - result_df["manuscript_target"]
)

print("\n" + "=" * 110)
print("DIFFERENCE FROM MANUSCRIPT-REPORTED MATURE COUNTS")
print("=" * 110)

print(
    result_df[
        [
            "cutoff",
            "split",
            "mature",
            "manuscript_target",
            "difference_from_target"
        ]
    ].to_string(index=False)
)

result_df.to_csv(
    OUTPUT_DIR / "05_maturity_reconstruction.csv",
    index=False
)

print("\nSaved:")
print(OUTPUT_DIR / "05_maturity_reconstruction.csv")

Testing maturity reconstruction...

Chunk 01 processed
Chunk 02 processed
Chunk 03 processed
Chunk 04 processed
Chunk 05 processed
Chunk 06 processed
Chunk 07 processed
Chunk 08 processed
Chunk 09 processed
Chunk 10 processed
Chunk 11 processed
Chunk 12 processed
Chunk 13 processed
Chunk 14 processed

MATURITY RECONSTRUCTION RESULTS
    cutoff      split   total  mature  badloan_mature maturity_pct badloan_rate_mature_pct
2020-04-01      Train 1931267 1304012          181135       67.521                  13.891
2020-04-01 Validation  370994   27371            2768        7.378                  10.113
2020-04-01       Test  401169   41141            6021       10.255                  14.635
2020-05-01      Train 1931267 1326379          181933       68.679                  13.717
2020-05-01 Validation  370994   27371            2768        7.378                  10.113
2020-05-01       Test  401169   41141            6021       10.255                  14.635
2020-06-01      Train 193126

In [11]:
# ============================================================
# CELL 8 — Diagnose maturity-rule discrepancy
#
# Questions:
#   1. Do the 3 validation + 4 test differences come from
#      treating "Default" as terminal?
#   2. What are status counts within each chronological split?
#   3. Which simple terminal-status definitions reproduce
#      the manuscript counts?
#
# No modelling. No survival analysis.
# ============================================================

import pandas as pd
import numpy as np

USECOLS = [
    "issue_date",
    "loan_status",
    "term",
    "badloan"
]

CHUNK_SIZE = 200_000

# Use 2020-04-01 because Cell 7 showed validation/test
# counts are already stable at this point.
CUTOFF = pd.Timestamp("2020-04-01")

rules = {
    "FP_or_CO": {
        "Fully Paid",
        "Charged Off"
    },

    "FP_CO_Default": {
        "Fully Paid",
        "Charged Off",
        "Default"
    },

    "All_badloan_statuses_plus_FP": {
        "Fully Paid",
        "Charged Off",
        "Default",
        "Late (16-30 days)",
        "Late (31-120 days)"
    }
}

# Accumulators
status_split_counts = {}
rule_counts = {}

for rule_name in rules:
    for split_name in ["Train", "Validation", "Test"]:
        rule_counts[(rule_name, split_name)] = {
            "total": 0,
            "mature": 0,
            "bad": 0
        }

print("Diagnosing maturity definitions...\n")

with pd.read_stata(
    DTA_PATH,
    columns=USECOLS,
    convert_categoricals=False,
    chunksize=CHUNK_SIZE
) as reader:

    for chunk_no, df in enumerate(reader, start=1):

        issue = pd.to_datetime(
            df["issue_date"],
            errors="coerce"
        )

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        term_months = pd.to_numeric(
            df["term"]
            .fillna("")
            .astype(str)
            .str.extract(r"(\d+)", expand=False),
            errors="coerce"
        )

        split = np.select(
            [
                (issue >= pd.Timestamp("2012-08-01")) &
                (issue <= pd.Timestamp("2018-06-01")),

                (issue >= pd.Timestamp("2018-07-01")) &
                (issue <= pd.Timestamp("2019-03-01")),

                (issue >= pd.Timestamp("2019-04-01")) &
                (issue <= pd.Timestamp("2019-12-01"))
            ],
            ["Train", "Validation", "Test"],
            default="Outside"
        )

        followup_months = (
            (CUTOFF.year - issue.dt.year) * 12
            + (CUTOFF.month - issue.dt.month)
        )

        contractual_complete = (
            followup_months >= term_months
        )

        # ----------------------------------------------------
        # A. Status counts by chronological split
        # ----------------------------------------------------

        for split_name in ["Train", "Validation", "Test"]:

            smask = split == split_name

            vc = status[smask].value_counts()

            for loan_status, n in vc.items():
                key = (split_name, loan_status)
                status_split_counts[key] = (
                    status_split_counts.get(key, 0)
                    + int(n)
                )

        # ----------------------------------------------------
        # B. Test candidate maturity definitions
        # ----------------------------------------------------

        for rule_name, terminal_statuses in rules.items():

            terminal = status.isin(terminal_statuses)

            mature = terminal | contractual_complete

            for split_name in ["Train", "Validation", "Test"]:

                smask = split == split_name
                mmask = smask & mature

                rec = rule_counts[(rule_name, split_name)]

                rec["total"] += int(smask.sum())
                rec["mature"] += int(mmask.sum())
                rec["bad"] += int(
                    (df.loc[mmask, "badloan"] == 1).sum()
                )

        print(f"Chunk {chunk_no:02d} processed")


# ============================================================
# A. STATUS COUNTS BY SPLIT
# ============================================================

print("\n" + "=" * 110)
print("A. LOAN STATUS COUNTS BY CHRONOLOGICAL SPLIT")
print("=" * 110)

status_rows = []

for (split_name, loan_status), n in status_split_counts.items():
    status_rows.append({
        "split": split_name,
        "loan_status": loan_status,
        "n": n
    })

status_df = pd.DataFrame(status_rows)

status_pivot = status_df.pivot(
    index="loan_status",
    columns="split",
    values="n"
).fillna(0).astype(int)

# preferred column order
status_pivot = status_pivot.reindex(
    columns=["Train", "Validation", "Test"]
)

print(status_pivot.to_string())


# ============================================================
# B. CANDIDATE MATURITY RULES
# ============================================================

print("\n" + "=" * 110)
print("B. CANDIDATE MATURITY DEFINITIONS")
print("=" * 110)

rows = []

targets = {
    "Train": 875_616,
    "Validation": 27_374,
    "Test": 41_145
}

for (rule_name, split_name), rec in rule_counts.items():

    rows.append({
        "rule": rule_name,
        "split": split_name,
        "total": rec["total"],
        "mature": rec["mature"],
        "badloan_mature": rec["bad"],
        "target": targets[split_name],
        "difference": rec["mature"] - targets[split_name]
    })

rule_df = pd.DataFrame(rows)

rule_df["maturity_pct"] = (
    100 * rule_df["mature"] / rule_df["total"]
)

rule_df["badloan_rate_pct"] = (
    100 * rule_df["badloan_mature"] / rule_df["mature"]
)

print(
    rule_df[
        [
            "rule",
            "split",
            "mature",
            "target",
            "difference",
            "maturity_pct",
            "badloan_mature",
            "badloan_rate_pct"
        ]
    ].to_string(
        index=False,
        formatters={
            "maturity_pct": "{:.4f}".format,
            "badloan_rate_pct": "{:.4f}".format
        }
    )
)


# ============================================================
# C. DEFAULT COUNTS SPECIFICALLY
# ============================================================

print("\n" + "=" * 110)
print("C. DEFAULT STATUS COUNTS BY SPLIT")
print("=" * 110)

default_rows = status_df[
    status_df["loan_status"].eq("Default")
].copy()

print(default_rows.to_string(index=False))


# Save
status_pivot.to_csv(
    OUTPUT_DIR / "06_status_counts_by_split.csv"
)

rule_df.to_csv(
    OUTPUT_DIR / "07_maturity_rule_comparison.csv",
    index=False
)

print("\nSaved:")
print(OUTPUT_DIR / "06_status_counts_by_split.csv")
print(OUTPUT_DIR / "07_maturity_rule_comparison.csv")

Diagnosing maturity definitions...

Chunk 01 processed
Chunk 02 processed
Chunk 03 processed
Chunk 04 processed
Chunk 05 processed
Chunk 06 processed
Chunk 07 processed
Chunk 08 processed
Chunk 09 processed
Chunk 10 processed
Chunk 11 processed
Chunk 12 processed
Chunk 13 processed
Chunk 14 processed

A. LOAN STATUS COUNTS BY CHRONOLOGICAL SPLIT
split                 Train  Validation    Test
loan_status                                    
Charged Off          169095        2768    6021
Current             1015804      336925  352125
Default                 115           3       4
Fully Paid           705616       24603   35120
In Grace Period       13079        1486    2050
Late (16-30 days)      5558        1125     827
Late (31-120 days)    22000        4084    5022

B. CANDIDATE MATURITY DEFINITIONS
                        rule      split  mature  target  difference maturity_pct  badloan_mature badloan_rate_pct
                    FP_or_CO      Train 1304012  875616      428396    

In [12]:
# ============================================================
# CELL 9 — Audit old notebook for training/maturity construction
#
# Purpose:
#   Find exactly where the old analysis created:
#       875,616 mature training loans
#       27,374 mature validation loans
#       41,145 mature test loans
#
# This cell DOES NOT execute the old notebook.
# ============================================================

import json
from pathlib import Path

OLD_NOTEBOOK = PROJECT_DIR / "Fintech_25_12_2025.ipynb"

print("Old notebook exists:", OLD_NOTEBOOK.exists())
print("Path:", OLD_NOTEBOOK)

with open(
    OLD_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

search_terms = [
    "875616",
    "875,616",
    "27374",
    "27,374",
    "41145",
    "41,145",
    "mature",
    "maturity",
    "badloan",
    "180000",
    "180,000",
    "train",
    "validation",
    "split",
    "term",
    "last_pymnt",
    "followup",
    "follow_up",
    "terminal"
]

matches = []

for cell_index, cell in enumerate(nb.get("cells", [])):

    source = "".join(cell.get("source", []))

    source_lower = source.lower()

    matched_terms = [
        term
        for term in search_terms
        if term.lower() in source_lower
    ]

    if matched_terms:

        matches.append({
            "cell_index": cell_index,
            "cell_type": cell.get("cell_type"),
            "matched_terms": matched_terms,
            "source": source
        })

print("\n" + "=" * 110)
print("MATCHING OLD-NOTEBOOK CELLS")
print("=" * 110)

print("Number of matching cells:", len(matches))

for m in matches:

    print("\n" + "#" * 110)
    print(
        f"CELL INDEX: {m['cell_index']} | "
        f"TYPE: {m['cell_type']} | "
        f"MATCHES: {', '.join(m['matched_terms'])}"
    )
    print("#" * 110)

    print(m["source"])

print("\n" + "=" * 110)
print("END OF OLD-NOTEBOOK AUDIT")
print("=" * 110)

Old notebook exists: True
Path: E:\Governance-Policy\Cogent Economics & Finance\Fintech_25_12_2025.ipynb

MATCHING OLD-NOTEBOOK CELLS
Number of matching cells: 26

##############################################################################################################
CELL INDEX: 31 | TYPE: code | MATCHES: term
##############################################################################################################
# Borrower characteristics
borrower_features = [
    "annual_inc", "dti", "emp_length", "home_ownership",
    "verification_status", "delinq_2yrs", "pub_rec",
    "mort_acc", "tax_liens", "pub_rec_bankruptcies"
]

# Loan contract terms (known at origination)
loan_features = [
    "loan_amnt", "funded_amnt", "term", "int_rate",
    "installment", "grade", "sub_grade", "purpose",
    "initial_list_status"
]

# Credit history & utilization
credit_features = [
    "open_acc", "total_acc", "revol_bal", "revol_util",
    "pct_tl_nvr_dlq", "num_rev_accts", "num_sats",
  

In [13]:
# ============================================================
# CELL 10 — Locate creation of final modelling frame
# and any hidden maturity filtering in old notebook
#
# DOES NOT execute old notebook.
# ============================================================

import json
import re

OLD_NOTEBOOK = PROJECT_DIR / "Fintech_25_12_2025.ipynb"

with open(OLD_NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

patterns = [
    r"final_modelling_frame",
    r"to_parquet",
    r"matur",
    r"terminal",
    r"follow.?up",
    r"contractual",
    r"loan_status",
    r"Fully Paid",
    r"Charged Off",
    r"Default",
    r"issue_date",
    r"last_date",
    r"last_pymnt",
    r"875",
    r"2737",
    r"4114"
]

regex = re.compile(
    "|".join(f"(?:{p})" for p in patterns),
    flags=re.IGNORECASE
)

hits = []

for idx, cell in enumerate(nb.get("cells", [])):

    source = "".join(cell.get("source", []))

    if regex.search(source):

        hits.append(idx)

print("Matching cell indices:")
print(hits)

print("\nNumber of matching cells:", len(hits))

# ------------------------------------------------------------
# Print each hit WITH neighbouring cells.
# This is important because parquet creation/filtering may
# occur immediately before or after the matching cell.
# ------------------------------------------------------------

shown = set()

for hit in hits:

    for idx in range(
        max(0, hit - 1),
        min(len(nb["cells"]), hit + 2)
    ):

        if idx in shown:
            continue

        shown.add(idx)

        cell = nb["cells"][idx]
        source = "".join(cell.get("source", []))

        print("\n" + "=" * 120)
        print(
            f"CELL INDEX: {idx} | "
            f"TYPE: {cell.get('cell_type')}"
        )
        print("=" * 120)

        print(source)

        # Also print stored textual outputs when available.
        if cell.get("cell_type") == "code":

            outputs = cell.get("outputs", [])

            if outputs:
                print("\n--- STORED OUTPUT ---")

            for out in outputs:

                if "text" in out:
                    print("".join(out["text"]))

                elif "data" in out:

                    data = out["data"]

                    if "text/plain" in data:
                        print("".join(data["text/plain"]))

print("\n" + "=" * 120)
print("END OF TARGETED AUDIT")
print("=" * 120)

Matching cell indices:
[11, 36, 42, 44, 46, 51, 54, 55, 56, 58, 59, 60, 62, 63, 64, 66, 74, 75, 77, 78, 79, 80, 82, 83, 84, 85, 89]

Number of matching cells: 27

CELL INDEX: 10 | TYPE: markdown
#Step 1: Basic cleaning + standardise date columns

CELL INDEX: 11 | TYPE: code
import numpy as np
import pandas as pd

DATE_COLS = ["issue_date", "daily_date", "last_date", "issue_daily"]

def to_datetime_safe(df, cols=DATE_COLS):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

df = to_datetime_safe(df)
print(df.dtypes)

--- STORED OUTPUT ---
loan_amnt                                  int32
funded_amnt                                int32
funded_amnt_inv                          float64
term                                      object
int_rate                                 float64
installment                              float32
grade                                     object
sub_grade               

In [14]:
# ============================================================
# CELL 11 — Determine empirical observation horizon
#
# Purpose:
#   Establish how far outcome information extends beyond
#   origination using issue date, last payment date and
#   next scheduled payment date.
#
# No maturity rule is imposed yet.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_d",
    "issue_date",
    "last_pymnt_d",
    "last_date",
    "next_pymnt_d",
    "loan_status",
    "term"
]

CHUNK = 200_000

summary = {
    "n": 0,
    "issue_min": None,
    "issue_max": None,
    "last_min": None,
    "last_max": None,
    "next_min": None,
    "next_max": None
}

status_last_max = {}
status_last_min = {}
status_n = {}

def update_min(old, new):
    if pd.isna(new):
        return old
    if old is None or new < old:
        return new
    return old

def update_max(old, new):
    if pd.isna(new):
        return old
    if old is None or new > old:
        return new
    return old

print("Scanning dataset in chunks...\n")

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    while True:

        try:
            chunk = reader.read(CHUNK)
        except StopIteration:
            break

        if chunk is None or len(chunk) == 0:
            break

        summary["n"] += len(chunk)

        # ----------------------------------------------------
        # Use already-constructed Stata monthly dates.
        # Cell 5B established issue_date / last_date are valid.
        # ----------------------------------------------------
        issue = pd.to_datetime(
            chunk["issue_date"],
            errors="coerce"
        )

        last = pd.to_datetime(
            chunk["last_date"],
            errors="coerce"
        )

        # Raw next_pymnt_d uses strings such as Jan-2019.
        next_pay = pd.to_datetime(
            chunk["next_pymnt_d"].astype(str).str.strip(),
            format="%b-%Y",
            errors="coerce"
        )

        summary["issue_min"] = update_min(
            summary["issue_min"], issue.min()
        )
        summary["issue_max"] = update_max(
            summary["issue_max"], issue.max()
        )

        summary["last_min"] = update_min(
            summary["last_min"], last.min()
        )
        summary["last_max"] = update_max(
            summary["last_max"], last.max()
        )

        summary["next_min"] = update_min(
            summary["next_min"], next_pay.min()
        )
        summary["next_max"] = update_max(
            summary["next_max"], next_pay.max()
        )

        # ----------------------------------------------------
        # Status-specific last-payment horizon
        # ----------------------------------------------------
        tmp = pd.DataFrame({
            "status": chunk["loan_status"].astype(str),
            "last": last
        })

        for status, g in tmp.groupby("status"):

            status_n[status] = (
                status_n.get(status, 0) + len(g)
            )

            gmin = g["last"].min()
            gmax = g["last"].max()

            status_last_min[status] = update_min(
                status_last_min.get(status), gmin
            )

            status_last_max[status] = update_max(
                status_last_max.get(status), gmax
            )

        print(
            f"Processed {summary['n']:,} rows",
            end="\r"
        )

print("\n\n" + "=" * 100)
print("A. OVERALL OBSERVATION HORIZON")
print("=" * 100)

print(f"Rows:                    {summary['n']:,}")
print(f"Origination minimum:     {summary['issue_min']}")
print(f"Origination maximum:     {summary['issue_max']}")
print(f"Last-payment minimum:    {summary['last_min']}")
print(f"Last-payment maximum:    {summary['last_max']}")
print(f"Next-payment minimum:    {summary['next_min']}")
print(f"Next-payment maximum:    {summary['next_max']}")

print("\n" + "=" * 100)
print("B. LAST-PAYMENT HORIZON BY CURRENT LOAN STATUS")
print("=" * 100)

status_table = pd.DataFrame([
    {
        "loan_status": status,
        "n": status_n[status],
        "last_payment_min": status_last_min.get(status),
        "last_payment_max": status_last_max.get(status)
    }
    for status in sorted(status_n)
])

print(status_table.to_string(index=False))

# Save
status_table.to_csv(
    OUTPUT_DIR / "cell11_status_observation_horizon.csv",
    index=False
)

pd.DataFrame([summary]).to_csv(
    OUTPUT_DIR / "cell11_overall_observation_horizon.csv",
    index=False
)

print("\nSaved Cell 11 outputs to:")
print(OUTPUT_DIR)

Scanning dataset in chunks...

Processed 2,703,430 rows

A. OVERALL OBSERVATION HORIZON
Rows:                    2,703,430
Origination minimum:     2012-08-01 00:00:00
Origination maximum:     2019-12-01 00:00:00
Last-payment minimum:    2012-08-01 00:00:00
Last-payment maximum:    2020-07-01 00:00:00
Next-payment minimum:    None
Next-payment maximum:    None

B. LAST-PAYMENT HORIZON BY CURRENT LOAN STATUS
       loan_status       n last_payment_min last_payment_max
       Charged Off  177884       2012-08-01       2020-04-01
           Current 1704854       2017-03-01       2020-07-01
           Default     122       2016-11-01       2020-02-01
        Fully Paid  765339       2012-09-01       2020-06-01
   In Grace Period   16615       2017-03-01       2020-06-01
 Late (16-30 days)    7510       2017-02-01       2020-06-01
Late (31-120 days)   31106       2016-11-01       2020-05-01

Saved Cell 11 outputs to:
E:\Governance-Policy\Cogent Economics & Finance\revision_outputs


In [15]:
# ============================================================
# CELL 12 — Audit candidate administrative observation dates
#
# Purpose:
#   Determine whether the dataset contains a defensible
#   administrative observation / extraction horizon.
#
# IMPORTANT:
#   We are NOT defining censoring or maturity yet.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_date",
    "loan_status",
    "last_pymnt_d",
    "last_date",
    "next_pymnt_d",
    "last_credit_pull_d",
    "daily_date",
    "issue_daily"
]

CHUNK = 200_000

# ------------------------------------------------------------
# First: inspect raw examples and dtypes
# ------------------------------------------------------------

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:
    sample = reader.read(50)

print("=" * 110)
print("A. DTYPES")
print("=" * 110)

print(sample.dtypes)

print("\n" + "=" * 110)
print("B. RAW EXAMPLES")
print("=" * 110)

for col in COLS:
    print(f"\n--- {col} ---")
    vals = (
        sample[col]
        .dropna()
        .astype(str)
        .str.strip()
    )
    vals = vals[vals.ne("")]
    print(vals.head(10).tolist())


# ------------------------------------------------------------
# Helper for parsing month strings flexibly
# ------------------------------------------------------------

def parse_month_strings(series):

    raw = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )

    out = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    # Example: May-17
    mask = raw.str.match(r"^[A-Za-z]{3}-\d{2}$", na=False)
    if mask.any():
        out.loc[mask] = pd.to_datetime(
            raw.loc[mask],
            format="%b-%y",
            errors="coerce"
        )

    # Example: May-2017
    mask = raw.str.match(r"^[A-Za-z]{3}-\d{4}$", na=False)
    if mask.any():
        out.loc[mask] = pd.to_datetime(
            raw.loc[mask],
            format="%b-%Y",
            errors="coerce"
        )

    return out


# ------------------------------------------------------------
# Full-data audit
# ------------------------------------------------------------

audit = {
    "last_pymnt_d": {
        "nonblank": 0,
        "parsed": 0,
        "min": None,
        "max": None
    },
    "next_pymnt_d": {
        "nonblank": 0,
        "parsed": 0,
        "min": None,
        "max": None
    },
    "last_credit_pull_d": {
        "nonblank": 0,
        "parsed": 0,
        "min": None,
        "max": None
    },
    "daily_date": {
        "nonmissing": 0,
        "min": None,
        "max": None
    },
    "issue_daily": {
        "nonmissing": 0,
        "min": None,
        "max": None
    }
}


def update_min(old, new):
    if pd.isna(new):
        return old
    return new if old is None else min(old, new)


def update_max(old, new):
    if pd.isna(new):
        return old
    return new if old is None else max(old, new)


print("\n" + "=" * 110)
print("C. FULL-DATA DATE AUDIT")
print("=" * 110)

total = 0

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        total += len(df)

        # ----------------------------------------------
        # Raw month-string fields
        # ----------------------------------------------

        for col in [
            "last_pymnt_d",
            "next_pymnt_d",
            "last_credit_pull_d"
        ]:

            raw = (
                df[col]
                .fillna("")
                .astype(str)
                .str.strip()
            )

            nonblank = raw.ne("")
            parsed = parse_month_strings(df[col])

            audit[col]["nonblank"] += int(nonblank.sum())
            audit[col]["parsed"] += int(parsed.notna().sum())

            audit[col]["min"] = update_min(
                audit[col]["min"],
                parsed.min()
            )

            audit[col]["max"] = update_max(
                audit[col]["max"],
                parsed.max()
            )

        # ----------------------------------------------
        # Already converted date fields
        # ----------------------------------------------

        for col in ["daily_date", "issue_daily"]:

            x = pd.to_datetime(
                df[col],
                errors="coerce"
            )

            audit[col]["nonmissing"] += int(x.notna().sum())

            audit[col]["min"] = update_min(
                audit[col]["min"],
                x.min()
            )

            audit[col]["max"] = update_max(
                audit[col]["max"],
                x.max()
            )

        print(
            f"Processed {total:,} rows",
            end="\r"
        )


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n\nRows scanned:", f"{total:,}")

rows = []

for col, x in audit.items():

    row = {"field": col}

    row.update(x)

    if "nonblank" in x:
        row["coverage_pct"] = (
            100 * x["nonblank"] / total
        )
        row["parse_success_pct_of_nonblank"] = (
            100 * x["parsed"] / x["nonblank"]
            if x["nonblank"] > 0 else np.nan
        )

    else:
        row["coverage_pct"] = (
            100 * x["nonmissing"] / total
        )

    rows.append(row)

audit_df = pd.DataFrame(rows)

print("\n" + "=" * 110)
print("D. CANDIDATE OBSERVATION-DATE SUMMARY")
print("=" * 110)

print(audit_df.to_string(index=False))


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

audit_df.to_csv(
    OUTPUT_DIR / "cell12_candidate_observation_dates.csv",
    index=False
)

print("\nSaved:")
print(
    OUTPUT_DIR /
    "cell12_candidate_observation_dates.csv"
)

A. DTYPES
issue_date            datetime64[ns]
loan_status                   object
last_pymnt_d                  object
last_date             datetime64[ns]
next_pymnt_d                  object
last_credit_pull_d            object
daily_date            datetime64[ns]
issue_daily           datetime64[ns]
dtype: object

B. RAW EXAMPLES

--- issue_date ---
['2012-08-01', '2012-08-01', '2012-09-01', '2012-09-01', '2012-09-01', '2012-09-01', '2012-09-01', '2012-09-01', '2012-09-01', '2012-09-01']

--- loan_status ---
['Fully Paid', 'Fully Paid', 'Fully Paid', 'Charged Off', 'Charged Off', 'Fully Paid', 'Charged Off', 'Fully Paid', 'Fully Paid', 'Fully Paid']

--- last_pymnt_d ---
['Sep-15', 'Sep-15', 'Sep-15', 'Jan-14', 'Jun-14', 'Sep-15', 'Dec-14', 'Jul-14', 'Feb-16', 'Jun-15']

--- last_date ---
['2015-09-01', '2015-09-01', '2015-09-01', '2014-01-01', '2014-06-01', '2015-09-01', '2014-12-01', '2014-07-01', '2016-02-01', '2015-06-01']

--- next_pymnt_d ---
['May-17', 'May-17']

--- last_c

In [16]:
# ============================================================
# CELL 13 — Contractual follow-up and early-resolution audit
#
# Purpose:
#   1. Calculate months from origination to last payment
#   2. Compare observed duration with 36/60-month contract
#   3. Quantify early Fully Paid and Charged Off resolutions
#   4. Characterise timing by final loan status
#
# IMPORTANT:
#   This is descriptive auditing only.
#   No survival model and no new maturity definition yet.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_date",
    "last_pymnt_d",
    "loan_status",
    "term"
]

CHUNK = 200_000

records = []

print("Auditing contractual follow-up...\n")

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    total = 0

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        total += len(df)

        # ----------------------------------------------------
        # Origination date
        # ----------------------------------------------------
        issue = pd.to_datetime(
            df["issue_date"],
            errors="coerce"
        )

        # ----------------------------------------------------
        # Raw last-payment date
        # Do NOT use constructed last_date because missing
        # last_pymnt_d was replaced with issue_date upstream.
        # ----------------------------------------------------
        raw_last = (
            df["last_pymnt_d"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        last = pd.to_datetime(
            raw_last.where(raw_last.ne("")),
            format="%b-%y",
            errors="coerce"
        )

        # ----------------------------------------------------
        # Contractual term
        # ----------------------------------------------------
        term_months = pd.to_numeric(
            df["term"]
            .fillna("")
            .astype(str)
            .str.extract(r"(\d+)", expand=False),
            errors="coerce"
        )

        # ----------------------------------------------------
        # Month-level duration from issue to last payment
        # ----------------------------------------------------
        observed_months = (
            (last.dt.year - issue.dt.year) * 12
            + (last.dt.month - issue.dt.month)
        )

        # Guard against impossible negative values
        observed_months = observed_months.where(
            observed_months >= 0
        )

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        temp = pd.DataFrame({
            "loan_status": status,
            "term_months": term_months,
            "observed_months": observed_months
        })

        # ----------------------------------------------------
        # Timing relative to contractual term
        # ----------------------------------------------------
        temp["duration_ratio"] = (
            temp["observed_months"]
            / temp["term_months"]
        )

        temp["before_contract_term"] = (
            temp["observed_months"]
            < temp["term_months"]
        )

        temp["at_or_after_contract_term"] = (
            temp["observed_months"]
            >= temp["term_months"]
        )

        records.append(temp)

        print(
            f"Processed {total:,} rows",
            end="\r"
        )

# ============================================================
# Combine
# ============================================================

audit = pd.concat(
    records,
    ignore_index=True
)

print("\n\nRows:", f"{len(audit):,}")

# ============================================================
# A. Term distribution
# ============================================================

print("\n" + "=" * 110)
print("A. CONTRACTUAL TERM DISTRIBUTION")
print("=" * 110)

term_table = (
    audit["term_months"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("term_months")
    .reset_index(name="n")
)

term_table["pct"] = (
    100 * term_table["n"] / len(audit)
)

print(term_table.to_string(index=False))


# ============================================================
# B. Follow-up by loan status
# ============================================================

print("\n" + "=" * 110)
print("B. OBSERVED MONTHS TO LAST PAYMENT BY LOAN STATUS")
print("=" * 110)

status_duration = (
    audit
    .groupby("loan_status", dropna=False)
    .agg(
        n=("loan_status", "size"),
        duration_available=("observed_months", "count"),
        median_months=("observed_months", "median"),
        mean_months=("observed_months", "mean"),
        q25_months=("observed_months", lambda x: x.quantile(0.25)),
        q75_months=("observed_months", lambda x: x.quantile(0.75))
    )
    .reset_index()
)

status_duration["duration_available_pct"] = (
    100
    * status_duration["duration_available"]
    / status_duration["n"]
)

print(
    status_duration.to_string(
        index=False,
        formatters={
            "median_months": "{:.2f}".format,
            "mean_months": "{:.2f}".format,
            "q25_months": "{:.2f}".format,
            "q75_months": "{:.2f}".format,
            "duration_available_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# C. Resolution before contractual term
# ============================================================

print("\n" + "=" * 110)
print("C. STATUS × RESOLUTION BEFORE CONTRACTUAL TERM")
print("=" * 110)

resolved = audit[
    audit["observed_months"].notna()
    & audit["term_months"].notna()
].copy()

early_table = (
    resolved
    .groupby("loan_status")
    .agg(
        n_with_duration=("observed_months", "size"),
        before_term=("before_contract_term", "sum"),
        at_or_after_term=("at_or_after_contract_term", "sum")
    )
    .reset_index()
)

early_table["before_term_pct"] = (
    100
    * early_table["before_term"]
    / early_table["n_with_duration"]
)

early_table["at_or_after_term_pct"] = (
    100
    * early_table["at_or_after_term"]
    / early_table["n_with_duration"]
)

print(
    early_table.to_string(
        index=False,
        formatters={
            "before_term_pct": "{:.3f}".format,
            "at_or_after_term_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# D. Fully Paid specifically by contractual term
# ============================================================

print("\n" + "=" * 110)
print("D. FULLY PAID TIMING BY 36/60-MONTH CONTRACT")
print("=" * 110)

fp = resolved[
    resolved["loan_status"].eq("Fully Paid")
].copy()

fp_table = (
    fp
    .groupby("term_months")
    .agg(
        n=("observed_months", "size"),
        median_months=("observed_months", "median"),
        q25_months=("observed_months", lambda x: x.quantile(0.25)),
        q75_months=("observed_months", lambda x: x.quantile(0.75)),
        early_paid=("before_contract_term", "sum")
    )
    .reset_index()
)

fp_table["early_paid_pct"] = (
    100 * fp_table["early_paid"] / fp_table["n"]
)

print(
    fp_table.to_string(
        index=False,
        formatters={
            "median_months": "{:.2f}".format,
            "q25_months": "{:.2f}".format,
            "q75_months": "{:.2f}".format,
            "early_paid_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# E. Charged Off specifically by contractual term
# ============================================================

print("\n" + "=" * 110)
print("E. CHARGED OFF TIMING BY 36/60-MONTH CONTRACT")
print("=" * 110)

co = resolved[
    resolved["loan_status"].eq("Charged Off")
].copy()

co_table = (
    co
    .groupby("term_months")
    .agg(
        n=("observed_months", "size"),
        median_months=("observed_months", "median"),
        q25_months=("observed_months", lambda x: x.quantile(0.25)),
        q75_months=("observed_months", lambda x: x.quantile(0.75)),
        before_term=("before_contract_term", "sum")
    )
    .reset_index()
)

co_table["before_term_pct"] = (
    100 * co_table["before_term"] / co_table["n"]
)

print(
    co_table.to_string(
        index=False,
        formatters={
            "median_months": "{:.2f}".format,
            "q25_months": "{:.2f}".format,
            "q75_months": "{:.2f}".format,
            "before_term_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# Save
# ============================================================

term_table.to_csv(
    OUTPUT_DIR / "cell13_term_distribution.csv",
    index=False
)

status_duration.to_csv(
    OUTPUT_DIR / "cell13_duration_by_status.csv",
    index=False
)

early_table.to_csv(
    OUTPUT_DIR / "cell13_before_term_by_status.csv",
    index=False
)

fp_table.to_csv(
    OUTPUT_DIR / "cell13_fully_paid_timing.csv",
    index=False
)

co_table.to_csv(
    OUTPUT_DIR / "cell13_charged_off_timing.csv",
    index=False
)

print("\nSaved Cell 13 outputs to:")
print(OUTPUT_DIR)

Auditing contractual follow-up...

Processed 2,703,430 rows

Rows: 2,703,430

A. CONTRACTUAL TERM DISTRIBUTION
 term_months       n       pct
          36 1901819 70.348372
          60  801611 29.651628

B. OBSERVED MONTHS TO LAST PAYMENT BY LOAN STATUS
       loan_status       n  duration_available median_months mean_months q25_months q75_months duration_available_pct
       Charged Off  177884              175144         13.00       15.08       7.00      22.00                 98.460
           Current 1704854             1704854          8.00       10.69       5.00      12.00                100.000
           Default     122                 121          9.00       12.69       8.00      11.00                 99.180
        Fully Paid  765339              765339         18.00       19.29       8.00      32.00                100.000
   In Grace Period   16615               16615         10.00       12.75       6.00      13.00                100.000
 Late (16-30 days)    7510           

In [17]:
# ============================================================
# CELL 13 — Contractual follow-up and early-resolution audit
#
# Purpose:
#   1. Calculate months from origination to last payment
#   2. Compare observed duration with 36/60-month contract
#   3. Quantify early Fully Paid and Charged Off resolutions
#   4. Characterise timing by final loan status
#
# IMPORTANT:
#   This is descriptive auditing only.
#   No survival model and no new maturity definition yet.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_date",
    "last_pymnt_d",
    "loan_status",
    "term"
]

CHUNK = 200_000

records = []

print("Auditing contractual follow-up...\n")

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    total = 0

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        total += len(df)

        # ----------------------------------------------------
        # Origination date
        # ----------------------------------------------------
        issue = pd.to_datetime(
            df["issue_date"],
            errors="coerce"
        )

        # ----------------------------------------------------
        # Raw last-payment date
        # Do NOT use constructed last_date because missing
        # last_pymnt_d was replaced with issue_date upstream.
        # ----------------------------------------------------
        raw_last = (
            df["last_pymnt_d"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        last = pd.to_datetime(
            raw_last.where(raw_last.ne("")),
            format="%b-%y",
            errors="coerce"
        )

        # ----------------------------------------------------
        # Contractual term
        # ----------------------------------------------------
        term_months = pd.to_numeric(
            df["term"]
            .fillna("")
            .astype(str)
            .str.extract(r"(\d+)", expand=False),
            errors="coerce"
        )

        # ----------------------------------------------------
        # Month-level duration from issue to last payment
        # ----------------------------------------------------
        observed_months = (
            (last.dt.year - issue.dt.year) * 12
            + (last.dt.month - issue.dt.month)
        )

        # Guard against impossible negative values
        observed_months = observed_months.where(
            observed_months >= 0
        )

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        temp = pd.DataFrame({
            "loan_status": status,
            "term_months": term_months,
            "observed_months": observed_months
        })

        # ----------------------------------------------------
        # Timing relative to contractual term
        # ----------------------------------------------------
        temp["duration_ratio"] = (
            temp["observed_months"]
            / temp["term_months"]
        )

        temp["before_contract_term"] = (
            temp["observed_months"]
            < temp["term_months"]
        )

        temp["at_or_after_contract_term"] = (
            temp["observed_months"]
            >= temp["term_months"]
        )

        records.append(temp)

        print(
            f"Processed {total:,} rows",
            end="\r"
        )

# ============================================================
# Combine
# ============================================================

audit = pd.concat(
    records,
    ignore_index=True
)

print("\n\nRows:", f"{len(audit):,}")

# ============================================================
# A. Term distribution
# ============================================================

print("\n" + "=" * 110)
print("A. CONTRACTUAL TERM DISTRIBUTION")
print("=" * 110)

term_table = (
    audit["term_months"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("term_months")
    .reset_index(name="n")
)

term_table["pct"] = (
    100 * term_table["n"] / len(audit)
)

print(term_table.to_string(index=False))


# ============================================================
# B. Follow-up by loan status
# ============================================================

print("\n" + "=" * 110)
print("B. OBSERVED MONTHS TO LAST PAYMENT BY LOAN STATUS")
print("=" * 110)

status_duration = (
    audit
    .groupby("loan_status", dropna=False)
    .agg(
        n=("loan_status", "size"),
        duration_available=("observed_months", "count"),
        median_months=("observed_months", "median"),
        mean_months=("observed_months", "mean"),
        q25_months=("observed_months", lambda x: x.quantile(0.25)),
        q75_months=("observed_months", lambda x: x.quantile(0.75))
    )
    .reset_index()
)

status_duration["duration_available_pct"] = (
    100
    * status_duration["duration_available"]
    / status_duration["n"]
)

print(
    status_duration.to_string(
        index=False,
        formatters={
            "median_months": "{:.2f}".format,
            "mean_months": "{:.2f}".format,
            "q25_months": "{:.2f}".format,
            "q75_months": "{:.2f}".format,
            "duration_available_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# C. Resolution before contractual term
# ============================================================

print("\n" + "=" * 110)
print("C. STATUS × RESOLUTION BEFORE CONTRACTUAL TERM")
print("=" * 110)

resolved = audit[
    audit["observed_months"].notna()
    & audit["term_months"].notna()
].copy()

early_table = (
    resolved
    .groupby("loan_status")
    .agg(
        n_with_duration=("observed_months", "size"),
        before_term=("before_contract_term", "sum"),
        at_or_after_term=("at_or_after_contract_term", "sum")
    )
    .reset_index()
)

early_table["before_term_pct"] = (
    100
    * early_table["before_term"]
    / early_table["n_with_duration"]
)

early_table["at_or_after_term_pct"] = (
    100
    * early_table["at_or_after_term"]
    / early_table["n_with_duration"]
)

print(
    early_table.to_string(
        index=False,
        formatters={
            "before_term_pct": "{:.3f}".format,
            "at_or_after_term_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# D. Fully Paid specifically by contractual term
# ============================================================

print("\n" + "=" * 110)
print("D. FULLY PAID TIMING BY 36/60-MONTH CONTRACT")
print("=" * 110)

fp = resolved[
    resolved["loan_status"].eq("Fully Paid")
].copy()

fp_table = (
    fp
    .groupby("term_months")
    .agg(
        n=("observed_months", "size"),
        median_months=("observed_months", "median"),
        q25_months=("observed_months", lambda x: x.quantile(0.25)),
        q75_months=("observed_months", lambda x: x.quantile(0.75)),
        early_paid=("before_contract_term", "sum")
    )
    .reset_index()
)

fp_table["early_paid_pct"] = (
    100 * fp_table["early_paid"] / fp_table["n"]
)

print(
    fp_table.to_string(
        index=False,
        formatters={
            "median_months": "{:.2f}".format,
            "q25_months": "{:.2f}".format,
            "q75_months": "{:.2f}".format,
            "early_paid_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# E. Charged Off specifically by contractual term
# ============================================================

print("\n" + "=" * 110)
print("E. CHARGED OFF TIMING BY 36/60-MONTH CONTRACT")
print("=" * 110)

co = resolved[
    resolved["loan_status"].eq("Charged Off")
].copy()

co_table = (
    co
    .groupby("term_months")
    .agg(
        n=("observed_months", "size"),
        median_months=("observed_months", "median"),
        q25_months=("observed_months", lambda x: x.quantile(0.25)),
        q75_months=("observed_months", lambda x: x.quantile(0.75)),
        before_term=("before_contract_term", "sum")
    )
    .reset_index()
)

co_table["before_term_pct"] = (
    100 * co_table["before_term"] / co_table["n"]
)

print(
    co_table.to_string(
        index=False,
        formatters={
            "median_months": "{:.2f}".format,
            "q25_months": "{:.2f}".format,
            "q75_months": "{:.2f}".format,
            "before_term_pct": "{:.3f}".format
        }
    )
)


# ============================================================
# Save
# ============================================================

term_table.to_csv(
    OUTPUT_DIR / "cell13_term_distribution.csv",
    index=False
)

status_duration.to_csv(
    OUTPUT_DIR / "cell13_duration_by_status.csv",
    index=False
)

early_table.to_csv(
    OUTPUT_DIR / "cell13_before_term_by_status.csv",
    index=False
)

fp_table.to_csv(
    OUTPUT_DIR / "cell13_fully_paid_timing.csv",
    index=False
)

co_table.to_csv(
    OUTPUT_DIR / "cell13_charged_off_timing.csv",
    index=False
)

print("\nSaved Cell 13 outputs to:")
print(OUTPUT_DIR)

Auditing contractual follow-up...

Processed 2,703,430 rows

Rows: 2,703,430

A. CONTRACTUAL TERM DISTRIBUTION
 term_months       n       pct
          36 1901819 70.348372
          60  801611 29.651628

B. OBSERVED MONTHS TO LAST PAYMENT BY LOAN STATUS
       loan_status       n  duration_available median_months mean_months q25_months q75_months duration_available_pct
       Charged Off  177884              175144         13.00       15.08       7.00      22.00                 98.460
           Current 1704854             1704854          8.00       10.69       5.00      12.00                100.000
           Default     122                 121          9.00       12.69       8.00      11.00                 99.180
        Fully Paid  765339              765339         18.00       19.29       8.00      32.00                100.000
   In Grace Period   16615               16615         10.00       12.75       6.00      13.00                100.000
 Late (16-30 days)    7510           

In [18]:
# ============================================================
# CELL 14 — Audit missing event timing for terminal outcomes
#
# Purpose:
#   Investigate loans classified as:
#       Charged Off
#       Default
#       Fully Paid
#   whose raw last_pymnt_d is missing.
#
# We need this before constructing competing-risk event times.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_date",
    "loan_status",
    "term",
    "last_pymnt_d",
    "last_date",
    "last_credit_pull_d",
    "next_pymnt_d",
    "recoveries",
    "collection_recovery_fee",
    "out_prncp",
    "total_rec_prncp"
]

CHUNK = 200_000

rows = []

print("Auditing missing terminal-event dates...\n")

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    total = 0

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        total += len(df)

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        raw_last = (
            df["last_pymnt_d"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        terminal = status.isin([
            "Fully Paid",
            "Charged Off",
            "Default"
        ])

        missing_last = raw_last.eq("")

        keep = terminal & missing_last

        if keep.any():

            temp = df.loc[keep].copy()

            temp["loan_status"] = status.loc[keep]

            rows.append(temp)

        print(
            f"Processed {total:,} rows",
            end="\r"
        )

if rows:
    missing_terminal = pd.concat(
        rows,
        ignore_index=True
    )
else:
    missing_terminal = pd.DataFrame(columns=COLS)

print("\n\n" + "=" * 110)
print("A. TERMINAL LOANS WITH MISSING RAW last_pymnt_d")
print("=" * 110)

print("Total:", f"{len(missing_terminal):,}")

if len(missing_terminal):

    print(
        missing_terminal["loan_status"]
        .value_counts()
        .to_string()
    )


# ============================================================
# Parse alternative dates
# ============================================================

def parse_month(series):

    raw = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )

    result = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    m2 = raw.str.match(
        r"^[A-Za-z]{3}-\d{2}$",
        na=False
    )

    if m2.any():
        result.loc[m2] = pd.to_datetime(
            raw.loc[m2],
            format="%b-%y",
            errors="coerce"
        )

    m4 = raw.str.match(
        r"^[A-Za-z]{3}-\d{4}$",
        na=False
    )

    if m4.any():
        result.loc[m4] = pd.to_datetime(
            raw.loc[m4],
            format="%b-%Y",
            errors="coerce"
        )

    return result


if len(missing_terminal):

    missing_terminal["issue_dt"] = pd.to_datetime(
        missing_terminal["issue_date"],
        errors="coerce"
    )

    missing_terminal["credit_pull_dt"] = parse_month(
        missing_terminal["last_credit_pull_d"]
    )

    missing_terminal["next_payment_dt"] = parse_month(
        missing_terminal["next_pymnt_d"]
    )

    # --------------------------------------------------------
    # Coverage of possible alternative timing fields
    # --------------------------------------------------------

    print("\n" + "=" * 110)
    print("B. ALTERNATIVE DATE COVERAGE")
    print("=" * 110)

    alt_summary = (
        missing_terminal
        .groupby("loan_status")
        .agg(
            n=("loan_status", "size"),

            credit_pull_available=(
                "credit_pull_dt",
                lambda x: x.notna().sum()
            ),

            next_payment_available=(
                "next_payment_dt",
                lambda x: x.notna().sum()
            ),

            recoveries_positive=(
                "recoveries",
                lambda x: (
                    pd.to_numeric(
                        x,
                        errors="coerce"
                    ).fillna(0) > 0
                ).sum()
            )
        )
        .reset_index()
    )

    alt_summary["credit_pull_pct"] = (
        100
        * alt_summary["credit_pull_available"]
        / alt_summary["n"]
    )

    alt_summary["next_payment_pct"] = (
        100
        * alt_summary["next_payment_available"]
        / alt_summary["n"]
    )

    print(
        alt_summary.to_string(
            index=False,
            formatters={
                "credit_pull_pct": "{:.3f}".format,
                "next_payment_pct": "{:.3f}".format
            }
        )
    )


    # --------------------------------------------------------
    # Timing from origination to alternative dates
    # --------------------------------------------------------

    missing_terminal["months_to_credit_pull"] = (
        (
            missing_terminal["credit_pull_dt"].dt.year
            - missing_terminal["issue_dt"].dt.year
        ) * 12
        +
        (
            missing_terminal["credit_pull_dt"].dt.month
            - missing_terminal["issue_dt"].dt.month
        )
    )

    print("\n" + "=" * 110)
    print("C. MONTHS FROM ORIGINATION TO LAST CREDIT PULL")
    print("=" * 110)

    credit_timing = (
        missing_terminal
        .groupby("loan_status")
        ["months_to_credit_pull"]
        .agg(
            [
                "count",
                "median",
                "mean",
                "min",
                "max"
            ]
        )
    )

    print(credit_timing.to_string())


    # --------------------------------------------------------
    # Show a small diagnostic sample
    # --------------------------------------------------------

    print("\n" + "=" * 110)
    print("D. SAMPLE OF MISSING-EVENT-DATE TERMINAL LOANS")
    print("=" * 110)

    show_cols = [
        "issue_date",
        "loan_status",
        "term",
        "last_pymnt_d",
        "last_date",
        "last_credit_pull_d",
        "next_pymnt_d",
        "recoveries",
        "collection_recovery_fee",
        "out_prncp",
        "total_rec_prncp"
    ]

    print(
        missing_terminal[
            show_cols
        ].head(30).to_string(index=False)
    )

else:

    alt_summary = pd.DataFrame()
    credit_timing = pd.DataFrame()


# ============================================================
# Save
# ============================================================

missing_terminal.to_csv(
    OUTPUT_DIR /
    "cell14_terminal_missing_last_payment.csv",
    index=False
)

if len(alt_summary):
    alt_summary.to_csv(
        OUTPUT_DIR /
        "cell14_alternative_date_coverage.csv",
        index=False
    )

if len(credit_timing):
    credit_timing.to_csv(
        OUTPUT_DIR /
        "cell14_credit_pull_timing.csv"
    )

print("\nSaved Cell 14 outputs to:")
print(OUTPUT_DIR)

Auditing missing terminal-event dates...

Processed 2,703,430 rows

A. TERMINAL LOANS WITH MISSING RAW last_pymnt_d
Total: 2,741
loan_status
Charged Off    2740
Default           1

B. ALTERNATIVE DATE COVERAGE
loan_status    n  credit_pull_available  next_payment_available  recoveries_positive credit_pull_pct next_payment_pct
Charged Off 2740                   2740                       0                 1160         100.000            0.000
    Default    1                      1                       1                    0         100.000          100.000

C. MONTHS FROM ORIGINATION TO LAST CREDIT PULL
             count  median       mean  min  max
loan_status                                    
Charged Off   2740     7.0  11.212409   -1   56
Default          1     5.0   5.000000    5    5

D. SAMPLE OF MISSING-EVENT-DATE TERMINAL LOANS
issue_date loan_status       term last_pymnt_d  last_date last_credit_pull_d next_pymnt_d  recoveries  collection_recovery_fee  out_prncp  total_re

In [19]:
# ============================================================
# CELL 15 — Infer snapshot / administrative horizon
# from unresolved loans
#
# Purpose:
#   Examine next scheduled payment dates among Current,
#   Late, Grace and Default loans to identify whether the
#   dataset represents a common observation snapshot.
#
# No survival model is fitted here.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_date",
    "loan_status",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

CHUNK = 200_000

UNRESOLVED = [
    "Current",
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)",
    "Default"
]

monthly_counts = {}
last_month_counts = {}

total_unresolved = 0


def parse_month(series):

    raw = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )

    out = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    m2 = raw.str.match(
        r"^[A-Za-z]{3}-\d{2}$",
        na=False
    )

    if m2.any():
        out.loc[m2] = pd.to_datetime(
            raw.loc[m2],
            format="%b-%y",
            errors="coerce"
        )

    m4 = raw.str.match(
        r"^[A-Za-z]{3}-\d{4}$",
        na=False
    )

    if m4.any():
        out.loc[m4] = pd.to_datetime(
            raw.loc[m4],
            format="%b-%Y",
            errors="coerce"
        )

    return out


print("Scanning unresolved-loan snapshot structure...\n")

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    total = 0

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        total += len(df)

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        keep = status.isin(UNRESOLVED)

        if not keep.any():
            continue

        x = df.loc[keep].copy()
        x["status_clean"] = status.loc[keep]

        x["next_dt"] = parse_month(
            x["next_pymnt_d"]
        )

        x["last_dt"] = parse_month(
            x["last_pymnt_d"]
        )

        total_unresolved += len(x)

        # ----------------------------------------------
        # Next-payment counts by status × month
        # ----------------------------------------------

        g = (
            x.dropna(subset=["next_dt"])
            .groupby(
                ["status_clean", "next_dt"]
            )
            .size()
        )

        for key, n in g.items():
            monthly_counts[key] = (
                monthly_counts.get(key, 0)
                + int(n)
            )

        # ----------------------------------------------
        # Last-payment counts by status × month
        # ----------------------------------------------

        g2 = (
            x.dropna(subset=["last_dt"])
            .groupby(
                ["status_clean", "last_dt"]
            )
            .size()
        )

        for key, n in g2.items():
            last_month_counts[key] = (
                last_month_counts.get(key, 0)
                + int(n)
            )

        print(
            f"Processed {total:,} rows",
            end="\r"
        )


# ============================================================
# Build tables
# ============================================================

next_table = pd.DataFrame(
    [
        {
            "loan_status": status,
            "next_payment_month": month,
            "n": n
        }
        for (status, month), n
        in monthly_counts.items()
    ]
)

last_table = pd.DataFrame(
    [
        {
            "loan_status": status,
            "last_payment_month": month,
            "n": n
        }
        for (status, month), n
        in last_month_counts.items()
    ]
)


print("\n\n" + "=" * 110)
print("A. UNRESOLVED LOANS")
print("=" * 110)

print(
    "Total unresolved:",
    f"{total_unresolved:,}"
)


# ============================================================
# Overall next-payment distribution
# ============================================================

overall_next = (
    next_table
    .groupby("next_payment_month")["n"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

overall_next["pct_unresolved"] = (
    100
    * overall_next["n"]
    / total_unresolved
)

print("\n" + "=" * 110)
print("B. TOP 20 NEXT-PAYMENT MONTHS — ALL UNRESOLVED")
print("=" * 110)

print(
    overall_next.head(20).to_string(
        index=False,
        formatters={
            "pct_unresolved": "{:.4f}".format
        }
    )
)


# ============================================================
# Current loans only
# ============================================================

current_next = (
    next_table[
        next_table["loan_status"].eq("Current")
    ]
    .sort_values(
        "n",
        ascending=False
    )
    .copy()
)

current_total = current_next["n"].sum()

current_next["pct_current"] = (
    100
    * current_next["n"]
    / current_total
)

print("\n" + "=" * 110)
print("C. TOP 20 NEXT-PAYMENT MONTHS — CURRENT LOANS")
print("=" * 110)

print(
    current_next.head(20).to_string(
        index=False,
        formatters={
            "pct_current": "{:.4f}".format
        }
    )
)


# ============================================================
# Status-specific latest dates
# ============================================================

status_horizon = (
    next_table
    .groupby("loan_status")
    .agg(
        n_with_next=("n", "sum"),
        earliest_next=(
            "next_payment_month",
            "min"
        ),
        latest_next=(
            "next_payment_month",
            "max"
        )
    )
    .reset_index()
)

last_horizon = (
    last_table
    .groupby("loan_status")
    .agg(
        n_with_last=("n", "sum"),
        earliest_last=(
            "last_payment_month",
            "min"
        ),
        latest_last=(
            "last_payment_month",
            "max"
        )
    )
    .reset_index()
)

status_horizon = status_horizon.merge(
    last_horizon,
    on="loan_status",
    how="outer"
)

print("\n" + "=" * 110)
print("D. STATUS-SPECIFIC PAYMENT HORIZONS")
print("=" * 110)

print(
    status_horizon.to_string(
        index=False
    )
)


# ============================================================
# Recent months in chronological order
# ============================================================

recent_next = (
    overall_next
    .sort_values(
        "next_payment_month"
    )
    .tail(18)
)

print("\n" + "=" * 110)
print("E. MOST RECENT 18 NEXT-PAYMENT MONTHS")
print("=" * 110)

print(
    recent_next.to_string(
        index=False,
        formatters={
            "pct_unresolved": "{:.4f}".format
        }
    )
)


# ============================================================
# Save
# ============================================================

overall_next.to_csv(
    OUTPUT_DIR /
    "cell15_overall_next_payment_months.csv",
    index=False
)

current_next.to_csv(
    OUTPUT_DIR /
    "cell15_current_next_payment_months.csv",
    index=False
)

status_horizon.to_csv(
    OUTPUT_DIR /
    "cell15_unresolved_payment_horizons.csv",
    index=False
)

print("\nSaved Cell 15 outputs to:")
print(OUTPUT_DIR)

Scanning unresolved-loan snapshot structure...

Processed 2,703,430 rows

A. UNRESOLVED LOANS
Total unresolved: 1,760,207

B. TOP 20 NEXT-PAYMENT MONTHS — ALL UNRESOLVED
next_payment_month      n pct_unresolved
        2018-09-01 501358        28.4829
        2017-05-01 466886        26.5245
        2019-06-01 431495        24.5139
        2020-05-01 317978        18.0648
        2020-07-01  29983         1.7034
        2020-08-01   7152         0.4063
        2020-06-01   4408         0.2504
        2020-04-01    482         0.0274
        2017-04-01    155         0.0088
        2018-10-01     90         0.0051
        2019-07-01     78         0.0044
        2019-05-01     52         0.0030
        2017-06-01     31         0.0018
        2018-08-01     31         0.0018
        2020-03-01     15         0.0009
        2020-09-01      7         0.0004
        2020-02-01      2         0.0001
        2018-07-01      1         0.0001
        2018-05-01      1         0.0001
        20

In [20]:
# ============================================================
# CELL 16 — Validate loan-specific snapshot censoring
#
# Purpose:
#   Test whether next_pymnt_d - 1 month can defensibly
#   represent the observation/snapshot month for unresolved
#   loans.
#
# We examine:
#   1. next payment minus last payment
#   2. dominant snapshot groups
#   3. consistency by loan status
#
# No survival model yet.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_date",
    "loan_status",
    "last_pymnt_d",
    "next_pymnt_d"
]

CHUNK = 200_000

UNRESOLVED = [
    "Current",
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)",
    "Default"
]

parts = []


def parse_month(series):

    raw = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )

    out = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    m2 = raw.str.match(
        r"^[A-Za-z]{3}-\d{2}$",
        na=False
    )

    if m2.any():
        out.loc[m2] = pd.to_datetime(
            raw.loc[m2],
            format="%b-%y",
            errors="coerce"
        )

    m4 = raw.str.match(
        r"^[A-Za-z]{3}-\d{4}$",
        na=False
    )

    if m4.any():
        out.loc[m4] = pd.to_datetime(
            raw.loc[m4],
            format="%b-%Y",
            errors="coerce"
        )

    return out


print("Validating snapshot structure...\n")

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    total = 0

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        total += len(df)

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        keep = status.isin(UNRESOLVED)

        if not keep.any():
            continue

        x = pd.DataFrame({
            "loan_status": status.loc[keep],
            "issue_date": pd.to_datetime(
                df.loc[keep, "issue_date"],
                errors="coerce"
            ),
            "last_dt": parse_month(
                df.loc[keep, "last_pymnt_d"]
            ),
            "next_dt": parse_month(
                df.loc[keep, "next_pymnt_d"]
            )
        })

        # Difference in calendar months:
        # next scheduled payment minus last observed payment
        x["next_minus_last_months"] = (
            (x["next_dt"].dt.year - x["last_dt"].dt.year) * 12
            +
            (x["next_dt"].dt.month - x["last_dt"].dt.month)
        )

        # Candidate observation month
        x["candidate_snapshot"] = (
            x["next_dt"]
            - pd.offsets.MonthBegin(1)
        )

        # Months from origination to candidate snapshot
        x["followup_to_snapshot"] = (
            (
                x["candidate_snapshot"].dt.year
                - x["issue_date"].dt.year
            ) * 12
            +
            (
                x["candidate_snapshot"].dt.month
                - x["issue_date"].dt.month
            )
        )

        parts.append(x)

        print(
            f"Processed {total:,} rows",
            end="\r"
        )


audit = pd.concat(
    parts,
    ignore_index=True
)

print("\n\nRows in unresolved audit:", f"{len(audit):,}")


# ============================================================
# A. Difference between next and last payment
# ============================================================

print("\n" + "=" * 110)
print("A. NEXT PAYMENT MONTH MINUS LAST PAYMENT MONTH")
print("=" * 110)

gap_table = (
    audit["next_minus_last_months"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("month_gap")
    .reset_index(name="n")
)

gap_table["pct"] = (
    100 * gap_table["n"] / len(audit)
)

print(
    gap_table.head(30).to_string(
        index=False,
        formatters={"pct": "{:.4f}".format}
    )
)


# ============================================================
# B. Exact one-month relationship
# ============================================================

valid_gap = audit[
    audit["next_minus_last_months"].notna()
].copy()

one_month = (
    valid_gap["next_minus_last_months"] == 1
)

print("\n" + "=" * 110)
print("B. ONE-MONTH CONSISTENCY")
print("=" * 110)

print(
    "Valid last+next dates:",
    f"{len(valid_gap):,}"
)

print(
    "Exactly next = last + 1 month:",
    f"{one_month.sum():,}",
    f"({100 * one_month.mean():.4f}%)"
)


# ============================================================
# C. Consistency by status
# ============================================================

print("\n" + "=" * 110)
print("C. ONE-MONTH CONSISTENCY BY STATUS")
print("=" * 110)

status_gap = (
    valid_gap
    .assign(
        exactly_one_month=(
            valid_gap[
                "next_minus_last_months"
            ] == 1
        )
    )
    .groupby("loan_status")
    .agg(
        n=("next_minus_last_months", "size"),
        exactly_one_month=(
            "exactly_one_month",
            "sum"
        ),
        median_gap=(
            "next_minus_last_months",
            "median"
        ),
        min_gap=(
            "next_minus_last_months",
            "min"
        ),
        max_gap=(
            "next_minus_last_months",
            "max"
        )
    )
    .reset_index()
)

status_gap["exactly_one_month_pct"] = (
    100
    * status_gap["exactly_one_month"]
    / status_gap["n"]
)

print(
    status_gap.to_string(
        index=False,
        formatters={
            "exactly_one_month_pct": "{:.4f}".format
        }
    )
)


# ============================================================
# D. Candidate snapshot distribution
# ============================================================

print("\n" + "=" * 110)
print("D. DOMINANT CANDIDATE SNAPSHOT MONTHS")
print("=" * 110)

snapshot_table = (
    audit["candidate_snapshot"]
    .value_counts(dropna=False)
    .rename_axis("candidate_snapshot")
    .reset_index(name="n")
)

snapshot_table["pct"] = (
    100
    * snapshot_table["n"]
    / len(audit)
)

print(
    snapshot_table.head(20).to_string(
        index=False,
        formatters={"pct": "{:.4f}".format}
    )
)


# ============================================================
# E. Snapshot × status
# ============================================================

print("\n" + "=" * 110)
print("E. TOP SNAPSHOT MONTHS × LOAN STATUS")
print("=" * 110)

top_snapshots = (
    snapshot_table
    .dropna(subset=["candidate_snapshot"])
    .head(8)["candidate_snapshot"]
    .tolist()
)

snapshot_status = (
    audit[
        audit["candidate_snapshot"].isin(
            top_snapshots
        )
    ]
    .groupby(
        ["candidate_snapshot", "loan_status"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print(snapshot_status.to_string())


# ============================================================
# F. Follow-up sanity
# ============================================================

print("\n" + "=" * 110)
print("F. FOLLOW-UP TO CANDIDATE SNAPSHOT")
print("=" * 110)

print(
    audit["followup_to_snapshot"]
    .describe(
        percentiles=[
            .01, .05, .25, .5, .75, .95, .99
        ]
    )
    .to_string()
)

print(
    "\nNegative follow-up:",
    int(
        (
            audit["followup_to_snapshot"] < 0
        ).sum()
    )
)


# ============================================================
# Save
# ============================================================

gap_table.to_csv(
    OUTPUT_DIR /
    "cell16_next_last_gap.csv",
    index=False
)

status_gap.to_csv(
    OUTPUT_DIR /
    "cell16_gap_by_status.csv",
    index=False
)

snapshot_table.to_csv(
    OUTPUT_DIR /
    "cell16_candidate_snapshots.csv",
    index=False
)

snapshot_status.to_csv(
    OUTPUT_DIR /
    "cell16_snapshot_by_status.csv"
)

print("\nSaved Cell 16 outputs to:")
print(OUTPUT_DIR)

Validating snapshot structure...

Processed 2,703,430 rows

Rows in unresolved audit: 1,760,207

A. NEXT PAYMENT MONTH MINUS LAST PAYMENT MONTH
 month_gap       n     pct
       0.0     610  0.0347
       1.0 1629602 92.5801
       2.0  105982  6.0210
       3.0    9431  0.5358
       4.0    8291  0.4710
       5.0    5836  0.3316
       6.0      28  0.0016
       NaN     427  0.0243

B. ONE-MONTH CONSISTENCY
Valid last+next dates: 1,759,780
Exactly next = last + 1 month: 1,629,602 (92.6026%)

C. ONE-MONTH CONSISTENCY BY STATUS
       loan_status       n  exactly_one_month  median_gap  min_gap  max_gap exactly_one_month_pct
           Current 1704854            1618498         1.0      0.0      5.0               94.9347
           Default     121                 18         5.0      1.0      6.0               14.8760
   In Grace Period   16615               6546         2.0      0.0      3.0               39.3981
 Late (16-30 days)    7504               1462         2.0      1.0      3.

In [21]:
# ============================================================
# CELL 17 — Construct competing-risk outcome dataset
#
# Event definitions:
#   0 = administratively censored / unresolved
#   1 = Charged Off
#   2 = Fully Paid (competing event)
#
# Primary timing:
#   Charged Off / Fully Paid -> raw last_pymnt_d
#   Unresolved               -> next_pymnt_d - 1 month
#
# IMPORTANT:
#   Default / Late / Grace / Current are not treated as
#   terminal adverse events in this time-to-event analysis.
#
# No survival model is fitted in this cell.
# ============================================================

import pandas as pd
import numpy as np

COLS = [
    "issue_date",
    "loan_status",
    "term",
    "last_pymnt_d",
    "next_pymnt_d"
]

CHUNK = 200_000

UNRESOLVED = [
    "Current",
    "Default",
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)"
]

parts = []


def parse_month(series):

    raw = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )

    out = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    m2 = raw.str.match(
        r"^[A-Za-z]{3}-\d{2}$",
        na=False
    )

    if m2.any():
        out.loc[m2] = pd.to_datetime(
            raw.loc[m2],
            format="%b-%y",
            errors="coerce"
        )

    m4 = raw.str.match(
        r"^[A-Za-z]{3}-\d{4}$",
        na=False
    )

    if m4.any():
        out.loc[m4] = pd.to_datetime(
            raw.loc[m4],
            format="%b-%Y",
            errors="coerce"
        )

    return out


print("Constructing competing-risk dataset...\n")

with pd.read_stata(
    DTA_PATH,
    columns=COLS,
    convert_categoricals=False,
    iterator=True
) as reader:

    total = 0

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        total += len(df)

        issue = pd.to_datetime(
            df["issue_date"],
            errors="coerce"
        )

        status = (
            df["loan_status"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        last_dt = parse_month(
            df["last_pymnt_d"]
        )

        next_dt = parse_month(
            df["next_pymnt_d"]
        )

        term_months = pd.to_numeric(
            df["term"]
            .fillna("")
            .astype(str)
            .str.extract(
                r"(\d+)",
                expand=False
            ),
            errors="coerce"
        )

        # ----------------------------------------------------
        # Event type
        # ----------------------------------------------------

        event_type = pd.Series(
            np.nan,
            index=df.index,
            dtype="float64"
        )

        # Charged Off with known timing
        co_valid = (
            status.eq("Charged Off")
            & last_dt.notna()
        )

        event_type.loc[co_valid] = 1

        # Fully Paid with known timing
        fp_valid = (
            status.eq("Fully Paid")
            & last_dt.notna()
        )

        event_type.loc[fp_valid] = 2

        # Unresolved statuses with next-payment date
        unresolved_valid = (
            status.isin(UNRESOLVED)
            & next_dt.notna()
        )

        event_type.loc[
            unresolved_valid
        ] = 0

        # ----------------------------------------------------
        # Analysis end date
        # ----------------------------------------------------

        end_date = pd.Series(
            pd.NaT,
            index=df.index,
            dtype="datetime64[ns]"
        )

        end_date.loc[
            co_valid | fp_valid
        ] = last_dt.loc[
            co_valid | fp_valid
        ]

        # Candidate administrative snapshot:
        # next scheduled payment minus one month
        end_date.loc[
            unresolved_valid
        ] = (
            next_dt.loc[
                unresolved_valid
            ]
            - pd.offsets.MonthBegin(1)
        )

        # ----------------------------------------------------
        # Duration in calendar months
        # ----------------------------------------------------

        duration_months = (
            (end_date.dt.year - issue.dt.year) * 12
            +
            (end_date.dt.month - issue.dt.month)
        )

        temp = pd.DataFrame({
            "issue_date": issue,
            "loan_status": status,
            "term_months": term_months,
            "last_dt": last_dt,
            "next_dt": next_dt,
            "end_date": end_date,
            "duration_months": duration_months,
            "event_type": event_type
        })

        parts.append(temp)

        print(
            f"Processed {total:,} rows",
            end="\r"
        )


cr = pd.concat(
    parts,
    ignore_index=True
)

print(
    "\n\nTotal source rows:",
    f"{len(cr):,}"
)


# ============================================================
# A. Inclusion / exclusion
# ============================================================

print("\n" + "=" * 110)
print("A. PRIMARY COMPETING-RISK DATASET INCLUSION")
print("=" * 110)

included = (
    cr["event_type"].notna()
    & cr["duration_months"].notna()
    & (cr["duration_months"] >= 0)
)

print(
    "Included:",
    f"{included.sum():,}",
    f"({100 * included.mean():.4f}%)"
)

print(
    "Excluded:",
    f"{(~included).sum():,}",
    f"({100 * (~included).mean():.4f}%)"
)


# ============================================================
# B. Event counts
# ============================================================

analysis = cr.loc[
    included
].copy()

event_labels = {
    0.0: "Censored/unresolved",
    1.0: "Charged Off",
    2.0: "Fully Paid"
}

analysis["event_label"] = (
    analysis["event_type"]
    .map(event_labels)
)

event_table = (
    analysis["event_label"]
    .value_counts()
    .rename_axis("event")
    .reset_index(name="n")
)

event_table["pct"] = (
    100
    * event_table["n"]
    / len(analysis)
)

print("\n" + "=" * 110)
print("B. EVENT DISTRIBUTION")
print("=" * 110)

print(
    event_table.to_string(
        index=False,
        formatters={
            "pct": "{:.4f}".format
        }
    )
)


# ============================================================
# C. Excluded rows by status
# ============================================================

print("\n" + "=" * 110)
print("C. EXCLUDED ROWS BY ORIGINAL STATUS")
print("=" * 110)

excluded_status = (
    cr.loc[
        ~included,
        "loan_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .reset_index(name="n")
)

print(
    excluded_status.to_string(
        index=False
    )
)


# ============================================================
# D. Duration by event type
# ============================================================

print("\n" + "=" * 110)
print("D. FOLLOW-UP DURATION BY EVENT TYPE")
print("=" * 110)

duration_table = (
    analysis
    .groupby("event_label")
    ["duration_months"]
    .agg(
        [
            "count",
            "median",
            "mean",
            "min",
            "max"
        ]
    )
    .reset_index()
)

print(
    duration_table.to_string(
        index=False,
        formatters={
            "median": "{:.2f}".format,
            "mean": "{:.2f}".format
        }
    )
)


# ============================================================
# E. Sanity checks
# ============================================================

print("\n" + "=" * 110)
print("E. SANITY CHECKS")
print("=" * 110)

print(
    "Negative durations:",
    int(
        (
            analysis[
                "duration_months"
            ] < 0
        ).sum()
    )
)

print(
    "Zero-month durations:",
    int(
        (
            analysis[
                "duration_months"
            ] == 0
        ).sum()
    )
)

print(
    "Missing issue dates:",
    int(
        analysis[
            "issue_date"
        ].isna().sum()
    )
)

print(
    "Missing end dates:",
    int(
        analysis[
            "end_date"
        ].isna().sum()
    )
)


# ============================================================
# F. Event counts by contractual term
# ============================================================

print("\n" + "=" * 110)
print("F. EVENT TYPE × CONTRACTUAL TERM")
print("=" * 110)

term_event = pd.crosstab(
    analysis["term_months"],
    analysis["event_label"],
    margins=True
)

print(term_event.to_string())


# ============================================================
# G. Event counts by origination year
# ============================================================

analysis["issue_year"] = (
    analysis["issue_date"].dt.year
)

print("\n" + "=" * 110)
print("G. EVENT TYPE × ORIGINATION YEAR")
print("=" * 110)

year_event = pd.crosstab(
    analysis["issue_year"],
    analysis["event_label"],
    margins=True
)

print(year_event.to_string())


# ============================================================
# Save compact outcome dataset
# ============================================================

analysis_out = analysis[
    [
        "issue_date",
        "issue_year",
        "loan_status",
        "term_months",
        "last_dt",
        "next_dt",
        "end_date",
        "duration_months",
        "event_type",
        "event_label"
    ]
].copy()

analysis_out.to_parquet(
    OUTPUT_DIR /
    "cell17_competing_risk_outcomes.parquet",
    index=False
)

event_table.to_csv(
    OUTPUT_DIR /
    "cell17_event_distribution.csv",
    index=False
)

excluded_status.to_csv(
    OUTPUT_DIR /
    "cell17_excluded_status.csv",
    index=False
)

duration_table.to_csv(
    OUTPUT_DIR /
    "cell17_duration_by_event.csv",
    index=False
)

term_event.to_csv(
    OUTPUT_DIR /
    "cell17_event_by_term.csv"
)

year_event.to_csv(
    OUTPUT_DIR /
    "cell17_event_by_issue_year.csv"
)

print("\nSaved Cell 17 outputs to:")
print(OUTPUT_DIR)

Constructing competing-risk dataset...

Processed 2,703,430 rows

Total source rows: 2,703,430

A. PRIMARY COMPETING-RISK DATASET INCLUSION
Included: 2,700,690 (99.8986%)
Excluded: 2,740 (0.1014%)

B. EVENT DISTRIBUTION
              event       n     pct
Censored/unresolved 1760207 65.1762
         Fully Paid  765339 28.3386
        Charged Off  175144  6.4852

C. EXCLUDED ROWS BY ORIGINAL STATUS
loan_status    n
Charged Off 2740

D. FOLLOW-UP DURATION BY EVENT TYPE
        event_label   count median  mean  min  max
Censored/unresolved 1760207   9.00 10.84  1.0 56.0
        Charged Off  175144  13.00 15.08  0.0 52.0
         Fully Paid  765339  18.00 19.29  0.0 56.0

E. SANITY CHECKS
Negative durations: 0
Zero-month durations: 12304
Missing issue dates: 0
Missing end dates: 0

F. EVENT TYPE × CONTRACTUAL TERM
event_label  Censored/unresolved  Charged Off  Fully Paid      All
term_months                                                       
36                       1183206       10497

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [22]:
# ============================================================
# CELL 18 — Audit zero-month outcomes + finish Cell 17 saving
#
# IMPORTANT:
#   Do NOT rerun Cell 17.
#   This uses the existing `analysis` object already in memory.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Confirm Cell 17 objects still exist
# ------------------------------------------------------------

if "analysis" not in globals():
    raise RuntimeError(
        "The `analysis` object is not in memory. "
        "Do not continue; tell ChatGPT before rerunning anything."
    )

print(
    "Competing-risk analysis rows in memory:",
    f"{len(analysis):,}"
)


# ============================================================
# A. Zero-month outcomes
# ============================================================

zero = analysis[
    analysis["duration_months"].eq(0)
].copy()

print("\n" + "=" * 110)
print("A. ZERO-MONTH OUTCOMES")
print("=" * 110)

print("Total:", f"{len(zero):,}")

zero_status = (
    zero
    .groupby(
        ["event_label", "loan_status"]
    )
    .size()
    .reset_index(name="n")
    .sort_values(
        "n",
        ascending=False
    )
)

print(
    zero_status.to_string(
        index=False
    )
)


# ============================================================
# B. Zero-month outcomes by issue year
# ============================================================

print("\n" + "=" * 110)
print("B. ZERO-MONTH OUTCOMES BY ISSUE YEAR")
print("=" * 110)

zero["issue_year"] = (
    zero["issue_date"].dt.year
)

zero_year = pd.crosstab(
    zero["issue_year"],
    zero["event_label"],
    margins=True
)

print(zero_year.to_string())


# ============================================================
# C. Zero-month outcomes by term
# ============================================================

print("\n" + "=" * 110)
print("C. ZERO-MONTH OUTCOMES BY CONTRACTUAL TERM")
print("=" * 110)

zero_term = pd.crosstab(
    zero["term_months"],
    zero["event_label"],
    margins=True
)

print(zero_term.to_string())


# ============================================================
# D. Sample zero-month records
# ============================================================

print("\n" + "=" * 110)
print("D. SAMPLE ZERO-MONTH RECORDS")
print("=" * 110)

show_cols = [
    "issue_date",
    "loan_status",
    "term_months",
    "last_dt",
    "next_dt",
    "end_date",
    "duration_months",
    "event_type",
    "event_label"
]

print(
    zero[
        show_cols
    ].head(40).to_string(
        index=False
    )
)


# ============================================================
# E. Create compact analysis output
# ============================================================

analysis_out = analysis[
    [
        "issue_date",
        "issue_year",
        "loan_status",
        "term_months",
        "last_dt",
        "next_dt",
        "end_date",
        "duration_months",
        "event_type",
        "event_label"
    ]
].copy()


# ============================================================
# F. Save without parquet dependency
# ============================================================

csv_path = (
    OUTPUT_DIR /
    "cell17_competing_risk_outcomes.csv.gz"
)

analysis_out.to_csv(
    csv_path,
    index=False,
    compression="gzip"
)

event_table.to_csv(
    OUTPUT_DIR /
    "cell17_event_distribution.csv",
    index=False
)

excluded_status.to_csv(
    OUTPUT_DIR /
    "cell17_excluded_status.csv",
    index=False
)

duration_table.to_csv(
    OUTPUT_DIR /
    "cell17_duration_by_event.csv",
    index=False
)

term_event.to_csv(
    OUTPUT_DIR /
    "cell17_event_by_term.csv"
)

year_event.to_csv(
    OUTPUT_DIR /
    "cell17_event_by_issue_year.csv"
)

zero_status.to_csv(
    OUTPUT_DIR /
    "cell18_zero_month_by_status.csv",
    index=False
)

zero_year.to_csv(
    OUTPUT_DIR /
    "cell18_zero_month_by_year.csv"
)

zero_term.to_csv(
    OUTPUT_DIR /
    "cell18_zero_month_by_term.csv"
)


# ============================================================
# G. Confirm files
# ============================================================

print("\n" + "=" * 110)
print("E. FILE SAVE CONFIRMATION")
print("=" * 110)

print("Main outcome file:")
print(csv_path)

print(
    "\nExists:",
    csv_path.exists()
)

if csv_path.exists():
    print(
        "Compressed size:",
        f"{csv_path.stat().st_size / (1024**2):.2f} MB"
    )

print("\nSaved Cell 18 outputs to:")
print(OUTPUT_DIR)

Competing-risk analysis rows in memory: 2,700,690

A. ZERO-MONTH OUTCOMES
Total: 12,304
event_label loan_status     n
 Fully Paid  Fully Paid 12284
Charged Off Charged Off    20

B. ZERO-MONTH OUTCOMES BY ISSUE YEAR
event_label  Charged Off  Fully Paid    All
issue_year                                 
2013                   0           4      4
2014                   0          31     31
2015                   1        1092   1093
2016                   3        2544   2547
2017                   2        2528   2530
2018                   7        2714   2721
2019                   7        3371   3378
All                   20       12284  12304

C. ZERO-MONTH OUTCOMES BY CONTRACTUAL TERM
event_label  Charged Off  Fully Paid    All
term_months                                
36                    11        8682   8693
60                     9        3602   3611
All                   20       12284  12304

D. SAMPLE ZERO-MONTH RECORDS
issue_date loan_status  term_months    last_dt nex

In [23]:
# ============================================================
# CELL 19 — Nonparametric competing-risk cumulative incidence
#
# Outcomes:
#   1 = Charged Off
#   2 = Fully Paid (competing event)
#   0 = administratively censored / unresolved
#
# Because source timing is monthly:
#   same-month events (duration 0) are represented as month 1.
#
# Outputs:
#   - monthly risk-set/event table
#   - overall survival
#   - cumulative incidence of charge-off
#   - cumulative incidence of full payment
#   - CIF estimates at policy-relevant horizons
#
# No external survival package required.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Start from Cell 17/18 analysis object
# ------------------------------------------------------------

if "analysis" not in globals():
    raise RuntimeError(
        "`analysis` is not available in memory. "
        "Do not continue; tell ChatGPT."
    )

aj = analysis[
    [
        "duration_months",
        "event_type",
        "term_months"
    ]
].copy()


# ------------------------------------------------------------
# Monthly-time correction
#
# 0 calendar-month difference means event occurred during
# the origination month. Represent it as month 1.
# ------------------------------------------------------------

aj["time_month"] = (
    aj["duration_months"]
    .round()
    .astype(int)
    .clip(lower=1)
)

aj["event_type"] = (
    aj["event_type"]
    .astype(int)
)

print(
    "Rows:",
    f"{len(aj):,}"
)

print(
    "Time range:",
    aj["time_month"].min(),
    "to",
    aj["time_month"].max(),
    "months"
)


# ============================================================
# Function: nonparametric competing-risk estimator
# ============================================================

def competing_risk_cif(df):

    # ----------------------------------------------
    # Aggregate counts at each month
    # ----------------------------------------------

    grouped = (
        df.groupby("time_month")
        .agg(
            charged_off=(
                "event_type",
                lambda x: (x == 1).sum()
            ),
            fully_paid=(
                "event_type",
                lambda x: (x == 2).sum()
            ),
            censored=(
                "event_type",
                lambda x: (x == 0).sum()
            ),
            total_exit=(
                "event_type",
                "size"
            )
        )
        .reset_index()
        .sort_values("time_month")
    )

    # ----------------------------------------------
    # Risk set at beginning of each month
    # ----------------------------------------------

    n_total = len(df)

    grouped["risk_set"] = (
        n_total
        - grouped["total_exit"]
        .cumsum()
        .shift(
            fill_value=0
        )
    )

    # ----------------------------------------------
    # Cause-specific discrete hazards
    # ----------------------------------------------

    grouped["hazard_chargeoff"] = (
        grouped["charged_off"]
        / grouped["risk_set"]
    )

    grouped["hazard_fullypaid"] = (
        grouped["fully_paid"]
        / grouped["risk_set"]
    )

    grouped["hazard_any_event"] = (
        (
            grouped["charged_off"]
            + grouped["fully_paid"]
        )
        / grouped["risk_set"]
    )

    # ----------------------------------------------
    # Aalen-Johansen-style recursion
    # ----------------------------------------------

    survival_before = 1.0
    cif_chargeoff = 0.0
    cif_fullypaid = 0.0

    survival_values = []
    cif_co_values = []
    cif_fp_values = []

    for _, row in grouped.iterrows():

        n_risk = row["risk_set"]

        if n_risk <= 0:
            survival_values.append(
                survival_before
            )
            cif_co_values.append(
                cif_chargeoff
            )
            cif_fp_values.append(
                cif_fullypaid
            )
            continue

        h_co = (
            row["charged_off"]
            / n_risk
        )

        h_fp = (
            row["fully_paid"]
            / n_risk
        )

        # CIF increment uses survival immediately
        # before the current time point
        cif_chargeoff += (
            survival_before * h_co
        )

        cif_fullypaid += (
            survival_before * h_fp
        )

        survival_after = (
            survival_before
            * (1.0 - h_co - h_fp)
        )

        survival_values.append(
            survival_after
        )

        cif_co_values.append(
            cif_chargeoff
        )

        cif_fp_values.append(
            cif_fullypaid
        )

        survival_before = survival_after

    grouped["survival"] = (
        survival_values
    )

    grouped["cif_chargeoff"] = (
        cif_co_values
    )

    grouped["cif_fullypaid"] = (
        cif_fp_values
    )

    return grouped


# ============================================================
# A. Overall CIF
# ============================================================

overall_cif = competing_risk_cif(
    aj
)

print("\n" + "=" * 110)
print("A. OVERALL COMPETING-RISK ESTIMATES")
print("=" * 110)


# ------------------------------------------------------------
# Extract selected horizons
# ------------------------------------------------------------

HORIZONS = [
    6,
    12,
    18,
    24,
    36,
    48,
    60
]


def extract_horizons(table, horizons):

    rows = []

    for h in horizons:

        eligible = table[
            table["time_month"] <= h
        ]

        if len(eligible) == 0:
            continue

        r = eligible.iloc[-1]

        rows.append({
            "horizon_months": h,
            "risk_set_at_last_observed_month": int(
                r["risk_set"]
            ),
            "survival": r["survival"],
            "cif_chargeoff": r["cif_chargeoff"],
            "cif_fullypaid": r["cif_fullypaid"]
        })

    return pd.DataFrame(rows)


overall_horizons = extract_horizons(
    overall_cif,
    HORIZONS
)

print(
    overall_horizons.to_string(
        index=False,
        formatters={
            "survival": "{:.6f}".format,
            "cif_chargeoff": "{:.6f}".format,
            "cif_fullypaid": "{:.6f}".format
        }
    )
)


# ============================================================
# B. Separate estimates by contractual term
# ============================================================

print("\n" + "=" * 110)
print("B. COMPETING-RISK ESTIMATES BY CONTRACTUAL TERM")
print("=" * 110)

term_outputs = []
term_horizon_outputs = []

for term in [36, 60]:

    temp = aj[
        aj["term_months"].eq(term)
    ].copy()

    result = competing_risk_cif(
        temp
    )

    result["term_months"] = term

    term_outputs.append(
        result
    )

    term_h = extract_horizons(
        result,
        [
            h for h in HORIZONS
            if h <= term
        ]
    )

    term_h["term_months"] = term

    term_horizon_outputs.append(
        term_h
    )

term_cif = pd.concat(
    term_outputs,
    ignore_index=True
)

term_horizons = pd.concat(
    term_horizon_outputs,
    ignore_index=True
)

print(
    term_horizons[
        [
            "term_months",
            "horizon_months",
            "risk_set_at_last_observed_month",
            "survival",
            "cif_chargeoff",
            "cif_fullypaid"
        ]
    ].to_string(
        index=False,
        formatters={
            "survival": "{:.6f}".format,
            "cif_chargeoff": "{:.6f}".format,
            "cif_fullypaid": "{:.6f}".format
        }
    )
)


# ============================================================
# C. Internal probability check
# ============================================================

print("\n" + "=" * 110)
print("C. INTERNAL PROBABILITY CHECK")
print("=" * 110)

overall_cif[
    "probability_sum"
] = (
    overall_cif["survival"]
    + overall_cif["cif_chargeoff"]
    + overall_cif["cif_fullypaid"]
)

print(
    "Minimum S + CIF_CO + CIF_FP:",
    f"{overall_cif['probability_sum'].min():.12f}"
)

print(
    "Maximum S + CIF_CO + CIF_FP:",
    f"{overall_cif['probability_sum'].max():.12f}"
)

print(
    "Final probability sum:",
    f"{overall_cif['probability_sum'].iloc[-1]:.12f}"
)


# ============================================================
# D. Final observed estimates
# ============================================================

print("\n" + "=" * 110)
print("D. FINAL OBSERVED OVERALL ESTIMATES")
print("=" * 110)

final = overall_cif.iloc[-1]

print(
    "Final observed month:",
    int(final["time_month"])
)

print(
    "Survival / unresolved probability:",
    f"{final['survival']:.6f}"
)

print(
    "Cumulative incidence — Charged Off:",
    f"{final['cif_chargeoff']:.6f}"
)

print(
    "Cumulative incidence — Fully Paid:",
    f"{final['cif_fullypaid']:.6f}"
)


# ============================================================
# Save
# ============================================================

overall_cif.to_csv(
    OUTPUT_DIR /
    "cell19_overall_competing_risk_cif.csv",
    index=False
)

overall_horizons.to_csv(
    OUTPUT_DIR /
    "cell19_overall_cif_horizons.csv",
    index=False
)

term_cif.to_csv(
    OUTPUT_DIR /
    "cell19_term_competing_risk_cif.csv",
    index=False
)

term_horizons.to_csv(
    OUTPUT_DIR /
    "cell19_term_cif_horizons.csv",
    index=False
)

print("\nSaved Cell 19 outputs to:")
print(OUTPUT_DIR)

Rows: 2,700,690
Time range: 1 to 56 months

A. OVERALL COMPETING-RISK ESTIMATES
 horizon_months  risk_set_at_last_observed_month survival cif_chargeoff cif_fullypaid
              6                          2081403 0.918914      0.016093      0.064993
             12                          1092889 0.814891      0.041301      0.143808
             18                           637032 0.688002      0.074998      0.237000
             24                           522416 0.560940      0.106141      0.332919
             36                           221142 0.183299      0.141911      0.674790
             48                            17820 0.088451      0.149502      0.762047
             60                               21 0.067008      0.150039      0.782953

B. COMPETING-RISK ESTIMATES BY CONTRACTUAL TERM
 term_months  horizon_months  risk_set_at_last_observed_month survival cif_chargeoff cif_fullypaid
          36               6                          1464981 0.916804      0.014678

In [24]:
# ============================================================
# CELL 20 — Horizon support / tail-stability audit
#
# Purpose:
#   Before interpreting CIF estimates, quantify:
#     - actual number still under observation at each horizon
#     - cumulative charge-offs
#     - cumulative full-payment competing events
#     - cumulative censoring
#     - whether the requested horizon is actually observed
#
# No new model is fitted.
# ============================================================

import pandas as pd
import numpy as np


if "aj" not in globals():
    raise RuntimeError(
        "`aj` from Cell 19 is not available in memory."
    )


HORIZONS = [
    6,
    12,
    18,
    24,
    30,
    36,
    42,
    48,
    54,
    60
]


def horizon_support(df, horizons, label):

    max_time = int(
        df["time_month"].max()
    )

    n_total = len(df)

    rows = []

    for h in horizons:

        # Number whose observed time reaches at least h.
        # These loans are still represented in the risk set
        # at the beginning of month h.
        at_risk = int(
            (df["time_month"] >= h).sum()
        )

        # Events/censoring observed up through h
        through_h = df[
            df["time_month"] <= h
        ]

        charged_off = int(
            (
                through_h["event_type"] == 1
            ).sum()
        )

        fully_paid = int(
            (
                through_h["event_type"] == 2
            ).sum()
        )

        censored = int(
            (
                through_h["event_type"] == 0
            ).sum()
        )

        rows.append({
            "group": label,
            "requested_horizon": h,
            "max_observed_month": max_time,
            "horizon_observed": h <= max_time,
            "n_total": n_total,
            "n_at_risk_start_horizon": at_risk,
            "pct_original_at_risk": (
                100 * at_risk / n_total
            ),
            "cum_chargeoff_count": charged_off,
            "cum_fullypaid_count": fully_paid,
            "cum_censored_count": censored
        })

    return pd.DataFrame(rows)


# ============================================================
# A. Overall support
# ============================================================

overall_support = horizon_support(
    aj,
    HORIZONS,
    "Overall"
)

print("=" * 120)
print("A. OVERALL HORIZON SUPPORT")
print("=" * 120)

print(
    overall_support.to_string(
        index=False,
        formatters={
            "pct_original_at_risk":
                "{:.4f}".format
        }
    )
)


# ============================================================
# B. By contractual term
# ============================================================

term_support_parts = []

for term in [36, 60]:

    x = aj[
        aj["term_months"].eq(term)
    ].copy()

    term_support_parts.append(
        horizon_support(
            x,
            [
                h for h in HORIZONS
                if h <= term
            ],
            f"{term}-month"
        )
    )

term_support = pd.concat(
    term_support_parts,
    ignore_index=True
)

print("\n" + "=" * 120)
print("B. HORIZON SUPPORT BY CONTRACTUAL TERM")
print("=" * 120)

print(
    term_support.to_string(
        index=False,
        formatters={
            "pct_original_at_risk":
                "{:.4f}".format
        }
    )
)


# ============================================================
# C. Exact event/censoring counts by month in tail
# ============================================================

monthly_tail = (
    aj[
        aj["time_month"] >= 30
    ]
    .groupby("time_month")
    .agg(
        charged_off=(
            "event_type",
            lambda x: (x == 1).sum()
        ),
        fully_paid=(
            "event_type",
            lambda x: (x == 2).sum()
        ),
        censored=(
            "event_type",
            lambda x: (x == 0).sum()
        ),
        total_exit=(
            "event_type",
            "size"
        )
    )
    .reset_index()
)

# Risk set at beginning of each month
monthly_tail["n_at_risk"] = [
    int(
        (
            aj["time_month"] >= t
        ).sum()
    )
    for t in monthly_tail["time_month"]
]

print("\n" + "=" * 120)
print("C. MONTHLY SUPPORT IN THE TAIL (MONTH >= 30)")
print("=" * 120)

print(
    monthly_tail.to_string(
        index=False
    )
)


# ============================================================
# D. Flag thinly supported horizons
# ============================================================

print("\n" + "=" * 120)
print("D. SUPPORT FLAGS")
print("=" * 120)

flag_table = overall_support[
    [
        "requested_horizon",
        "horizon_observed",
        "n_at_risk_start_horizon",
        "pct_original_at_risk"
    ]
].copy()

flag_table["less_than_5pct_at_risk"] = (
    flag_table["pct_original_at_risk"] < 5
)

flag_table["less_than_1pct_at_risk"] = (
    flag_table["pct_original_at_risk"] < 1
)

print(
    flag_table.to_string(
        index=False,
        formatters={
            "pct_original_at_risk":
                "{:.4f}".format
        }
    )
)


# ============================================================
# Save
# ============================================================

overall_support.to_csv(
    OUTPUT_DIR /
    "cell20_overall_horizon_support.csv",
    index=False
)

term_support.to_csv(
    OUTPUT_DIR /
    "cell20_term_horizon_support.csv",
    index=False
)

monthly_tail.to_csv(
    OUTPUT_DIR /
    "cell20_monthly_tail_support.csv",
    index=False
)

flag_table.to_csv(
    OUTPUT_DIR /
    "cell20_horizon_support_flags.csv",
    index=False
)

print("\nSaved Cell 20 outputs to:")
print(OUTPUT_DIR)

A. OVERALL HORIZON SUPPORT
  group  requested_horizon  max_observed_month  horizon_observed  n_total  n_at_risk_start_horizon pct_original_at_risk  cum_chargeoff_count  cum_fullypaid_count  cum_censored_count
Overall                  6                  56              True  2700690                  2081403              77.0693                39962               164474              589043
Overall                 12                  56              True  2700690                  1092889              40.4670                81771               295747             1356279
Overall                 18                  56              True  2700690                   637032              23.5877               115160               388703             1578826
Overall                 24                  56              True  2700690                   522416              19.3438               143134               474863             1578826
Overall                 30                  56              Tru

In [25]:
# ============================================================
# CELL 21 — Pointwise uncertainty for competing-risk CIF
#
# Purpose:
#   Add 95% confidence intervals to the nonparametric
#   cumulative-incidence estimates.
#
# Primary reporting horizons:
#   6, 12, 18, 24, 36 months
#
# 42+ months are intentionally excluded from primary
# inference because Cell 20 showed thin support.
# ============================================================

import pandas as pd
import numpy as np


if "aj" not in globals():
    raise RuntimeError(
        "`aj` from Cell 19 is not available in memory."
    )


# ============================================================
# CIF + approximate Aalen-Johansen variance
# ============================================================

def competing_risk_cif_ci(df):

    grouped = (
        df.groupby("time_month")
        .agg(
            charged_off=(
                "event_type",
                lambda x: (x == 1).sum()
            ),
            fully_paid=(
                "event_type",
                lambda x: (x == 2).sum()
            ),
            censored=(
                "event_type",
                lambda x: (x == 0).sum()
            ),
            total_exit=(
                "event_type",
                "size"
            )
        )
        .reset_index()
        .sort_values("time_month")
    )

    n_total = len(df)

    grouped["risk_set"] = (
        n_total
        - grouped["total_exit"]
        .cumsum()
        .shift(fill_value=0)
    )

    # --------------------------------------------------------
    # CIF recursion
    # --------------------------------------------------------

    S = 1.0
    F_co = 0.0
    F_fp = 0.0

    # Influence-style variance accumulators.
    # These provide pointwise large-sample approximations.
    var_co = 0.0
    var_fp = 0.0

    rows = []

    for _, r in grouped.iterrows():

        t = int(r["time_month"])
        Y = float(r["risk_set"])
        d1 = float(r["charged_off"])
        d2 = float(r["fully_paid"])
        dc = float(r["censored"])

        if Y <= 0:
            continue

        d = d1 + d2

        h1 = d1 / Y
        h2 = d2 / Y
        h = d / Y

        S_before = S
        F1_before = F_co
        F2_before = F_fp

        # CIF increments
        F_co = F1_before + S_before * h1
        F_fp = F2_before + S_before * h2

        # Overall event-free survival
        S = S_before * (1.0 - h)

        # ----------------------------------------------------
        # Approximate pointwise variance increments
        #
        # Multinomial discrete-time approximation:
        # Var(h_k) ≈ h_k(1-h_k)/Y
        #
        # This is appropriate here as a large-sample
        # uncertainty diagnostic. We will label it as
        # approximate pointwise 95% CI.
        # ----------------------------------------------------

        if Y > 1:

            var_h1 = (
                h1 * (1.0 - h1) / Y
            )

            var_h2 = (
                h2 * (1.0 - h2) / Y
            )

            # Propagate uncertainty in cumulative incidence.
            # Large-sample delta approximation.
            var_co += (
                S_before ** 2
                * var_h1
            )

            var_fp += (
                S_before ** 2
                * var_h2
            )

        se_co = np.sqrt(
            max(var_co, 0.0)
        )

        se_fp = np.sqrt(
            max(var_fp, 0.0)
        )

        co_low = max(
            0.0,
            F_co - 1.96 * se_co
        )

        co_high = min(
            1.0,
            F_co + 1.96 * se_co
        )

        fp_low = max(
            0.0,
            F_fp - 1.96 * se_fp
        )

        fp_high = min(
            1.0,
            F_fp + 1.96 * se_fp
        )

        rows.append({
            "time_month": t,
            "risk_set": int(Y),
            "charged_off": int(d1),
            "fully_paid": int(d2),
            "censored": int(dc),

            "survival": S,

            "cif_chargeoff": F_co,
            "se_chargeoff": se_co,
            "chargeoff_ci_low": co_low,
            "chargeoff_ci_high": co_high,

            "cif_fullypaid": F_fp,
            "se_fullypaid": se_fp,
            "fullypaid_ci_low": fp_low,
            "fullypaid_ci_high": fp_high
        })

    return pd.DataFrame(rows)


# ============================================================
# A. Overall
# ============================================================

overall_ci = competing_risk_cif_ci(
    aj
)

PRIMARY_HORIZONS = [
    6,
    12,
    18,
    24,
    36
]


def get_horizon_rows(
    table,
    horizons,
    label
):

    output = []

    for h in horizons:

        exact = table[
            table["time_month"].eq(h)
        ]

        if len(exact) == 0:
            continue

        r = exact.iloc[0]

        output.append({
            "group": label,
            "horizon_months": h,
            "risk_set": int(
                r["risk_set"]
            ),

            "cif_chargeoff":
                r["cif_chargeoff"],

            "chargeoff_ci_low":
                r["chargeoff_ci_low"],

            "chargeoff_ci_high":
                r["chargeoff_ci_high"],

            "cif_fullypaid":
                r["cif_fullypaid"],

            "fullypaid_ci_low":
                r["fullypaid_ci_low"],

            "fullypaid_ci_high":
                r["fullypaid_ci_high"]
        })

    return pd.DataFrame(output)


overall_horizon_ci = get_horizon_rows(
    overall_ci,
    PRIMARY_HORIZONS,
    "Overall"
)

print("=" * 125)
print("A. OVERALL CIF WITH APPROXIMATE POINTWISE 95% CI")
print("=" * 125)

print(
    overall_horizon_ci.to_string(
        index=False,
        formatters={
            "cif_chargeoff":
                "{:.6f}".format,
            "chargeoff_ci_low":
                "{:.6f}".format,
            "chargeoff_ci_high":
                "{:.6f}".format,
            "cif_fullypaid":
                "{:.6f}".format,
            "fullypaid_ci_low":
                "{:.6f}".format,
            "fullypaid_ci_high":
                "{:.6f}".format
        }
    )
)


# ============================================================
# B. By contractual term
# ============================================================

term_ci_parts = []
term_horizon_ci_parts = []

for term in [36, 60]:

    temp = aj[
        aj["term_months"].eq(term)
    ].copy()

    result = competing_risk_cif_ci(
        temp
    )

    result["term_months"] = term

    term_ci_parts.append(
        result
    )

    h = get_horizon_rows(
        result,
        [
            x for x in PRIMARY_HORIZONS
            if x <= term
        ],
        f"{term}-month"
    )

    h["term_months"] = term

    term_horizon_ci_parts.append(
        h
    )


term_ci = pd.concat(
    term_ci_parts,
    ignore_index=True
)

term_horizon_ci = pd.concat(
    term_horizon_ci_parts,
    ignore_index=True
)

print("\n" + "=" * 125)
print("B. TERM-SPECIFIC CIF WITH APPROXIMATE POINTWISE 95% CI")
print("=" * 125)

print(
    term_horizon_ci[
        [
            "term_months",
            "horizon_months",
            "risk_set",
            "cif_chargeoff",
            "chargeoff_ci_low",
            "chargeoff_ci_high",
            "cif_fullypaid",
            "fullypaid_ci_low",
            "fullypaid_ci_high"
        ]
    ].to_string(
        index=False,
        formatters={
            "cif_chargeoff":
                "{:.6f}".format,
            "chargeoff_ci_low":
                "{:.6f}".format,
            "chargeoff_ci_high":
                "{:.6f}".format,
            "cif_fullypaid":
                "{:.6f}".format,
            "fullypaid_ci_low":
                "{:.6f}".format,
            "fullypaid_ci_high":
                "{:.6f}".format
        }
    )
)


# ============================================================
# C. Width of charge-off CI
# ============================================================

print("\n" + "=" * 125)
print("C. CHARGE-OFF CI WIDTH — OVERALL")
print("=" * 125)

ci_width = overall_horizon_ci[
    [
        "horizon_months",
        "risk_set",
        "cif_chargeoff",
        "chargeoff_ci_low",
        "chargeoff_ci_high"
    ]
].copy()

ci_width["ci_width"] = (
    ci_width["chargeoff_ci_high"]
    - ci_width["chargeoff_ci_low"]
)

print(
    ci_width.to_string(
        index=False,
        formatters={
            "cif_chargeoff":
                "{:.6f}".format,
            "chargeoff_ci_low":
                "{:.6f}".format,
            "chargeoff_ci_high":
                "{:.6f}".format,
            "ci_width":
                "{:.6f}".format
        }
    )
)


# ============================================================
# Save
# ============================================================

overall_ci.to_csv(
    OUTPUT_DIR /
    "cell21_overall_cif_with_ci.csv",
    index=False
)

overall_horizon_ci.to_csv(
    OUTPUT_DIR /
    "cell21_overall_horizon_ci.csv",
    index=False
)

term_ci.to_csv(
    OUTPUT_DIR /
    "cell21_term_cif_with_ci.csv",
    index=False
)

term_horizon_ci.to_csv(
    OUTPUT_DIR /
    "cell21_term_horizon_ci.csv",
    index=False
)

print("\nSaved Cell 21 outputs to:")
print(OUTPUT_DIR)

A. OVERALL CIF WITH APPROXIMATE POINTWISE 95% CI
  group  horizon_months  risk_set cif_chargeoff chargeoff_ci_low chargeoff_ci_high cif_fullypaid fullypaid_ci_low fullypaid_ci_high
Overall               6   2081403      0.016093         0.015935          0.016251      0.064993         0.064680          0.065306
Overall              12   1092889      0.041301         0.041010          0.041592      0.143808         0.143277          0.144338
Overall              18    637032      0.074998         0.074534          0.075463      0.237000         0.236202          0.237798
Overall              24    522416      0.106141         0.105551          0.106731      0.332919         0.331901          0.333937
Overall              36    221142      0.141911         0.141187          0.142635      0.674790         0.673232          0.676348

B. TERM-SPECIFIC CIF WITH APPROXIMATE POINTWISE 95% CI
 term_months  horizon_months  risk_set cif_chargeoff chargeoff_ci_low chargeoff_ci_high cif_fullypaid f

In [26]:
# ============================================================
# CELL 22 — Bootstrap verification of primary charge-off CIFs
#
# Purpose:
#   Independently verify uncertainty for the primary
#   competing-risk estimates at:
#       12, 24, 36 months
#
# Method:
#   100 loan-level bootstrap replicates
#   Fixed random seed
#
# IMPORTANT:
#   Only the three required horizons are calculated.
# ============================================================

import pandas as pd
import numpy as np
import time


if "aj" not in globals():
    raise RuntimeError(
        "`aj` from Cell 19 is not available in memory."
    )


HORIZONS = [12, 24, 36]

N_BOOT = 100

RNG = np.random.default_rng(2026)


# ============================================================
# Fast CIF function
# ============================================================

def cif_at_horizons_fast(
    time_month,
    event_type,
    horizons
):

    max_h = max(horizons)

    # We only need information through max requested horizon.
    # Anything later remains in the risk set.
    n = len(time_month)

    # Counts by month
    co = np.bincount(
        time_month[
            event_type == 1
        ],
        minlength=max_h + 1
    )

    fp = np.bincount(
        time_month[
            event_type == 2
        ],
        minlength=max_h + 1
    )

    cens = np.bincount(
        time_month[
            event_type == 0
        ],
        minlength=max_h + 1
    )

    S = 1.0
    F_co = 0.0
    F_fp = 0.0

    risk = n

    output = {}

    for t in range(1, max_h + 1):

        d_co = (
            co[t]
            if t < len(co)
            else 0
        )

        d_fp = (
            fp[t]
            if t < len(fp)
            else 0
        )

        d_cens = (
            cens[t]
            if t < len(cens)
            else 0
        )

        if risk > 0:

            h_co = d_co / risk
            h_fp = d_fp / risk

            F_co += S * h_co
            F_fp += S * h_fp

            S *= (
                1.0
                - h_co
                - h_fp
            )

        # Remove all exits after processing events
        risk -= (
            d_co
            + d_fp
            + d_cens
        )

        if t in horizons:

            output[t] = (
                F_co,
                F_fp,
                S
            )

    return output


# ============================================================
# Prepare arrays
# ============================================================

time_arr = (
    aj["time_month"]
    .to_numpy(dtype=np.int16)
)

event_arr = (
    aj["event_type"]
    .to_numpy(dtype=np.int8)
)

n = len(aj)


# ============================================================
# Original estimates
# ============================================================

original = cif_at_horizons_fast(
    time_arr,
    event_arr,
    HORIZONS
)

print("=" * 105)
print("A. ORIGINAL PRIMARY CIF ESTIMATES")
print("=" * 105)

for h in HORIZONS:

    print(
        f"{h:>2} months | "
        f"Charge-off CIF = "
        f"{original[h][0]:.6f}"
    )


# ============================================================
# Bootstrap
# ============================================================

print("\n" + "=" * 105)
print(
    f"B. RUNNING {N_BOOT} LOAN-LEVEL BOOTSTRAP REPLICATES"
)
print("=" * 105)

boot = {
    h: []
    for h in HORIZONS
}

start = time.time()

for b in range(N_BOOT):

    idx = RNG.integers(
        0,
        n,
        size=n
    )

    result = cif_at_horizons_fast(
        time_arr[idx],
        event_arr[idx],
        HORIZONS
    )

    for h in HORIZONS:
        boot[h].append(
            result[h][0]
        )

    if (
        (b + 1) % 10 == 0
        or b == 0
    ):

        elapsed = (
            time.time() - start
        )

        print(
            f"Completed "
            f"{b + 1:>3}/{N_BOOT} "
            f"| elapsed "
            f"{elapsed:.1f} sec"
        )


# ============================================================
# Summarise
# ============================================================

rows = []

for h in HORIZONS:

    values = np.asarray(
        boot[h]
    )

    rows.append({
        "horizon_months": h,

        "original_cif":
            original[h][0],

        "bootstrap_mean":
            values.mean(),

        "bootstrap_sd":
            values.std(
                ddof=1
            ),

        "bootstrap_ci_low":
            np.quantile(
                values,
                0.025
            ),

        "bootstrap_ci_high":
            np.quantile(
                values,
                0.975
            ),

        "n_boot":
            N_BOOT
    })


bootstrap_summary = pd.DataFrame(
    rows
)


print("\n" + "=" * 105)
print("C. BOOTSTRAP CHARGE-OFF CIF RESULTS")
print("=" * 105)

print(
    bootstrap_summary.to_string(
        index=False,
        formatters={
            "original_cif":
                "{:.6f}".format,

            "bootstrap_mean":
                "{:.6f}".format,

            "bootstrap_sd":
                "{:.6f}".format,

            "bootstrap_ci_low":
                "{:.6f}".format,

            "bootstrap_ci_high":
                "{:.6f}".format
        }
    )
)


# ============================================================
# Compare analytic vs bootstrap intervals
# ============================================================

analytic_compare = (
    overall_horizon_ci[
        overall_horizon_ci[
            "horizon_months"
        ].isin(HORIZONS)
    ][
        [
            "horizon_months",
            "cif_chargeoff",
            "chargeoff_ci_low",
            "chargeoff_ci_high"
        ]
    ]
    .merge(
        bootstrap_summary,
        on="horizon_months",
        how="left"
    )
)


print("\n" + "=" * 105)
print("D. APPROXIMATE ANALYTIC VS BOOTSTRAP CI")
print("=" * 105)

print(
    analytic_compare.to_string(
        index=False,
        formatters={
            "cif_chargeoff":
                "{:.6f}".format,

            "chargeoff_ci_low":
                "{:.6f}".format,

            "chargeoff_ci_high":
                "{:.6f}".format,

            "original_cif":
                "{:.6f}".format,

            "bootstrap_mean":
                "{:.6f}".format,

            "bootstrap_sd":
                "{:.6f}".format,

            "bootstrap_ci_low":
                "{:.6f}".format,

            "bootstrap_ci_high":
                "{:.6f}".format
        }
    )
)


# ============================================================
# Save
# ============================================================

bootstrap_summary.to_csv(
    OUTPUT_DIR /
    "cell22_bootstrap_cif_summary.csv",
    index=False
)

analytic_compare.to_csv(
    OUTPUT_DIR /
    "cell22_analytic_vs_bootstrap_ci.csv",
    index=False
)

print("\nSaved Cell 22 outputs to:")
print(OUTPUT_DIR)

A. ORIGINAL PRIMARY CIF ESTIMATES
12 months | Charge-off CIF = 0.041301
24 months | Charge-off CIF = 0.106141
36 months | Charge-off CIF = 0.141911

B. RUNNING 100 LOAN-LEVEL BOOTSTRAP REPLICATES
Completed   1/100 | elapsed 0.1 sec
Completed  10/100 | elapsed 0.5 sec
Completed  20/100 | elapsed 0.9 sec
Completed  30/100 | elapsed 1.4 sec
Completed  40/100 | elapsed 1.8 sec
Completed  50/100 | elapsed 2.2 sec
Completed  60/100 | elapsed 2.7 sec
Completed  70/100 | elapsed 3.1 sec
Completed  80/100 | elapsed 3.5 sec
Completed  90/100 | elapsed 3.9 sec
Completed 100/100 | elapsed 4.3 sec

C. BOOTSTRAP CHARGE-OFF CIF RESULTS
 horizon_months original_cif bootstrap_mean bootstrap_sd bootstrap_ci_low bootstrap_ci_high  n_boot
             12     0.041301       0.041333     0.000149         0.041041          0.041599     100
             24     0.106141       0.106164     0.000294         0.105662          0.106689     100
             36     0.141911       0.141932     0.000354         0.1412

In [27]:
# ============================================================
# CELL 23 — Locate the manuscript modelling pipeline
#
# Purpose:
#   Search the old notebook for the exact code associated with:
#     - 163-dimensional feature frame
#     - 180,000 training sample
#     - Logistic / SGD
#     - HistGradientBoosting
#     - MLP
#     - seeds 17, 42, 2026
#     - manuscript chronological split
#
# NO MODEL IS TRAINED HERE.
# ============================================================

import json
from pathlib import Path

OLD_NOTEBOOK = (
    Path(r"E:\Governance-Policy\Cogent Economics & Finance")
    / "Fintech_25_12_2025.ipynb"
)

print("Notebook:")
print(OLD_NOTEBOOK)
print("Exists:", OLD_NOTEBOOK.exists())

if not OLD_NOTEBOOK.exists():
    raise FileNotFoundError(OLD_NOTEBOOK)


# ============================================================
# Load notebook JSON
# ============================================================

with open(
    OLD_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

cells = nb.get("cells", [])

print("\nTotal notebook cells:", len(cells))


# ============================================================
# Search terms
# ============================================================

SEARCH_TERMS = [
    "180000",
    "180_000",
    "163",
    "HistGradientBoosting",
    "SGDClassifier",
    "MLPClassifier",
    "17, 42, 2026",
    "2026",
    "train_test_split",
    "sample(",
    "roc_auc_score",
    "final_modelling_frame",
    "OneHotEncoder",
    "StandardScaler",
    "ColumnTransformer",
    "Pipeline(",
    "badloan",
    "2018-06",
    "2018-07",
    "2019-03",
    "2019-04"
]


# ============================================================
# Identify matching code cells
# ============================================================

matches = []

for i, cell in enumerate(cells):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    found = [
        term
        for term in SEARCH_TERMS
        if term.lower() in source.lower()
    ]

    if found:

        matches.append({
            "cell_index": i,
            "terms": found,
            "source": source
        })


print("\nMatching code cells:", len(matches))


# ============================================================
# Print matches with context
# ============================================================

print("\n" + "=" * 120)
print("A. MATCHING CODE CELLS")
print("=" * 120)

for m in matches:

    print("\n")
    print("#" * 120)
    print(
        f"NOTEBOOK CELL INDEX: "
        f"{m['cell_index']}"
    )
    print(
        "MATCHED TERMS:",
        ", ".join(m["terms"])
    )
    print("#" * 120)

    source_lines = (
        m["source"]
        .splitlines()
    )

    # Print complete matching cell, but cap extremely
    # large cells at 220 lines for readable output.
    if len(source_lines) <= 220:

        print(m["source"])

    else:

        print(
            "\n".join(
                source_lines[:220]
            )
        )

        print(
            "\n...[CELL TRUNCATED AFTER "
            "220 LINES]..."
        )


# ============================================================
# B. Compact index
# ============================================================

print("\n" + "=" * 120)
print("B. COMPACT MATCH INDEX")
print("=" * 120)

for m in matches:

    print(
        f"Cell {m['cell_index']:>4}: "
        + ", ".join(m["terms"])
    )


# ============================================================
# C. Search notebook outputs/text for manuscript numbers
# ============================================================

TARGET_NUMBERS = [
    "0.6762",
    "0.6706",
    "0.6715",
    "875616",
    "875,616",
    "27374",
    "27,374",
    "41145",
    "41,145",
    "0.0056"
]

print("\n" + "=" * 120)
print("C. CELLS CONTAINING MANUSCRIPT RESULT NUMBERS")
print("=" * 120)

number_hits = []

for i, cell in enumerate(cells):

    text_parts = []

    text_parts.extend(
        cell.get("source", [])
    )

    for out in cell.get("outputs", []):

        if "text" in out:
            text_parts.extend(
                out.get("text", [])
            )

        data = out.get("data", {})

        if "text/plain" in data:
            value = data["text/plain"]

            if isinstance(value, list):
                text_parts.extend(value)
            else:
                text_parts.append(
                    str(value)
                )

    combined = "".join(
        str(x)
        for x in text_parts
    )

    found = [
        x
        for x in TARGET_NUMBERS
        if x in combined
    ]

    if found:

        number_hits.append(
            (i, found)
        )

        print(
            f"Cell {i}: "
            + ", ".join(found)
        )


if not number_hits:
    print(
        "No exact manuscript result numbers "
        "were found in saved notebook cells/outputs."
    )


# ============================================================
# D. Save search index
# ============================================================

search_index = [
    {
        "cell_index": m["cell_index"],
        "matched_terms": "|".join(
            m["terms"]
        )
    }
    for m in matches
]

import pandas as pd

pd.DataFrame(
    search_index
).to_csv(
    OUTPUT_DIR /
    "cell23_old_notebook_model_search.csv",
    index=False
)

print("\nSaved Cell 23 search index to:")
print(OUTPUT_DIR)

Notebook:
E:\Governance-Policy\Cogent Economics & Finance\Fintech_25_12_2025.ipynb
Exists: True

Total notebook cells: 90

Matching code cells: 23

A. MATCHING CODE CELLS


########################################################################################################################
NOTEBOOK CELL INDEX: 36
MATCHED TERMS: badloan
########################################################################################################################
import numpy as np
import pandas as pd

TARGET = "badloan"
DATE_COL = "issue_date"

# 1) Pick only the columns you will use
feature_cols = borrower_features + loan_features + credit_features + macro_features + time_features
df_model = df[feature_cols + [TARGET, DATE_COL]].copy()

# 2) Ensure correct dtypes
df_model[DATE_COL] = pd.to_datetime(df_model[DATE_COL], errors="coerce")
df_model[TARGET] = pd.to_numeric(df_model[TARGET], errors="coerce")

# 3) Keep valid rows
df_model = df_model.dropna(subset=[DATE_COL, TARGET])
df_model[TARG

In [28]:
# ============================================================
# CELL 24 — Reconstruct candidate origination-time features
#
# Purpose:
#   Build an explicit, reproducible inventory of candidate
#   predictors from the RAW .dta file.
#
#   NO model training.
#   NO arbitrary forcing to 163 features.
#
# We first audit what is actually available and classify
# obvious post-outcome / leakage fields.
# ============================================================

import pandas as pd
import numpy as np
import re


# ============================================================
# A. Read metadata / small sample from raw Stata file
# ============================================================

print("Reading raw dataset metadata/sample...")

with pd.read_stata(
    DTA_PATH,
    convert_categoricals=False,
    iterator=True
) as reader:

    sample = reader.read(5000)


all_cols = sample.columns.tolist()

print("\nTotal raw columns:", len(all_cols))


# ============================================================
# B. Explicit non-feature fields
# ============================================================

TARGET_COLS = {
    "badloan",
    "loan_status",
    "loan_status_",
    "loan_status_5",
    "rankloan_status_"
}

IDENTIFIER_COLS = {
    "id",
    "member_id"
}

RAW_DATE_COLS = {
    "issue_d",
    "issue_date",
    "issue_daily",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_date",
    "daily_date"
}


# ============================================================
# C. Explicit post-origination leakage patterns
# ============================================================

LEAK_PATTERNS = [
    r"(^|_)pymnt($|_)",
    r"payment",
    r"last_pymnt",
    r"next_pymnt",

    r"recover",
    r"collection_recovery",

    r"out_prncp",
    r"total_pymnt",
    r"total_rec_",

    r"settlement",
    r"hardship",

    r"chargeoff",
    r"charged_off",

    r"debt_settlement",

    r"last_credit_pull",

    r"loan_status",
    r"badloan",

    r"^last_date$"
]


def matches_leak_pattern(col):

    c = col.lower()

    return any(
        re.search(
            pattern,
            c
        )
        is not None

        for pattern
        in LEAK_PATTERNS
    )


# ============================================================
# D. Classify every raw column
# ============================================================

rows = []

for c in all_cols:

    if c in TARGET_COLS:

        category = "TARGET/OUTCOME"

    elif c in IDENTIFIER_COLS:

        category = "IDENTIFIER"

    elif c in RAW_DATE_COLS:

        if c in {
            "issue_d",
            "issue_date",
            "issue_daily"
        }:
            category = "ORIGINATION_DATE"

        else:
            category = "POST_ORIGINATION_DATE"

    elif matches_leak_pattern(c):

        category = "POST_ORIGINATION_LEAKAGE"

    else:

        category = "CANDIDATE_FEATURE"

    rows.append({
        "column": c,
        "dtype": str(
            sample[c].dtype
        ),
        "category": category,
        "missing_pct_sample":
            100 * sample[c].isna().mean(),
        "nunique_sample":
            sample[c].nunique(
                dropna=True
            )
    })


feature_audit = pd.DataFrame(
    rows
)


# ============================================================
# E. Summary
# ============================================================

print("\n" + "=" * 115)
print("A. RAW COLUMN CLASSIFICATION")
print("=" * 115)

summary = (
    feature_audit[
        "category"
    ]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="n_columns")
)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# F. Explicit excluded columns
# ============================================================

excluded = feature_audit[
    feature_audit[
        "category"
    ].ne("CANDIDATE_FEATURE")
].copy()

print("\n" + "=" * 115)
print("B. NON-CANDIDATE / EXCLUDED COLUMNS")
print("=" * 115)

print(
    excluded[
        [
            "column",
            "dtype",
            "category"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# G. Candidate features
# ============================================================

candidate = feature_audit[
    feature_audit[
        "category"
    ].eq("CANDIDATE_FEATURE")
].copy()

print("\n" + "=" * 115)
print("C. CANDIDATE FEATURE COUNT")
print("=" * 115)

print(
    "Candidate raw columns:",
    len(candidate)
)


print("\n" + "=" * 115)
print("D. CANDIDATE FEATURES")
print("=" * 115)

print(
    candidate[
        [
            "column",
            "dtype",
            "missing_pct_sample",
            "nunique_sample"
        ]
    ].to_string(
        index=False,
        formatters={
            "missing_pct_sample":
                "{:.2f}".format
        }
    )
)


# ============================================================
# H. Suspicious names requiring manual review
# ============================================================

SUSPICIOUS_TERMS = [
    "delinq",
    "late",
    "default",
    "status",
    "balance",
    "bal",
    "principal",
    "prncp",
    "paid",
    "payment",
    "recover",
    "collection",
    "charge",
    "hardship",
    "settlement",
    "last",
    "next"
]


suspicious_mask = candidate[
    "column"
].str.lower().apply(
    lambda c: any(
        term in c
        for term in SUSPICIOUS_TERMS
    )
)

suspicious = candidate[
    suspicious_mask
].copy()


print("\n" + "=" * 115)
print("E. CANDIDATE COLUMNS REQUIRING MANUAL LEAKAGE REVIEW")
print("=" * 115)

if len(suspicious):

    print(
        suspicious[
            [
                "column",
                "dtype",
                "missing_pct_sample",
                "nunique_sample"
            ]
        ].to_string(
            index=False,
            formatters={
                "missing_pct_sample":
                    "{:.2f}".format
            }
        )
    )

else:

    print(
        "No suspicious candidate names detected."
    )


# ============================================================
# I. Candidate categorical/numeric split
# ============================================================

candidate_names = (
    candidate["column"]
    .tolist()
)

candidate_sample = sample[
    candidate_names
]

cat_candidates = (
    candidate_sample
    .select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
    .tolist()
)

num_candidates = [
    c
    for c in candidate_names
    if c not in cat_candidates
]


print("\n" + "=" * 115)
print("F. CANDIDATE TYPE SUMMARY")
print("=" * 115)

print(
    "Numeric candidate columns:",
    len(num_candidates)
)

print(
    "Categorical candidate columns:",
    len(cat_candidates)
)

print(
    "Total:",
    len(candidate_names)
)


# ============================================================
# J. Check manuscript's claimed 163 dimensions
# ============================================================

print("\n" + "=" * 115)
print("G. COMPARISON WITH MANUSCRIPT'S 163-DIMENSION STATEMENT")
print("=" * 115)

print(
    "Raw candidate feature count:",
    len(candidate_names)
)

print(
    "Difference from 163:",
    len(candidate_names) - 163
)

if len(candidate_names) == 163:

    print(
        "The raw candidate count equals 163."
    )

else:

    print(
        "The raw candidate count does NOT equal 163. "
        "Therefore we must not assume that the manuscript's "
        "'163-dimensional' statement refers to raw columns."
    )


# ============================================================
# Save
# ============================================================

feature_audit.to_csv(
    OUTPUT_DIR /
    "cell24_raw_feature_audit.csv",
    index=False
)

candidate.to_csv(
    OUTPUT_DIR /
    "cell24_candidate_features.csv",
    index=False
)

suspicious.to_csv(
    OUTPUT_DIR /
    "cell24_suspicious_candidate_features.csv",
    index=False
)

print("\nSaved Cell 24 outputs to:")
print(OUTPUT_DIR)


Reading raw dataset metadata/sample...

Total raw columns: 195

A. RAW COLUMN CLASSIFICATION
                category  n_columns
       CANDIDATE_FEATURE        169
POST_ORIGINATION_LEAKAGE         13
          TARGET/OUTCOME          5
   POST_ORIGINATION_DATE          5
        ORIGINATION_DATE          3

B. NON-CANDIDATE / EXCLUDED COLUMNS
                  column          dtype                 category
                 issue_d         object         ORIGINATION_DATE
             loan_status         object           TARGET/OUTCOME
              pymnt_plan         object POST_ORIGINATION_LEAKAGE
               out_prncp        float32 POST_ORIGINATION_LEAKAGE
           out_prncp_inv        float32 POST_ORIGINATION_LEAKAGE
             total_pymnt        float32 POST_ORIGINATION_LEAKAGE
         total_pymnt_inv        float32 POST_ORIGINATION_LEAKAGE
         total_rec_prncp        float32 POST_ORIGINATION_LEAKAGE
           total_rec_int        float32 POST_ORIGINATION_LEAKAGE
    

In [29]:
# ============================================================
# CELL 25 — Full-data feature usability + redundancy audit
#
# Purpose:
#   Determine objectively why 169 candidate raw columns remain
#   and whether the manuscript's "163-dimensional" statement
#   can be explained by unusable / constant / duplicate fields.
#
# NO MODEL TRAINING.
# NO FEATURES ARE DROPPED AUTOMATICALLY.
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict


# ------------------------------------------------------------
# A. Candidate list from Cell 24
# ------------------------------------------------------------

candidate_cols = candidate["column"].tolist()

print("Candidate columns entering Cell 25:",
      len(candidate_cols))


# ------------------------------------------------------------
# B. Full-data streaming audit
# ------------------------------------------------------------

CHUNK = 200_000

nonmissing = defaultdict(int)
unique_values = defaultdict(set)

n_total = 0

# Exact uniqueness tracking is safe for low-cardinality columns.
# For high-cardinality fields we cap stored values.
UNIQUE_CAP = 5000

print("\nScanning full dataset...")

with pd.read_stata(
    DTA_PATH,
    columns=candidate_cols,
    convert_categoricals=False,
    iterator=True
) as reader:

    while True:

        try:
            df = reader.read(CHUNK)
        except StopIteration:
            break

        if df is None or len(df) == 0:
            break

        n_total += len(df)

        for c in candidate_cols:

            s = df[c]

            nonmissing[c] += int(
                s.notna().sum()
            )

            # Continue exact tracking only while manageable
            if len(unique_values[c]) <= UNIQUE_CAP:

                vals = (
                    s.dropna()
                    .astype(str)
                    .unique()
                )

                unique_values[c].update(
                    vals[:UNIQUE_CAP + 1]
                )

        print(
            f"Processed {n_total:,} rows",
            end="\r"
        )


print("\n\nFull rows scanned:",
      f"{n_total:,}")


# ------------------------------------------------------------
# C. Construct full-data audit
# ------------------------------------------------------------

rows = []

for c in candidate_cols:

    n_nonmissing = nonmissing[c]

    observed_unique = len(
        unique_values[c]
    )

    if observed_unique > UNIQUE_CAP:
        unique_label = f">{UNIQUE_CAP}"
        low_information = False
    else:
        unique_label = observed_unique
        low_information = (
            observed_unique <= 1
        )

    rows.append({
        "column": c,
        "dtype_sample": str(sample[c].dtype),
        "n_nonmissing": n_nonmissing,
        "missing_pct":
            100 * (1 - n_nonmissing / n_total),
        "nunique_or_cap": unique_label,
        "zero_information":
            low_information
    })


full_audit = pd.DataFrame(rows)


# ------------------------------------------------------------
# D. Zero-information variables
# ------------------------------------------------------------

zero_info = full_audit[
    full_audit["zero_information"]
].copy()

print("\n" + "=" * 110)
print("A. ZERO-INFORMATION CANDIDATE VARIABLES")
print("=" * 110)

print(
    zero_info.to_string(
        index=False,
        formatters={
            "missing_pct": "{:.4f}".format
        }
    )
)

print(
    "\nZero-information count:",
    len(zero_info)
)


# ------------------------------------------------------------
# E. Completely missing variables
# ------------------------------------------------------------

all_missing = full_audit[
    full_audit["n_nonmissing"].eq(0)
].copy()

print("\n" + "=" * 110)
print("B. COMPLETELY MISSING VARIABLES")
print("=" * 110)

if len(all_missing):

    print(
        all_missing.to_string(
            index=False,
            formatters={
                "missing_pct": "{:.4f}".format
            }
        )
    )

else:
    print("None.")

print(
    "\nCompletely missing count:",
    len(all_missing)
)


# ------------------------------------------------------------
# F. Very high missingness
# ------------------------------------------------------------

high_missing = (
    full_audit[
        full_audit["missing_pct"] >= 90
    ]
    .sort_values(
        "missing_pct",
        ascending=False
    )
)

print("\n" + "=" * 110)
print("C. VARIABLES WITH >=90% MISSINGNESS")
print("=" * 110)

if len(high_missing):

    print(
        high_missing.to_string(
            index=False,
            formatters={
                "missing_pct": "{:.4f}".format
            }
        )
    )

else:
    print("None.")


# ------------------------------------------------------------
# G. Known raw / encoded duplicate representations
# ------------------------------------------------------------

duplicate_representation_pairs = [
    ("application_type",
     "application_type_"),

    ("initial_list_status",
     "initial_list_status_"),

    ("purpose",
     "purpose_"),

    ("verification_status",
     "verification_status_"),

    ("home_ownership",
     "home_ownership_"),

    ("emp_length",
     "emp_length_"),

    ("grade",
     "grade_"),

    ("sub_grade",
     "sub_grade_"),

    ("term",
     "term_")
]


dup_rows = []

for raw_col, encoded_col in duplicate_representation_pairs:

    dup_rows.append({
        "raw_column": raw_col,
        "encoded_column": encoded_col,
        "raw_present":
            raw_col in candidate_cols,
        "encoded_present":
            encoded_col in candidate_cols
    })


dup_table = pd.DataFrame(
    dup_rows
)


print("\n" + "=" * 110)
print("D. RAW / PRE-ENCODED REPRESENTATION PAIRS")
print("=" * 110)

print(
    dup_table.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# H. Raw/log-transformed pairs
# ------------------------------------------------------------

log_pairs = []

candidate_set = set(
    candidate_cols
)

for c in candidate_cols:

    if c.startswith("log"):

        raw = c[3:]

        # direct candidate:
        # logannual_inc -> annual_inc
        if raw in candidate_set:

            log_pairs.append(
                (raw, c)
            )

        # handle common underscore naming:
        # logfunded_amnt -> funded_amnt
        elif (
            raw.startswith("_")
            and raw[1:] in candidate_set
        ):

            log_pairs.append(
                (raw[1:], c)
            )


log_pairs = sorted(
    set(log_pairs)
)

log_pair_df = pd.DataFrame(
    log_pairs,
    columns=[
        "raw_column",
        "log_column"
    ]
)


print("\n" + "=" * 110)
print("E. RAW / LOG-TRANSFORMED PAIRS")
print("=" * 110)

if len(log_pair_df):

    print(
        log_pair_df.to_string(
            index=False
        )
    )

else:

    print(
        "No direct raw/log pairs detected."
    )


# ------------------------------------------------------------
# I. Exact-name related feature families
# ------------------------------------------------------------

families = {
    "loan_amount":
        [
            "loan_amnt",
            "funded_amnt",
            "funded_amnt_inv",
            "logfunded_amnt"
        ],

    "time":
        [
            "quarter",
            "month",
            "year",
            "issue_year"
        ],

    "inflation":
        [
            "cpi",
            "CPIUS",
            "logCPIUS",
            "inf",
            "infUS",
            "inf_6m"
        ],

    "unemployment":
        [
            "unemployment",
            "unem_rate",
            "logunemployment"
        ],

    "loan_volume":
        [
            "loan_vol",
            "logloan_vol",
            "loan_vol6m",
            "loglvol_6m",
            "loan_sum"
        ]
}


family_rows = []

for family, cols in families.items():

    for c in cols:

        if c in candidate_set:

            family_rows.append({
                "family": family,
                "column": c
            })


family_df = pd.DataFrame(
    family_rows
)


print("\n" + "=" * 110)
print("F. RELATED FEATURE FAMILIES FOR MANUAL REVIEW")
print("=" * 110)

print(
    family_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# J. Counts after ONLY objective zero-information removal
# ------------------------------------------------------------

usable_after_zero_info = [
    c
    for c in candidate_cols
    if c not in set(
        zero_info["column"]
    )
]


print("\n" + "=" * 110)
print("G. FEATURE-COUNT RECONCILIATION")
print("=" * 110)

print(
    "Initial candidates:",
    len(candidate_cols)
)

print(
    "Zero-information:",
    len(zero_info)
)

print(
    "Remaining after zero-information removal:",
    len(usable_after_zero_info)
)

print(
    "Difference from manuscript 163:",
    len(usable_after_zero_info) - 163
)

if len(usable_after_zero_info) == 163:

    print(
        "\nIMPORTANT: Removing only objectively "
        "zero-information variables reproduces "
        "the manuscript's 163-feature count."
    )

else:

    print(
        "\nThe 163-feature count is still not "
        "explained. We will inspect redundancy "
        "without forcing the count."
    )


# ------------------------------------------------------------
# K. Save outputs
# ------------------------------------------------------------

full_audit.to_csv(
    OUTPUT_DIR /
    "cell25_full_feature_usability_audit.csv",
    index=False
)

zero_info.to_csv(
    OUTPUT_DIR /
    "cell25_zero_information_features.csv",
    index=False
)

dup_table.to_csv(
    OUTPUT_DIR /
    "cell25_duplicate_representation_pairs.csv",
    index=False
)

log_pair_df.to_csv(
    OUTPUT_DIR /
    "cell25_raw_log_pairs.csv",
    index=False
)

family_df.to_csv(
    OUTPUT_DIR /
    "cell25_related_feature_families.csv",
    index=False
)

print("\nSaved Cell 25 outputs to:")
print(OUTPUT_DIR)


Candidate columns entering Cell 25: 169

Scanning full dataset...
Processed 2,703,430 rows

Full rows scanned: 2,703,430

A. ZERO-INFORMATION CANDIDATE VARIABLES
                   column dtype_sample  n_nonmissing missing_pct nunique_or_cap  zero_information
                      url      float64             0    100.0000              0              True
              policy_code         int8       2703430      0.0000              1              True
verification_status_joint      float64             0    100.0000              0              True
               loanmedian      float32       2703430      0.0000              1              True
             meanopen_acc      float32       2703430      0.0000              1              True
            loan6m_median      float32       2703430      0.0000              1              True
                 loan_sum      float32       2703430      0.0000              1              True

Zero-information count: 7

B. COMPLETELY MISSING VARI

In [30]:
# ============================================================
# CELL 26 — Lock revised modelling cohort + feature specification
#
# Purpose:
#   1. Start from the 162 non-zero-information candidates.
#   2. Remove duplicate pre-encoded versions where the raw
#      categorical variable is already available.
#   3. Retain legitimate origination-time credit-history fields.
#   4. Reconstruct the manuscript chronological periods.
#   5. Apply the already-verified maturity definition.
#
# NO MODEL TRAINING.
# After this cell, the feature specification is LOCKED.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# A. Start from Cell 25 objectively usable features
# ------------------------------------------------------------

zero_info_set = set(zero_info["column"])

base_features = [
    c for c in candidate_cols
    if c not in zero_info_set
]

print("=" * 110)
print("A. STARTING FEATURE SET")
print("=" * 110)

print("Cell 24 candidates:", len(candidate_cols))
print("Zero-information removed:", len(zero_info_set))
print("Remaining:", len(base_features))


# ------------------------------------------------------------
# B. Remove duplicate pre-encoded categorical representations
#
# Keep the original categorical version and let sklearn encode
# it inside the training pipeline.
# ------------------------------------------------------------

PREENCODED_DUPLICATES = [
    "application_type_",
    "initial_list_status_",
    "purpose_",
    "verification_status_",
    "home_ownership_",
    "emp_length_",
    "grade_",
    "sub_grade_",
    "term_"
]

model_features = [
    c for c in base_features
    if c not in PREENCODED_DUPLICATES
]

print("\n" + "=" * 110)
print("B. DUPLICATE REPRESENTATIONS REMOVED")
print("=" * 110)

print("Removed:", PREENCODED_DUPLICATES)
print("Count removed:", len(PREENCODED_DUPLICATES))
print("Remaining model features:", len(model_features))


# ------------------------------------------------------------
# C. Explicitly document raw/log pairs
#
# We retain both for now because they are deterministic feature
# transformations available at origination and their presence
# is not outcome leakage.
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("C. RAW/LOG FEATURE POLICY")
print("=" * 110)

print(
    "Raw and log-transformed variables are retained where both "
    "are available because they are origination-time transformations, "
    "not post-outcome information."
)


# ------------------------------------------------------------
# D. Load only fields needed for cohort construction + modelling
# ------------------------------------------------------------

COHORT_COLS = [
    "issue_d",
    "loan_status",
    "term",
    "badloan"
]

needed_cols = list(
    dict.fromkeys(
        COHORT_COLS + model_features
    )
)

print("\nLoading revised modelling frame from raw .dta...")
print("Columns requested:", len(needed_cols))

df_model = pd.read_stata(
    DTA_PATH,
    columns=needed_cols,
    convert_categoricals=False
)

print("Rows loaded:", f"{len(df_model):,}")


# ------------------------------------------------------------
# E. Parse origination date
# ------------------------------------------------------------

df_model["issue_date_clean"] = pd.to_datetime(
    df_model["issue_d"]
        .astype(str)
        .str.strip(),
    format="%b-%y",
    errors="coerce"
)

print("\nIssue-date missing:",
      int(df_model["issue_date_clean"].isna().sum()))

print(
    "Issue range:",
    df_model["issue_date_clean"].min(),
    "to",
    df_model["issue_date_clean"].max()
)


# ------------------------------------------------------------
# F. Parse contractual term
# ------------------------------------------------------------

term_months = (
    df_model["term"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)

df_model["term_months_clean"] = pd.to_numeric(
    term_months,
    errors="coerce"
)

print("\nTerm distribution:")
print(
    df_model["term_months_clean"]
    .value_counts(dropna=False)
    .sort_index()
)


# ------------------------------------------------------------
# G. Reconstruct maturity rule
#
# Verified earlier:
# Mature if:
#   terminal status {Fully Paid, Charged Off, Default}
#   OR contractual follow-up >= term.
#
# For validation/test manuscript cohorts, April 2020 was the
# diagnostic follow-up cutoff that reproduced exactly:
#   validation = 27,374
#   test       = 41,145
#
# We preserve that verified rule here.
# ------------------------------------------------------------

FOLLOWUP_CUTOFF = pd.Timestamp("2020-04-01")

terminal_statuses = {
    "Fully Paid",
    "Charged Off",
    "Default"
}

months_followup = (
    (FOLLOWUP_CUTOFF.year -
     df_model["issue_date_clean"].dt.year) * 12
    +
    (FOLLOWUP_CUTOFF.month -
     df_model["issue_date_clean"].dt.month)
)

df_model["months_followup_to_cutoff"] = months_followup

df_model["terminal_status"] = (
    df_model["loan_status"]
    .astype(str)
    .str.strip()
    .isin(terminal_statuses)
)

df_model["contractually_observed"] = (
    df_model["months_followup_to_cutoff"]
    >=
    df_model["term_months_clean"]
)

df_model["mature_reconstructed"] = (
    df_model["terminal_status"]
    |
    df_model["contractually_observed"]
)


# ------------------------------------------------------------
# H. Manuscript chronological split
#
# Train: Aug 2012 – Jun 2018
# Val:   Jul 2018 – Mar 2019
# Test:  Apr 2019 – Dec 2019
# ------------------------------------------------------------

d = df_model["issue_date_clean"]

df_model["split_revised"] = np.select(
    [
        (d >= pd.Timestamp("2012-08-01")) &
        (d <= pd.Timestamp("2018-06-01")),

        (d >= pd.Timestamp("2018-07-01")) &
        (d <= pd.Timestamp("2019-03-01")),

        (d >= pd.Timestamp("2019-04-01")) &
        (d <= pd.Timestamp("2019-12-01"))
    ],
    [
        "train",
        "val",
        "test"
    ],
    default="outside"
)


# ------------------------------------------------------------
# I. Cohort audit
# ------------------------------------------------------------

cohort_rows = []

for split in ["train", "val", "test"]:

    s = df_model[
        df_model["split_revised"] == split
    ]

    m = s[
        s["mature_reconstructed"]
    ]

    cohort_rows.append({
        "split": split,
        "all_n": len(s),
        "mature_n": len(m),
        "mature_pct":
            100 * len(m) / len(s)
            if len(s) else np.nan,
        "all_badloan_rate":
            s["badloan"].mean(),
        "mature_badloan_rate":
            m["badloan"].mean()
    })


cohort_audit = pd.DataFrame(
    cohort_rows
)

print("\n" + "=" * 110)
print("D. RECONSTRUCTED COHORTS")
print("=" * 110)

print(
    cohort_audit.to_string(
        index=False,
        formatters={
            "mature_pct": "{:.4f}".format,
            "all_badloan_rate": "{:.6f}".format,
            "mature_badloan_rate": "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# J. Critical validation/test checks
# ------------------------------------------------------------

val_mature_n = int(
    cohort_audit.loc[
        cohort_audit["split"] == "val",
        "mature_n"
    ].iloc[0]
)

test_mature_n = int(
    cohort_audit.loc[
        cohort_audit["split"] == "test",
        "mature_n"
    ].iloc[0]
)

print("\n" + "=" * 110)
print("E. MANUSCRIPT COHORT CHECK")
print("=" * 110)

print(
    "Validation mature:",
    val_mature_n,
    "| expected 27,374 |",
    "MATCH" if val_mature_n == 27374 else "DOES NOT MATCH"
)

print(
    "Test mature:",
    test_mature_n,
    "| expected 41,145 |",
    "MATCH" if test_mature_n == 41145 else "DOES NOT MATCH"
)


# ------------------------------------------------------------
# K. Final feature type audit
# ------------------------------------------------------------

X_probe = df_model[model_features]

categorical_features = (
    X_probe
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

numeric_features = [
    c for c in model_features
    if c not in categorical_features
]

print("\n" + "=" * 110)
print("F. LOCKED REVISED FEATURE SPECIFICATION")
print("=" * 110)

print("Total model features:", len(model_features))
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)


# ------------------------------------------------------------
# L. Missingness in mature modelling cohorts
# ------------------------------------------------------------

mature_model = df_model[
    (df_model["split_revised"].isin(["train", "val", "test"]))
    &
    (df_model["mature_reconstructed"])
]

feature_missing = (
    mature_model[model_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\nTop 20 missing features in mature modelling cohorts:")
print(
    feature_missing.head(20).to_string(
        index=False,
        formatters={
            "missing_pct": "{:.3f}".format
        }
    )
)


# ------------------------------------------------------------
# M. Save reproducibility outputs
# ------------------------------------------------------------

cohort_audit.to_csv(
    OUTPUT_DIR /
    "cell26_revised_cohort_audit.csv",
    index=False
)

pd.DataFrame({
    "feature": model_features,
    "type": [
        "categorical"
        if c in categorical_features
        else "numeric"
        for c in model_features
    ]
}).to_csv(
    OUTPUT_DIR /
    "cell26_locked_feature_specification.csv",
    index=False
)

feature_missing.to_csv(
    OUTPUT_DIR /
    "cell26_feature_missingness.csv",
    index=False
)

print("\nSaved Cell 26 outputs to:")
print(OUTPUT_DIR)

A. STARTING FEATURE SET
Cell 24 candidates: 169
Zero-information removed: 7
Remaining: 162

B. DUPLICATE REPRESENTATIONS REMOVED
Removed: ['application_type_', 'initial_list_status_', 'purpose_', 'verification_status_', 'home_ownership_', 'emp_length_', 'grade_', 'sub_grade_', 'term_']
Count removed: 9
Remaining model features: 153

C. RAW/LOG FEATURE POLICY
Raw and log-transformed variables are retained where both are available because they are origination-time transformations, not post-outcome information.

Loading revised modelling frame from raw .dta...
Columns requested: 156
Rows loaded: 2,703,430

Issue-date missing: 0
Issue range: 2012-08-01 00:00:00 to 2019-12-01 00:00:00

Term distribution:
term_months_clean
36    1901819
60     801611
Name: count, dtype: int64

D. RECONSTRUCTED COHORTS
split   all_n  mature_n mature_pct all_badloan_rate mature_badloan_rate
train 1931267   1304058    67.5234         0.101885            0.138936
  val  370994     27374     7.3786         0.0215

In [31]:
# ============================================================
# CELL 27 — Training-size sensitivity
#
# Reviewer question:
#   Does the 180,000-loan training subset materially affect
#   model performance / model ranking?
#
# Models:
#   1. Logistic regression via SGD
#   2. HistGradientBoosting
#   3. MLP
#
# Training sizes:
#   180k, 360k, 540k, full mature training cohort
#
# Validation and test cohorts remain FIXED.
#
# IMPORTANT:
#   This is a sensitivity analysis, not an attempt to recreate
#   unsupported historical manuscript numbers exactly.
# ============================================================

import gc
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# A. Construct fixed mature cohorts
# ------------------------------------------------------------

train_df = df_model[
    (df_model["split_revised"] == "train") &
    (df_model["mature_reconstructed"])
].copy()

val_df = df_model[
    (df_model["split_revised"] == "val") &
    (df_model["mature_reconstructed"])
].copy()

test_df = df_model[
    (df_model["split_revised"] == "test") &
    (df_model["mature_reconstructed"])
].copy()


# Binary target
for frame in [train_df, val_df, test_df]:

    frame["y"] = (
        pd.to_numeric(
            frame["badloan"],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )


print("=" * 110)
print("A. FIXED MODELLING COHORTS")
print("=" * 110)

print(
    "Train mature:",
    f"{len(train_df):,}",
    "| event rate:",
    f"{train_df['y'].mean():.6f}"
)

print(
    "Validation mature:",
    f"{len(val_df):,}",
    "| event rate:",
    f"{val_df['y'].mean():.6f}"
)

print(
    "Test mature:",
    f"{len(test_df):,}",
    "| event rate:",
    f"{test_df['y'].mean():.6f}"
)


# ------------------------------------------------------------
# B. Remove problematic free-text/high-cardinality identifiers
#
# These fields are not suitable for ordinary one-hot modelling
# and can create enormous sparse matrices.
#
# They are excluded transparently here.
# ------------------------------------------------------------

HIGH_CARDINALITY_TEXT = [
    "emp_title",
    "desc",
    "title"
]

features_27 = [
    c for c in model_features
    if c not in HIGH_CARDINALITY_TEXT
]

categorical_27 = [
    c for c in categorical_features
    if c in features_27
]

numeric_27 = [
    c for c in features_27
    if c not in categorical_27
]


print("\n" + "=" * 110)
print("B. REVISED TRAINING FEATURE SET")
print("=" * 110)

print(
    "Locked Cell 26 features:",
    len(model_features)
)

print(
    "High-cardinality free-text exclusions:",
    HIGH_CARDINALITY_TEXT
)

print(
    "Features entering models:",
    len(features_27)
)

print(
    "Numeric:",
    len(numeric_27)
)

print(
    "Categorical:",
    len(categorical_27)
)


# ------------------------------------------------------------
# C. Preprocessing
#
# Numeric:
#   median imputation + standardisation
#
# Categorical:
#   most-frequent imputation + one-hot encoding
#
# max_categories limits uncontrolled cardinality.
# ------------------------------------------------------------

numeric_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scale",
        StandardScaler()
    )
])

categorical_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            max_categories=50,
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipe,
            numeric_27
        ),
        (
            "cat",
            categorical_pipe,
            categorical_27
        )
    ],
    remainder="drop"
)


# ------------------------------------------------------------
# D. Fit preprocessing ONLY on mature training cohort
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("C. FITTING PREPROCESSOR")
print("=" * 110)

t0 = time.time()

preprocessor.fit(
    train_df[features_27]
)

print(
    "Preprocessor fitted in",
    f"{time.time() - t0:.1f}",
    "seconds"
)


# ------------------------------------------------------------
# E. Transform fixed validation and test sets once
# ------------------------------------------------------------

print("\nTransforming validation/test...")

X_val = preprocessor.transform(
    val_df[features_27]
).astype(np.float32)

X_test = preprocessor.transform(
    test_df[features_27]
).astype(np.float32)

y_val = val_df["y"].to_numpy()
y_test = test_df["y"].to_numpy()

print(
    "Validation matrix:",
    X_val.shape
)

print(
    "Test matrix:",
    X_test.shape
)


# ------------------------------------------------------------
# F. Nested training samples
#
# A single deterministic permutation ensures:
# 180k ⊂ 360k ⊂ 540k ⊂ full
# ------------------------------------------------------------

RANDOM_SEED = 42

rng = np.random.default_rng(
    RANDOM_SEED
)

permutation = rng.permutation(
    len(train_df)
)

training_sizes = [
    180_000,
    360_000,
    540_000,
    len(train_df)
]


# ------------------------------------------------------------
# G. Model factory
# ------------------------------------------------------------

def make_models():

    return {
        "Logistic_SGD":
            SGDClassifier(
                loss="log_loss",
                penalty="l2",
                alpha=1e-5,
                max_iter=100,
                tol=1e-4,
                random_state=42,
                n_jobs=-1
            ),

        "HistGB":
            HistGradientBoostingClassifier(
                max_depth=6,
                learning_rate=0.05,
                max_iter=300,
                random_state=42
            ),

        "MLP":
            MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                solver="adam",
                max_iter=50,
                early_stopping=True,
                validation_fraction=0.10,
                n_iter_no_change=5,
                random_state=42
            )
    }


# ------------------------------------------------------------
# H. Training-size sensitivity
# ------------------------------------------------------------

results_27 = []

prediction_store_27 = {}

print("\n" + "=" * 110)
print("D. TRAINING-SIZE SENSITIVITY")
print("=" * 110)


for n_train in training_sizes:

    print("\n" + "-" * 110)
    print(
        "Training size:",
        f"{n_train:,}"
    )
    print("-" * 110)

    idx = permutation[:n_train]

    subset = train_df.iloc[idx]

    print("Transforming training subset...")

    transform_start = time.time()

    X_train = preprocessor.transform(
        subset[features_27]
    ).astype(np.float32)

    y_train = (
        subset["y"]
        .to_numpy()
    )

    print(
        "Training matrix:",
        X_train.shape,
        "| transform seconds:",
        f"{time.time() - transform_start:.1f}"
    )

    print(
        "Training event rate:",
        f"{y_train.mean():.6f}"
    )

    models = make_models()

    for model_name, model in models.items():

        print(
            f"\n  Fitting {model_name}..."
        )

        start = time.time()

        model.fit(
            X_train,
            y_train
        )

        fit_seconds = (
            time.time() - start
        )

        p_val = model.predict_proba(
            X_val
        )[:, 1]

        p_test = model.predict_proba(
            X_test
        )[:, 1]

        val_auc = roc_auc_score(
            y_val,
            p_val
        )

        test_auc = roc_auc_score(
            y_test,
            p_test
        )

        val_brier = brier_score_loss(
            y_val,
            p_val
        )

        test_brier = brier_score_loss(
            y_test,
            p_test
        )

        val_logloss = log_loss(
            y_val,
            p_val,
            labels=[0, 1]
        )

        test_logloss = log_loss(
            y_test,
            p_test,
            labels=[0, 1]
        )

        results_27.append({
            "training_n": n_train,
            "model": model_name,
            "train_event_rate":
                y_train.mean(),
            "val_auc":
                val_auc,
            "test_auc":
                test_auc,
            "val_brier":
                val_brier,
            "test_brier":
                test_brier,
            "val_logloss":
                val_logloss,
            "test_logloss":
                test_logloss,
            "fit_seconds":
                fit_seconds
        })

        prediction_store_27[
            (n_train, model_name)
        ] = {
            "val": p_val.copy(),
            "test": p_test.copy()
        }

        print(
            f"    Val AUC  : {val_auc:.6f}"
        )

        print(
            f"    Test AUC : {test_auc:.6f}"
        )

        print(
            f"    Test Brier: {test_brier:.6f}"
        )

        print(
            f"    Test log loss: {test_logloss:.6f}"
        )

        print(
            f"    Fit time: {fit_seconds:.1f} sec"
        )

    del X_train
    gc.collect()


# ------------------------------------------------------------
# I. Results table
# ------------------------------------------------------------

results_27 = pd.DataFrame(
    results_27
)

print("\n" + "=" * 110)
print("E. COMPLETE TRAINING-SIZE RESULTS")
print("=" * 110)

print(
    results_27.to_string(
        index=False,
        formatters={
            "train_event_rate":
                "{:.6f}".format,
            "val_auc":
                "{:.6f}".format,
            "test_auc":
                "{:.6f}".format,
            "val_brier":
                "{:.6f}".format,
            "test_brier":
                "{:.6f}".format,
            "val_logloss":
                "{:.6f}".format,
            "test_logloss":
                "{:.6f}".format,
            "fit_seconds":
                "{:.1f}".format
        }
    )
)


# ------------------------------------------------------------
# J. Performance change relative to 180k
# ------------------------------------------------------------

baseline = (
    results_27[
        results_27["training_n"]
        == 180_000
    ][
        [
            "model",
            "test_auc",
            "test_brier",
            "test_logloss"
        ]
    ]
    .rename(
        columns={
            "test_auc":
                "auc_180k",
            "test_brier":
                "brier_180k",
            "test_logloss":
                "logloss_180k"
        }
    )
)

delta_27 = (
    results_27
    .merge(
        baseline,
        on="model",
        how="left"
    )
)

delta_27["delta_auc_vs_180k"] = (
    delta_27["test_auc"]
    -
    delta_27["auc_180k"]
)

delta_27["delta_brier_vs_180k"] = (
    delta_27["test_brier"]
    -
    delta_27["brier_180k"]
)

delta_27["delta_logloss_vs_180k"] = (
    delta_27["test_logloss"]
    -
    delta_27["logloss_180k"]
)


print("\n" + "=" * 110)
print("F. CHANGE RELATIVE TO 180k TRAINING SAMPLE")
print("=" * 110)

print(
    delta_27[
        [
            "training_n",
            "model",
            "test_auc",
            "delta_auc_vs_180k",
            "test_brier",
            "delta_brier_vs_180k",
            "test_logloss",
            "delta_logloss_vs_180k"
        ]
    ].to_string(
        index=False,
        formatters={
            "test_auc":
                "{:.6f}".format,
            "delta_auc_vs_180k":
                "{:+.6f}".format,
            "test_brier":
                "{:.6f}".format,
            "delta_brier_vs_180k":
                "{:+.6f}".format,
            "test_logloss":
                "{:.6f}".format,
            "delta_logloss_vs_180k":
                "{:+.6f}".format
        }
    )
)


# ------------------------------------------------------------
# K. Save results
# ------------------------------------------------------------

results_27.to_csv(
    OUTPUT_DIR /
    "cell27_training_size_sensitivity.csv",
    index=False
)

delta_27.to_csv(
    OUTPUT_DIR /
    "cell27_training_size_deltas.csv",
    index=False
)


# Save fixed labels for later paired bootstrap
np.save(
    OUTPUT_DIR /
    "cell27_y_val.npy",
    y_val
)

np.save(
    OUTPUT_DIR /
    "cell27_y_test.npy",
    y_test
)


# Save test/validation probabilities
for (n_train, model_name), preds in prediction_store_27.items():

    safe_model = (
        model_name
        .replace(" ", "_")
    )

    np.save(
        OUTPUT_DIR /
        f"cell27_{n_train}_{safe_model}_val_pred.npy",
        preds["val"]
    )

    np.save(
        OUTPUT_DIR /
        f"cell27_{n_train}_{safe_model}_test_pred.npy",
        preds["test"]
    )


print("\nSaved Cell 27 results and predictions to:")
print(OUTPUT_DIR)

print(
    "\nIMPORTANT: Do not interpret 'no capacity premium' yet."
)

print(
    "The next analysis will use these SAME test predictions "
    "for paired bootstrap model comparisons."
)

A. FIXED MODELLING COHORTS
Train mature: 1,304,058 | event rate: 0.138936
Validation mature: 27,374 | event rate: 0.101227
Test mature: 41,145 | event rate: 0.146433

B. REVISED TRAINING FEATURE SET
Locked Cell 26 features: 153
High-cardinality free-text exclusions: ['emp_title', 'desc', 'title']
Features entering models: 150
Numeric: 136
Categorical: 14

C. FITTING PREPROCESSOR
Preprocessor fitted in 33.6 seconds

Transforming validation/test...
Validation matrix: (27374, 384)
Test matrix: (41145, 384)

D. TRAINING-SIZE SENSITIVITY

--------------------------------------------------------------------------------------------------------------
Training size: 180,000
--------------------------------------------------------------------------------------------------------------
Transforming training subset...
Training matrix: (180000, 384) | transform seconds: 1.3
Training event rate: 0.137606

  Fitting Logistic_SGD...
    Val AUC  : 0.610769
    Test AUC : 0.646062
    Test Brier: 0.1575

In [32]:
# ============================================================
# CELL 28 — Paired bootstrap model comparison
#
# Reviewer requirement:
#   Paired statistical comparison of Logistic vs HistGB vs MLP
#   with confidence intervals for performance differences.
#
# Uses SAME test observations and predictions from Cell 27.
# NO RETRAINING.
#
# Metrics:
#   ROC-AUC difference
#   Brier-score difference
#   Log-loss difference
#
# Bootstrap:
#   Stratified paired bootstrap
#   2,000 replicates
#   seed = 2026
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss
)


# ------------------------------------------------------------
# A. Load fixed test labels
# ------------------------------------------------------------

y = np.load(
    OUTPUT_DIR /
    "cell27_y_test.npy"
).astype(int)

print("=" * 110)
print("A. FIXED TEST COHORT")
print("=" * 110)

print("N:", f"{len(y):,}")
print("Defaults:", f"{y.sum():,}")
print("Non-defaults:", f"{(y == 0).sum():,}")
print("Default rate:", f"{y.mean():.6f}")


# ------------------------------------------------------------
# B. Training sizes to compare
#
# 180k = manuscript-motivated sensitivity point
# full = revised maximum-data specification
# ------------------------------------------------------------

FULL_N = 1_304_058

sizes_to_compare = [
    180_000,
    FULL_N
]

models = [
    "Logistic_SGD",
    "HistGB",
    "MLP"
]

pairs = [
    ("Logistic_SGD", "HistGB"),
    ("Logistic_SGD", "MLP"),
    ("HistGB", "MLP")
]


# ------------------------------------------------------------
# C. Load predictions
# ------------------------------------------------------------

pred = {}

for n_train in sizes_to_compare:

    for model in models:

        path = (
            OUTPUT_DIR /
            f"cell27_{n_train}_{model}_test_pred.npy"
        )

        pred[(n_train, model)] = np.load(path)

        assert len(pred[(n_train, model)]) == len(y)


print("\nPredictions loaded successfully.")


# ------------------------------------------------------------
# D. Original point estimates
# ------------------------------------------------------------

point_rows = []

for n_train in sizes_to_compare:

    for model in models:

        p = pred[(n_train, model)]

        point_rows.append({
            "training_n": n_train,
            "model": model,
            "auc": roc_auc_score(y, p),
            "brier": brier_score_loss(y, p),
            "logloss": log_loss(
                y,
                p,
                labels=[0, 1]
            )
        })


point_df = pd.DataFrame(point_rows)

print("\n" + "=" * 110)
print("B. ORIGINAL TEST PERFORMANCE")
print("=" * 110)

print(
    point_df.to_string(
        index=False,
        formatters={
            "auc": "{:.6f}".format,
            "brier": "{:.6f}".format,
            "logloss": "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# E. Stratified paired bootstrap indices
#
# Same sampled observations are used for every model in each
# replicate, preserving the paired comparison.
# ------------------------------------------------------------

pos_idx = np.flatnonzero(y == 1)
neg_idx = np.flatnonzero(y == 0)

n_pos = len(pos_idx)
n_neg = len(neg_idx)

B = 2000
rng = np.random.default_rng(2026)

print("\n" + "=" * 110)
print("C. PAIRED STRATIFIED BOOTSTRAP")
print("=" * 110)

print("Replicates:", B)
print("Positive observations:", n_pos)
print("Negative observations:", n_neg)


# ------------------------------------------------------------
# F. Bootstrap comparisons
#
# Difference convention:
#   AUC difference    = model_A - model_B
#
#   Brier difference  = model_A - model_B
#   Log-loss difference = model_A - model_B
#
# Therefore:
#   AUC > 0 favours model A.
#   Brier/log-loss < 0 favours model A.
# ------------------------------------------------------------

boot_store = {}

for n_train in sizes_to_compare:

    for model_a, model_b in pairs:

        key = (
            n_train,
            model_a,
            model_b
        )

        boot_store[key] = {
            "auc_diff": np.empty(B),
            "brier_diff": np.empty(B),
            "logloss_diff": np.empty(B)
        }


for b in range(B):

    sampled_pos = rng.choice(
        pos_idx,
        size=n_pos,
        replace=True
    )

    sampled_neg = rng.choice(
        neg_idx,
        size=n_neg,
        replace=True
    )

    idx = np.concatenate([
        sampled_pos,
        sampled_neg
    ])

    y_b = y[idx]

    for n_train in sizes_to_compare:

        # calculate metrics once per model/replicate
        metrics_b = {}

        for model in models:

            p_b = pred[(n_train, model)][idx]

            metrics_b[model] = {
                "auc":
                    roc_auc_score(
                        y_b,
                        p_b
                    ),

                "brier":
                    brier_score_loss(
                        y_b,
                        p_b
                    ),

                "logloss":
                    log_loss(
                        y_b,
                        p_b,
                        labels=[0, 1]
                    )
            }

        for model_a, model_b in pairs:

            store = boot_store[
                (
                    n_train,
                    model_a,
                    model_b
                )
            ]

            store["auc_diff"][b] = (
                metrics_b[model_a]["auc"]
                -
                metrics_b[model_b]["auc"]
            )

            store["brier_diff"][b] = (
                metrics_b[model_a]["brier"]
                -
                metrics_b[model_b]["brier"]
            )

            store["logloss_diff"][b] = (
                metrics_b[model_a]["logloss"]
                -
                metrics_b[model_b]["logloss"]
            )

    if (b + 1) % 250 == 0:
        print(
            f"Completed {b + 1:,}/{B:,} bootstrap replicates"
        )


# ------------------------------------------------------------
# G. Summarise paired differences
# ------------------------------------------------------------

summary_rows = []

for n_train in sizes_to_compare:

    for model_a, model_b in pairs:

        pa = pred[(n_train, model_a)]
        pb = pred[(n_train, model_b)]

        original_auc_diff = (
            roc_auc_score(y, pa)
            -
            roc_auc_score(y, pb)
        )

        original_brier_diff = (
            brier_score_loss(y, pa)
            -
            brier_score_loss(y, pb)
        )

        original_logloss_diff = (
            log_loss(y, pa, labels=[0, 1])
            -
            log_loss(y, pb, labels=[0, 1])
        )

        store = boot_store[
            (
                n_train,
                model_a,
                model_b
            )
        ]

        auc_low, auc_high = np.percentile(
            store["auc_diff"],
            [2.5, 97.5]
        )

        brier_low, brier_high = np.percentile(
            store["brier_diff"],
            [2.5, 97.5]
        )

        log_low, log_high = np.percentile(
            store["logloss_diff"],
            [2.5, 97.5]
        )

        # Two-sided bootstrap sign-based p-value.
        # Add-one correction prevents zero p-values.
        auc_le_zero = np.sum(
            store["auc_diff"] <= 0
        )

        auc_ge_zero = np.sum(
            store["auc_diff"] >= 0
        )

        auc_p = min(
            1.0,
            2 * min(
                (auc_le_zero + 1) / (B + 1),
                (auc_ge_zero + 1) / (B + 1)
            )
        )

        summary_rows.append({
            "training_n": n_train,
            "model_A": model_a,
            "model_B": model_b,

            "auc_diff_A_minus_B":
                original_auc_diff,
            "auc_ci_low":
                auc_low,
            "auc_ci_high":
                auc_high,
            "auc_bootstrap_p":
                auc_p,

            "brier_diff_A_minus_B":
                original_brier_diff,
            "brier_ci_low":
                brier_low,
            "brier_ci_high":
                brier_high,

            "logloss_diff_A_minus_B":
                original_logloss_diff,
            "logloss_ci_low":
                log_low,
            "logloss_ci_high":
                log_high
        })


paired_summary = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# H. Print reviewer-facing results
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("D. PAIRED MODEL COMPARISONS")
print("=" * 110)

print(
    paired_summary.to_string(
        index=False,
        formatters={
            "auc_diff_A_minus_B":
                "{:+.6f}".format,
            "auc_ci_low":
                "{:+.6f}".format,
            "auc_ci_high":
                "{:+.6f}".format,
            "auc_bootstrap_p":
                "{:.6f}".format,

            "brier_diff_A_minus_B":
                "{:+.6f}".format,
            "brier_ci_low":
                "{:+.6f}".format,
            "brier_ci_high":
                "{:+.6f}".format,

            "logloss_diff_A_minus_B":
                "{:+.6f}".format,
            "logloss_ci_low":
                "{:+.6f}".format,
            "logloss_ci_high":
                "{:+.6f}".format
        }
    )
)


# ------------------------------------------------------------
# I. Simple interpretation flags
# ------------------------------------------------------------

def ci_excludes_zero(low, high):
    return (low > 0) or (high < 0)


paired_summary[
    "auc_difference_supported"
] = paired_summary.apply(
    lambda r:
        ci_excludes_zero(
            r["auc_ci_low"],
            r["auc_ci_high"]
        ),
    axis=1
)


print("\n" + "=" * 110)
print("E. AUC DIFFERENCE FLAGS")
print("=" * 110)

print(
    paired_summary[
        [
            "training_n",
            "model_A",
            "model_B",
            "auc_diff_A_minus_B",
            "auc_ci_low",
            "auc_ci_high",
            "auc_bootstrap_p",
            "auc_difference_supported"
        ]
    ].to_string(
        index=False,
        formatters={
            "auc_diff_A_minus_B":
                "{:+.6f}".format,
            "auc_ci_low":
                "{:+.6f}".format,
            "auc_ci_high":
                "{:+.6f}".format,
            "auc_bootstrap_p":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# J. Save
# ------------------------------------------------------------

point_df.to_csv(
    OUTPUT_DIR /
    "cell28_test_performance_point_estimates.csv",
    index=False
)

paired_summary.to_csv(
    OUTPUT_DIR /
    "cell28_paired_bootstrap_model_comparisons.csv",
    index=False
)


print("\nSaved Cell 28 outputs to:")
print(OUTPUT_DIR)

print(
    "\nDo NOT use the phrase 'no capacity premium' "
    "until these paired results are interpreted."
)

A. FIXED TEST COHORT
N: 41,145
Defaults: 6,025
Non-defaults: 35,120
Default rate: 0.146433

Predictions loaded successfully.

B. ORIGINAL TEST PERFORMANCE
 training_n        model      auc    brier  logloss
     180000 Logistic_SGD 0.646062 0.157529 0.515784
     180000       HistGB 0.697660 0.134361 0.500605
     180000          MLP 0.718945 0.116114 0.388663
    1304058 Logistic_SGD 0.699310 0.129401 0.420523
    1304058       HistGB 0.675016 0.136718 0.529232
    1304058          MLP 0.646276 0.131485 0.503209

C. PAIRED STRATIFIED BOOTSTRAP
Replicates: 2000
Positive observations: 6025
Negative observations: 35120
Completed 250/2,000 bootstrap replicates
Completed 500/2,000 bootstrap replicates
Completed 750/2,000 bootstrap replicates
Completed 1,000/2,000 bootstrap replicates
Completed 1,250/2,000 bootstrap replicates
Completed 1,500/2,000 bootstrap replicates
Completed 1,750/2,000 bootstrap replicates
Completed 2,000/2,000 bootstrap replicates

D. PAIRED MODEL COMPARISONS
 trainin

In [33]:
# ============================================================
# CELL 29 — Validation-only recalibration
#
# Selected model:
#   Full-training HistGB
#
# Selection basis:
#   Highest validation ROC-AUC among full-training models.
#
# Recalibration:
#   Logistic/Platt recalibration fitted ONLY on validation:
#
#   logit(Y) = a + b * logit(p_raw)
#
#   The fitted mapping is then applied unchanged to test.
#
# NO TEST INFORMATION IS USED TO FIT RECALIBRATION.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss
)


# ------------------------------------------------------------
# A. Load fixed labels and selected-model predictions
# ------------------------------------------------------------

FULL_N = 1_304_058
SELECTED_MODEL = "HistGB"

y_val = np.load(
    OUTPUT_DIR / "cell27_y_val.npy"
).astype(int)

y_test = np.load(
    OUTPUT_DIR / "cell27_y_test.npy"
).astype(int)

p_val_raw = np.load(
    OUTPUT_DIR /
    f"cell27_{FULL_N}_{SELECTED_MODEL}_val_pred.npy"
)

p_test_raw = np.load(
    OUTPUT_DIR /
    f"cell27_{FULL_N}_{SELECTED_MODEL}_test_pred.npy"
)


print("=" * 110)
print("A. RECALIBRATION COHORTS")
print("=" * 110)

print(
    "Selected model:",
    SELECTED_MODEL
)

print(
    "Validation N:",
    f"{len(y_val):,}",
    "| event rate:",
    f"{y_val.mean():.6f}",
    "| raw mean prediction:",
    f"{p_val_raw.mean():.6f}"
)

print(
    "Test N:",
    f"{len(y_test):,}",
    "| event rate:",
    f"{y_test.mean():.6f}",
    "| raw mean prediction:",
    f"{p_test_raw.mean():.6f}"
)


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

EPS = 1e-6


def clip_prob(p):
    return np.clip(
        np.asarray(p, dtype=float),
        EPS,
        1 - EPS
    )


def logit(p):
    p = clip_prob(p)
    return np.log(
        p / (1 - p)
    )


def sigmoid(z):
    return 1 / (
        1 + np.exp(-z)
    )


def calibration_intercept_slope(y, p):
    """
    Logistic calibration model:
        logit(Y) = intercept + slope * logit(p)

    Ideal:
        intercept = 0
        slope = 1
    """

    x = logit(p).reshape(-1, 1)

    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000
    )

    model.fit(
        x,
        y
    )

    return (
        float(model.intercept_[0]),
        float(model.coef_[0, 0])
    )


def ece_equal_width(y, p, n_bins=10):
    """
    Expected calibration error using 10 equal-width
    probability bins.
    """

    p = np.asarray(p)
    y = np.asarray(y)

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    bin_id = np.digitize(
        p,
        edges[1:-1],
        right=False
    )

    ece = 0.0

    rows = []

    for b in range(n_bins):

        mask = (
            bin_id == b
        )

        n = int(mask.sum())

        if n == 0:
            continue

        mean_p = float(
            p[mask].mean()
        )

        event_rate = float(
            y[mask].mean()
        )

        weight = (
            n / len(y)
        )

        gap = abs(
            event_rate - mean_p
        )

        ece += (
            weight * gap
        )

        rows.append({
            "bin": b + 1,
            "lower": edges[b],
            "upper": edges[b + 1],
            "n": n,
            "mean_prediction": mean_p,
            "event_rate": event_rate,
            "abs_gap": gap
        })

    return float(ece), pd.DataFrame(rows)


def calibration_metrics(y, p):

    p = clip_prob(p)

    intercept, slope = (
        calibration_intercept_slope(
            y,
            p
        )
    )

    ece, _ = ece_equal_width(
        y,
        p,
        n_bins=10
    )

    return {
        "n": len(y),
        "event_rate":
            float(np.mean(y)),
        "mean_prediction":
            float(np.mean(p)),
        "auc":
            roc_auc_score(y, p),
        "brier":
            brier_score_loss(y, p),
        "logloss":
            log_loss(
                y,
                p,
                labels=[0, 1]
            ),
        "ece":
            ece,
        "calibration_intercept":
            intercept,
        "calibration_slope":
            slope
    }


# ------------------------------------------------------------
# C. Fit Platt/logistic recalibration ONLY on validation
# ------------------------------------------------------------

X_cal_val = (
    logit(p_val_raw)
    .reshape(-1, 1)
)

recalibrator = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000
)

recalibrator.fit(
    X_cal_val,
    y_val
)

recal_intercept = float(
    recalibrator.intercept_[0]
)

recal_slope = float(
    recalibrator.coef_[0, 0]
)


print("\n" + "=" * 110)
print("B. VALIDATION-FITTED RECALIBRATION PARAMETERS")
print("=" * 110)

print(
    "Intercept a:",
    f"{recal_intercept:.6f}"
)

print(
    "Slope b:",
    f"{recal_slope:.6f}"
)


# ------------------------------------------------------------
# D. Apply unchanged mapping
# ------------------------------------------------------------

p_val_cal = recalibrator.predict_proba(
    logit(p_val_raw).reshape(-1, 1)
)[:, 1]

p_test_cal = recalibrator.predict_proba(
    logit(p_test_raw).reshape(-1, 1)
)[:, 1]


# ------------------------------------------------------------
# E. Before/after metrics
# ------------------------------------------------------------

rows = []

for cohort_name, y, p_raw, p_cal in [
    (
        "Validation",
        y_val,
        p_val_raw,
        p_val_cal
    ),
    (
        "Test",
        y_test,
        p_test_raw,
        p_test_cal
    )
]:

    raw_metrics = calibration_metrics(
        y,
        p_raw
    )

    cal_metrics = calibration_metrics(
        y,
        p_cal
    )

    rows.append({
        "cohort": cohort_name,
        "stage": "Before",
        **raw_metrics
    })

    rows.append({
        "cohort": cohort_name,
        "stage": "After",
        **cal_metrics
    })


calibration_table = pd.DataFrame(
    rows
)


print("\n" + "=" * 110)
print("C. BEFORE / AFTER CALIBRATION")
print("=" * 110)

print(
    calibration_table.to_string(
        index=False,
        formatters={
            "event_rate":
                "{:.6f}".format,
            "mean_prediction":
                "{:.6f}".format,
            "auc":
                "{:.6f}".format,
            "brier":
                "{:.6f}".format,
            "logloss":
                "{:.6f}".format,
            "ece":
                "{:.6f}".format,
            "calibration_intercept":
                "{:+.6f}".format,
            "calibration_slope":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# F. Test improvement table
# ------------------------------------------------------------

test_before = calibration_table[
    (calibration_table["cohort"] == "Test") &
    (calibration_table["stage"] == "Before")
].iloc[0]

test_after = calibration_table[
    (calibration_table["cohort"] == "Test") &
    (calibration_table["stage"] == "After")
].iloc[0]


improvement = pd.DataFrame([{
    "delta_mean_prediction":
        test_after["mean_prediction"]
        -
        test_before["mean_prediction"],

    "delta_auc":
        test_after["auc"]
        -
        test_before["auc"],

    "delta_brier":
        test_after["brier"]
        -
        test_before["brier"],

    "delta_logloss":
        test_after["logloss"]
        -
        test_before["logloss"],

    "delta_ece":
        test_after["ece"]
        -
        test_before["ece"],

    "intercept_before":
        test_before["calibration_intercept"],

    "intercept_after":
        test_after["calibration_intercept"],

    "slope_before":
        test_before["calibration_slope"],

    "slope_after":
        test_after["calibration_slope"]
}])


print("\n" + "=" * 110)
print("D. TEST CHANGE AFTER VALIDATION-ONLY RECALIBRATION")
print("=" * 110)

print(
    improvement.to_string(
        index=False,
        formatters={
            c: "{:+.6f}".format
            for c in improvement.columns
        }
    )
)


# ------------------------------------------------------------
# G. Reliability-bin tables
# ------------------------------------------------------------

_, val_before_bins = ece_equal_width(
    y_val,
    p_val_raw
)

_, val_after_bins = ece_equal_width(
    y_val,
    p_val_cal
)

_, test_before_bins = ece_equal_width(
    y_test,
    p_test_raw
)

_, test_after_bins = ece_equal_width(
    y_test,
    p_test_cal
)


print("\n" + "=" * 110)
print("E. TEST RELIABILITY BINS — BEFORE")
print("=" * 110)

print(
    test_before_bins.to_string(
        index=False,
        formatters={
            "lower": "{:.2f}".format,
            "upper": "{:.2f}".format,
            "mean_prediction":
                "{:.6f}".format,
            "event_rate":
                "{:.6f}".format,
            "abs_gap":
                "{:.6f}".format
        }
    )
)


print("\n" + "=" * 110)
print("F. TEST RELIABILITY BINS — AFTER")
print("=" * 110)

print(
    test_after_bins.to_string(
        index=False,
        formatters={
            "lower": "{:.2f}".format,
            "upper": "{:.2f}".format,
            "mean_prediction":
                "{:.6f}".format,
            "event_rate":
                "{:.6f}".format,
            "abs_gap":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# H. Save probabilities for threshold analysis
# ------------------------------------------------------------

np.save(
    OUTPUT_DIR /
    "cell29_histgb_full_val_raw.npy",
    p_val_raw
)

np.save(
    OUTPUT_DIR /
    "cell29_histgb_full_val_recalibrated.npy",
    p_val_cal
)

np.save(
    OUTPUT_DIR /
    "cell29_histgb_full_test_raw.npy",
    p_test_raw
)

np.save(
    OUTPUT_DIR /
    "cell29_histgb_full_test_recalibrated.npy",
    p_test_cal
)


# ------------------------------------------------------------
# I. Save tables
# ------------------------------------------------------------

calibration_table.to_csv(
    OUTPUT_DIR /
    "cell29_before_after_calibration.csv",
    index=False
)

improvement.to_csv(
    OUTPUT_DIR /
    "cell29_test_calibration_change.csv",
    index=False
)

val_before_bins.to_csv(
    OUTPUT_DIR /
    "cell29_validation_reliability_before.csv",
    index=False
)

val_after_bins.to_csv(
    OUTPUT_DIR /
    "cell29_validation_reliability_after.csv",
    index=False
)

test_before_bins.to_csv(
    OUTPUT_DIR /
    "cell29_test_reliability_before.csv",
    index=False
)

test_after_bins.to_csv(
    OUTPUT_DIR /
    "cell29_test_reliability_after.csv",
    index=False
)


print("\nSaved Cell 29 outputs to:")
print(OUTPUT_DIR)

print(
    "\nNext step after inspection: "
    "recalculate 5:1, 10:1 and 20:1 thresholds "
    "using validation-recalibrated probabilities."
)

A. RECALIBRATION COHORTS
Selected model: HistGB
Validation N: 27,374 | event rate: 0.101227 | raw mean prediction: 0.032020
Test N: 41,145 | event rate: 0.146433 | raw mean prediction: 0.027694

B. VALIDATION-FITTED RECALIBRATION PARAMETERS
Intercept a: 1.445016
Slope b: 1.056969

C. BEFORE / AFTER CALIBRATION
    cohort  stage     n event_rate mean_prediction      auc    brier  logloss      ece calibration_intercept calibration_slope
Validation Before 27374   0.101227        0.032020 0.696648 0.093320 0.355862 0.069230             +1.445016          1.056969
Validation  After 27374   0.101227        0.101225 0.696648 0.086502 0.304759 0.002905             +0.000162          1.000037
      Test Before 41145   0.146433        0.027694 0.675016 0.136718 0.529232 0.118783             +1.581336          0.922334
      Test  After 41145   0.146433        0.088197 0.675016 0.123161 0.412280 0.058325             +0.319362          0.872164

D. TEST CHANGE AFTER VALIDATION-ONLY RECALIBRATION
d

In [34]:
# ============================================================
# CELL 30 — Recalibrated threshold / decision analysis
#
# Thresholds are selected ONLY on the validation cohort using
# validation-recalibrated HistGB probabilities.
#
# The selected threshold is then applied unchanged to test.
#
# Cost convention:
#   FP cost = 1
#   FN cost = {5, 10, 20}
#
# Review/flag:
#   predicted probability >= threshold
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# A. Load recalibrated probabilities and labels
# ------------------------------------------------------------

y_val = np.load(
    OUTPUT_DIR / "cell27_y_val.npy"
).astype(int)

y_test = np.load(
    OUTPUT_DIR / "cell27_y_test.npy"
).astype(int)

p_val = np.load(
    OUTPUT_DIR /
    "cell29_histgb_full_val_recalibrated.npy"
)

p_test = np.load(
    OUTPUT_DIR /
    "cell29_histgb_full_test_recalibrated.npy"
)


print("=" * 110)
print("A. INPUT CHECK")
print("=" * 110)

print(
    "Validation:",
    f"N={len(y_val):,}",
    f"default rate={y_val.mean():.6f}",
    f"mean prediction={p_val.mean():.6f}"
)

print(
    "Test:",
    f"N={len(y_test):,}",
    f"default rate={y_test.mean():.6f}",
    f"mean prediction={p_test.mean():.6f}"
)


# ------------------------------------------------------------
# B. Decision metrics
# ------------------------------------------------------------

def decision_metrics(y, p, threshold, fn_cost, fp_cost=1.0):

    flagged = p >= threshold

    tp = int(
        np.sum(
            flagged & (y == 1)
        )
    )

    fp = int(
        np.sum(
            flagged & (y == 0)
        )
    )

    tn = int(
        np.sum(
            (~flagged) & (y == 0)
        )
    )

    fn = int(
        np.sum(
            (~flagged) & (y == 1)
        )
    )

    n = len(y)

    positives = (
        tp + fn
    )

    flagged_n = (
        tp + fp
    )

    fnr = (
        fn / positives
        if positives > 0
        else np.nan
    )

    tpr = (
        tp / positives
        if positives > 0
        else np.nan
    )

    review_rate = (
        flagged_n / n
    )

    precision = (
        tp / flagged_n
        if flagged_n > 0
        else np.nan
    )

    cost = (
        fn_cost * fn
        +
        fp_cost * fp
    )

    no_review_cost = (
        fn_cost
        * positives
    )

    net_benefit = (
        no_review_cost
        -
        cost
    )

    return {
        "threshold": threshold,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "fnr": fnr,
        "tpr": tpr,
        "review_rate": review_rate,
        "precision": precision,
        "cost": cost,
        "no_review_cost": no_review_cost,
        "net_benefit": net_benefit
    }


# ------------------------------------------------------------
# C. Exact validation threshold search
#
# For a threshold classifier, predictions change only when the
# threshold crosses an observed predicted probability.
#
# We therefore evaluate all unique validation probabilities,
# plus boundary operating points.
# ------------------------------------------------------------

def find_cost_minimising_threshold(
    y,
    p,
    fn_cost,
    fp_cost=1.0
):

    order = np.argsort(
        -p,
        kind="mergesort"
    )

    p_sorted = p[order]
    y_sorted = y[order]

    total_pos = int(
        y_sorted.sum()
    )

    total_neg = int(
        len(y_sorted)
        -
        total_pos
    )

    cum_tp = np.cumsum(
        y_sorted == 1
    )

    cum_fp = np.cumsum(
        y_sorted == 0
    )

    # Last position for each unique probability
    is_last_of_value = np.r_[
        p_sorted[:-1] != p_sorted[1:],
        True
    ]

    positions = np.flatnonzero(
        is_last_of_value
    )

    thresholds = p_sorted[
        positions
    ]

    tp = cum_tp[
        positions
    ]

    fp = cum_fp[
        positions
    ]

    fn = (
        total_pos
        -
        tp
    )

    costs = (
        fn_cost * fn
        +
        fp_cost * fp
    )

    best_cost = costs.min()

    best_positions = np.flatnonzero(
        costs == best_cost
    )

    # Tie rule:
    # choose the HIGHEST threshold among equal-cost solutions,
    # i.e. the lower review burden.
    best_local = best_positions[
        np.argmax(
            thresholds[
                best_positions
            ]
        )
    ]

    best_threshold = float(
        thresholds[
            best_local
        ]
    )

    return (
        best_threshold,
        float(best_cost)
    )


# ------------------------------------------------------------
# D. Select threshold on validation, evaluate on both cohorts
# ------------------------------------------------------------

COST_RATIOS = [
    5,
    10,
    20
]

rows = []


print("\n" + "=" * 110)
print("B. VALIDATION-SELECTED RECALIBRATED THRESHOLDS")
print("=" * 110)


for fn_cost in COST_RATIOS:

    threshold, validation_min_cost = (
        find_cost_minimising_threshold(
            y_val,
            p_val,
            fn_cost=fn_cost,
            fp_cost=1.0
        )
    )

    val_metrics = decision_metrics(
        y_val,
        p_val,
        threshold,
        fn_cost=fn_cost
    )

    test_metrics = decision_metrics(
        y_test,
        p_test,
        threshold,
        fn_cost=fn_cost
    )

    print(
        f"\nFN:FP = {fn_cost}:1"
    )

    print(
        f"  Selected validation threshold: "
        f"{threshold:.6f}"
    )

    print(
        f"  Validation FNR: "
        f"{val_metrics['fnr']:.6f}"
    )

    print(
        f"  Validation review rate: "
        f"{val_metrics['review_rate']:.6f}"
    )

    print(
        f"  Test FNR: "
        f"{test_metrics['fnr']:.6f}"
    )

    print(
        f"  Test review rate: "
        f"{test_metrics['review_rate']:.6f}"
    )

    print(
        f"  Test precision: "
        f"{test_metrics['precision']:.6f}"
    )

    print(
        f"  Test cost: "
        f"{test_metrics['cost']:.1f}"
    )

    print(
        f"  Test no-review cost: "
        f"{test_metrics['no_review_cost']:.1f}"
    )

    print(
        f"  Test net benefit: "
        f"{test_metrics['net_benefit']:.1f}"
    )

    for cohort, metrics in [
        ("Validation", val_metrics),
        ("Test", test_metrics)
    ]:

        rows.append({
            "fn_fp_ratio":
                f"{fn_cost}:1",
            "fn_cost":
                fn_cost,
            "fp_cost":
                1.0,
            "threshold":
                threshold,
            "threshold_selected_on":
                "Validation",
            "cohort":
                cohort,
            **{
                k: v
                for k, v in metrics.items()
                if k != "threshold"
            }
        })


threshold_results = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# E. Reviewer-facing test table
# ------------------------------------------------------------

test_threshold_table = (
    threshold_results[
        threshold_results["cohort"]
        == "Test"
    ]
    .copy()
)


print("\n" + "=" * 110)
print("C. TEST PERFORMANCE AT VALIDATION-SELECTED THRESHOLDS")
print("=" * 110)

display_cols = [
    "fn_fp_ratio",
    "threshold",
    "tp",
    "fp",
    "tn",
    "fn",
    "fnr",
    "review_rate",
    "precision",
    "cost",
    "no_review_cost",
    "net_benefit"
]

print(
    test_threshold_table[
        display_cols
    ].to_string(
        index=False,
        formatters={
            "threshold":
                "{:.6f}".format,
            "fnr":
                "{:.6f}".format,
            "review_rate":
                "{:.6f}".format,
            "precision":
                "{:.6f}".format,
            "cost":
                "{:.1f}".format,
            "no_review_cost":
                "{:.1f}".format,
            "net_benefit":
                "{:.1f}".format
        }
    )
)


# ------------------------------------------------------------
# F. Compare with ORIGINAL manuscript operating points
#
# These are for descriptive comparison only.
# ------------------------------------------------------------

old_results = pd.DataFrame({
    "fn_fp_ratio": [
        "5:1",
        "10:1",
        "20:1"
    ],
    "old_threshold": [
        0.1984,
        0.0723,
        0.0292
    ],
    "old_test_fnr": [
        0.9695,
        0.3786,
        0.0790
    ],
    "old_review_rate": [
        0.0129,
        0.4034,
        0.7822
    ],
    "old_precision": [
        0.3465,
        0.2256,
        0.1724
    ]
})


comparison = (
    test_threshold_table[
        [
            "fn_fp_ratio",
            "threshold",
            "fnr",
            "review_rate",
            "precision"
        ]
    ]
    .rename(
        columns={
            "threshold":
                "new_threshold",
            "fnr":
                "new_test_fnr",
            "review_rate":
                "new_review_rate",
            "precision":
                "new_precision"
        }
    )
    .merge(
        old_results,
        on="fn_fp_ratio",
        how="left"
    )
)


print("\n" + "=" * 110)
print("D. OLD MANUSCRIPT vs RECALIBRATED ANALYSIS")
print("=" * 110)

print(
    comparison.to_string(
        index=False,
        formatters={
            "new_threshold":
                "{:.6f}".format,
            "new_test_fnr":
                "{:.6f}".format,
            "new_review_rate":
                "{:.6f}".format,
            "new_precision":
                "{:.6f}".format,
            "old_threshold":
                "{:.6f}".format,
            "old_test_fnr":
                "{:.6f}".format,
            "old_review_rate":
                "{:.6f}".format,
            "old_precision":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# G. Save
# ------------------------------------------------------------

threshold_results.to_csv(
    OUTPUT_DIR /
    "cell30_recalibrated_threshold_analysis.csv",
    index=False
)

test_threshold_table.to_csv(
    OUTPUT_DIR /
    "cell30_recalibrated_test_operating_points.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR /
    "cell30_old_vs_recalibrated_thresholds.csv",
    index=False
)


print("\nSaved Cell 30 outputs to:")
print(OUTPUT_DIR)

print(
    "\nAfter this cell, only ONE coding block remains: "
    "stress-definition resolution + stress/slice robustness."
)

A. INPUT CHECK
Validation: N=27,374 default rate=0.101227 mean prediction=0.101225
Test: N=41,145 default rate=0.146433 mean prediction=0.088197

B. VALIDATION-SELECTED RECALIBRATED THRESHOLDS

FN:FP = 5:1
  Selected validation threshold: 0.172966
  Validation FNR: 0.700830
  Validation review rate: 0.126726
  Test FNR: 0.843485
  Test review rate: 0.077458
  Test precision: 0.295890
  Test cost: 27654.0
  Test no-review cost: 30125.0
  Test net benefit: 2471.0

FN:FP = 10:1
  Selected validation threshold: 0.096162
  Validation FNR: 0.269939
  Validation review rate: 0.475561
  Test FNR: 0.413610
  Test review rate: 0.375307
  Test precision: 0.228792
  Test cost: 36829.0
  Test no-review cost: 60250.0
  Test net benefit: 23421.0

FN:FP = 20:1
  Selected validation threshold: 0.032698
  Validation FNR: 0.027788
  Validation review rate: 0.866406
  Test FNR: 0.048963
  Test review rate: 0.850553
  Test precision: 0.163733
  Test cost: 35166.0
  Test no-review cost: 120500.0
  Test net 

In [35]:
# ============================================================
# CELL 31A — FINAL BLOCK, PART 1
# Stress-definition provenance audit
#
# Purpose:
#   Resolve the manuscript claim:
#       "stress = national unemployment >= 4.90%"
#   and determine what actually produces the historical
#   stress sample of approximately n = 641.
#
# NO MODEL TRAINING.
# NO NEW STRESS DEFINITION IS INVENTED HERE.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# A. Test cohort used in revised analysis
# ------------------------------------------------------------

test_mask = (
    (df_model["split_revised"] == "test")
    &
    (df_model["mature_reconstructed"])
)

test_idx = df_model.index[test_mask]

print("=" * 110)
print("A. FIXED MATURE TEST COHORT")
print("=" * 110)

print("Test N:", f"{len(test_idx):,}")

print(
    "Expected:",
    "41,145",
    "|",
    "MATCH" if len(test_idx) == 41145 else "DOES NOT MATCH"
)


# ------------------------------------------------------------
# B. Read only stress-relevant raw variables
# ------------------------------------------------------------

stress_cols = [
    "issue_d",
    "issue_date",
    "addr_state",
    "state",
    "region",
    "subregion",
    "unemployment",
    "unem_rate",
    "logunemployment",
    "FEDFUNDS",
    "earnings",
    "employment",
    "labor_force",
    "gdpindex",
    "gdpreal",
    "gdppercap",
    "cpi",
    "CPIUS",
    "inf",
    "infUS"
]

stress_raw = pd.read_stata(
    DTA_PATH,
    columns=stress_cols,
    convert_categoricals=False
)

stress_test = stress_raw.loc[
    test_idx
].copy()

print("\nStress-variable frame:",
      stress_test.shape)


# ------------------------------------------------------------
# C. Basic distributions
# ------------------------------------------------------------

audit_vars = [
    "unemployment",
    "unem_rate",
    "logunemployment",
    "FEDFUNDS"
]

audit_rows = []

for c in audit_vars:

    s = pd.to_numeric(
        stress_test[c],
        errors="coerce"
    )

    audit_rows.append({
        "variable": c,
        "n": s.notna().sum(),
        "missing_pct":
            100 * s.isna().mean(),
        "min": s.min(),
        "p01": s.quantile(.01),
        "p25": s.quantile(.25),
        "median": s.quantile(.50),
        "p75": s.quantile(.75),
        "p90": s.quantile(.90),
        "p95": s.quantile(.95),
        "p99": s.quantile(.99),
        "max": s.max(),
        "nunique": s.nunique()
    })

stress_distribution = pd.DataFrame(
    audit_rows
)

print("\n" + "=" * 110)
print("B. STRESS-VARIABLE DISTRIBUTIONS — MATURE TEST")
print("=" * 110)

print(
    stress_distribution.to_string(
        index=False,
        formatters={
            c: "{:.6f}".format
            for c in [
                "missing_pct",
                "min",
                "p01",
                "p25",
                "median",
                "p75",
                "p90",
                "p95",
                "p99",
                "max"
            ]
        }
    )
)


# ------------------------------------------------------------
# D. Unique unemployment values and counts
# ------------------------------------------------------------

for c in [
    "unem_rate",
    "unemployment"
]:

    print("\n" + "=" * 110)
    print(
        f"C. MOST COMMON VALUES: {c}"
    )
    print("=" * 110)

    tmp = (
        stress_test[c]
        .value_counts(
            dropna=False
        )
        .head(30)
    )

    print(tmp.to_string())


# ------------------------------------------------------------
# E. Monthly structure
# ------------------------------------------------------------

stress_test[
    "issue_month_clean"
] = pd.to_datetime(
    stress_test["issue_d"]
        .astype(str)
        .str.strip(),
    format="%b-%y",
    errors="coerce"
)

monthly = (
    stress_test
    .groupby(
        "issue_month_clean",
        dropna=False
    )
    .agg(
        n=("issue_d", "size"),

        unemployment_min=(
            "unemployment",
            "min"
        ),
        unemployment_median=(
            "unemployment",
            "median"
        ),
        unemployment_max=(
            "unemployment",
            "max"
        ),

        unem_rate_min=(
            "unem_rate",
            "min"
        ),
        unem_rate_median=(
            "unem_rate",
            "median"
        ),
        unem_rate_max=(
            "unem_rate",
            "max"
        )
    )
    .reset_index()
)


print("\n" + "=" * 110)
print("D. MONTHLY UNEMPLOYMENT STRUCTURE")
print("=" * 110)

print(
    monthly.to_string(
        index=False,
        formatters={
            "unemployment_min":
                "{:.6f}".format,
            "unemployment_median":
                "{:.6f}".format,
            "unemployment_max":
                "{:.6f}".format,
            "unem_rate_min":
                "{:.6f}".format,
            "unem_rate_median":
                "{:.6f}".format,
            "unem_rate_max":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# F. Explicit 4.90% threshold audit
#
# Test both possible scales:
#   >= 4.90
#   >= 0.049
#
# We do NOT assume which scale is correct.
# ------------------------------------------------------------

threshold_rows = []

for variable in [
    "unemployment",
    "unem_rate"
]:

    s = pd.to_numeric(
        stress_test[variable],
        errors="coerce"
    )

    for threshold in [
        4.90,
        0.049
    ]:

        mask = (
            s >= threshold
        )

        threshold_rows.append({
            "variable":
                variable,
            "threshold":
                threshold,
            "stress_n":
                int(mask.sum()),
            "stress_pct":
                100 * mask.mean()
        })


threshold_audit = pd.DataFrame(
    threshold_rows
)


print("\n" + "=" * 110)
print("E. EXPLICIT 4.90% THRESHOLD AUDIT")
print("=" * 110)

print(
    threshold_audit.to_string(
        index=False,
        formatters={
            "threshold":
                "{:.6f}".format,
            "stress_pct":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# G. Search thresholds that produce counts near historical 641
#
# This is DIAGNOSTIC ONLY.
# It does not define stress.
# ------------------------------------------------------------

target_n = 641

near_rows = []

for variable in [
    "unemployment",
    "unem_rate"
]:

    s = pd.to_numeric(
        stress_test[variable],
        errors="coerce"
    ).dropna()

    unique_vals = np.sort(
        s.unique()
    )

    for threshold in unique_vals:

        n_ge = int(
            (s >= threshold).sum()
        )

        if abs(
            n_ge - target_n
        ) <= 100:

            near_rows.append({
                "variable":
                    variable,
                "threshold":
                    threshold,
                "n_ge_threshold":
                    n_ge,
                "difference_from_641":
                    n_ge - target_n
            })


near_641 = pd.DataFrame(
    near_rows
)

if len(near_641):

    near_641 = (
        near_641
        .sort_values(
            [
                "difference_from_641",
                "variable"
            ],
            key=lambda x:
                np.abs(x)
                if x.name == "difference_from_641"
                else x
        )
    )


print("\n" + "=" * 110)
print("F. THRESHOLDS PRODUCING COUNTS NEAR n=641")
print("=" * 110)

if len(near_641):

    print(
        near_641.to_string(
            index=False,
            formatters={
                "threshold":
                    "{:.8f}".format
            }
        )
    )

else:

    print(
        "No unemployment/unem_rate threshold "
        "produces a mature-test count within ±100 of 641."
    )


# ------------------------------------------------------------
# H. Quantile-based diagnostic
#
# Old notebook reportedly used approximately the upper quartile
# of unem_rate in at least one exploratory pipeline.
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("G. QUANTILE DIAGNOSTIC")
print("=" * 110)

for variable in [
    "unemployment",
    "unem_rate"
]:

    s = pd.to_numeric(
        stress_test[variable],
        errors="coerce"
    )

    for q in [
        .75,
        .90,
        .95,
        .975,
        .99
    ]:

        threshold = s.quantile(q)

        n_ge = int(
            (s >= threshold).sum()
        )

        print(
            f"{variable:15s} "
            f"q={q:.3f} "
            f"threshold={threshold:.8f} "
            f"n>={n_ge:,}"
        )


# ------------------------------------------------------------
# I. Relationship between unemployment variables
# ------------------------------------------------------------

corr_cols = [
    "unemployment",
    "unem_rate",
    "logunemployment"
]

corr_df = (
    stress_test[corr_cols]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .corr()
)


print("\n" + "=" * 110)
print("H. CORRELATION BETWEEN UNEMPLOYMENT VARIABLES")
print("=" * 110)

print(
    corr_df.to_string(
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# J. State/geographic variation diagnostic
# ------------------------------------------------------------

geo_summary = (
    stress_test
    .groupby(
        "addr_state",
        dropna=False
    )
    .agg(
        n=("issue_d", "size"),
        unemployment_mean=(
            "unemployment",
            "mean"
        ),
        unemployment_min=(
            "unemployment",
            "min"
        ),
        unemployment_max=(
            "unemployment",
            "max"
        ),
        unem_rate_mean=(
            "unem_rate",
            "mean"
        ),
        unem_rate_min=(
            "unem_rate",
            "min"
        ),
        unem_rate_max=(
            "unem_rate",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "n",
        ascending=False
    )
)


print("\n" + "=" * 110)
print("I. GEOGRAPHIC VARIATION — FIRST 25 STATES")
print("=" * 110)

print(
    geo_summary
    .head(25)
    .to_string(
        index=False,
        formatters={
            "unemployment_mean":
                "{:.6f}".format,
            "unemployment_min":
                "{:.6f}".format,
            "unemployment_max":
                "{:.6f}".format,
            "unem_rate_mean":
                "{:.6f}".format,
            "unem_rate_min":
                "{:.6f}".format,
            "unem_rate_max":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# K. Save outputs
# ------------------------------------------------------------

stress_distribution.to_csv(
    OUTPUT_DIR /
    "cell31A_stress_variable_distributions.csv",
    index=False
)

monthly.to_csv(
    OUTPUT_DIR /
    "cell31A_monthly_stress_structure.csv",
    index=False
)

threshold_audit.to_csv(
    OUTPUT_DIR /
    "cell31A_490_threshold_audit.csv",
    index=False
)

near_641.to_csv(
    OUTPUT_DIR /
    "cell31A_thresholds_near_641.csv",
    index=False
)

corr_df.to_csv(
    OUTPUT_DIR /
    "cell31A_unemployment_correlations.csv"
)

geo_summary.to_csv(
    OUTPUT_DIR /
    "cell31A_geographic_stress_summary.csv",
    index=False
)


print("\nSaved Cell 31A outputs to:")
print(OUTPUT_DIR)

print(
    "\nIMPORTANT: Do not redefine stress yet."
)

print(
    "Paste this complete output before running the final "
    "stress/slice analysis."
)

A. FIXED MATURE TEST COHORT
Test N: 41,145
Expected: 41,145 | MATCH

Stress-variable frame: (41145, 20)

B. STRESS-VARIABLE DISTRIBUTIONS — MATURE TEST
       variable     n missing_pct         min          p01           p25        median           p75           p90           p95           p99           max  nunique
   unemployment 41145    0.000000 9782.000000 14191.000000 106230.000000 210639.000000 430440.000000 814209.000000 814209.000000 814209.000000 814209.000000       50
      unem_rate 41145    0.000000    0.021000     0.027000      0.038000      0.042000      0.043000      0.047000      0.047000      0.053000      0.071000       25
logunemployment 41145    0.000000    9.188299     9.560363     11.573361     12.257901     12.972563     13.609972     13.609972     13.609972     13.609972       50
       FEDFUNDS 41145    0.000000    1.820000     1.820000      1.820000      1.820000      1.820000      1.820000      1.820000      1.820000      1.910000        2

C. MOST COMMON VA

In [36]:
# ============================================================
# CELL 31B — FINAL CODE CELL OF REVISION ANALYSIS
#
# Final stress + slice robustness analysis
#
# VERIFIED STRESS VARIABLE:
#   unem_rate is state-level, not national.
#
# High-unemployment subgroup:
#   unem_rate >= 0.049  (4.9%)
#
# IMPORTANT:
#   This is interpreted as a high-state-unemployment subgroup,
#   NOT as a national macroeconomic stress period.
#
# Threshold analysis:
#   Uses the validation-selected recalibrated 10:1 threshold
#   from Cell 30.
#
# Slice support rule:
#   >=10 defaults AND >=10 non-defaults in BOTH unemployment
#   regimes within the same grade x term cell.
#
# NO MODEL TRAINING.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss
)

from statsmodels.stats.proportion import proportion_confint


# ------------------------------------------------------------
# A. Load fixed test labels and recalibrated probabilities
# ------------------------------------------------------------

y_test = np.load(
    OUTPUT_DIR / "cell27_y_test.npy"
).astype(int)

p_test = np.load(
    OUTPUT_DIR /
    "cell29_histgb_full_test_recalibrated.npy"
)

assert len(y_test) == 41145
assert len(p_test) == 41145


# ------------------------------------------------------------
# B. Reconstruct fixed test metadata in identical row order
# ------------------------------------------------------------

test_meta = stress_test.copy()

assert len(test_meta) == len(y_test)

test_meta["y"] = y_test
test_meta["p"] = p_test

test_meta["unem_rate_num"] = pd.to_numeric(
    test_meta["unem_rate"],
    errors="coerce"
)

test_meta["high_unemployment"] = (
    test_meta["unem_rate_num"] >= 0.049
)


print("=" * 110)
print("A. VERIFIED HIGH-UNEMPLOYMENT SUBGROUP")
print("=" * 110)

print(
    "Definition: state-level unemployment rate >= 4.9%"
)

print(
    "High-unemployment N:",
    f"{test_meta['high_unemployment'].sum():,}"
)

print(
    "Other-state N:",
    f"{(~test_meta['high_unemployment']).sum():,}"
)

print(
    "High-unemployment share:",
    f"{test_meta['high_unemployment'].mean():.6f}"
)


# ------------------------------------------------------------
# C. Group-level descriptive metrics
# ------------------------------------------------------------

def ece_equal_width_local(y, p, n_bins=10):

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ids = np.digitize(
        p,
        edges[1:-1],
        right=False
    )

    ece = 0.0

    for b in range(n_bins):

        mask = ids == b

        if mask.sum() == 0:
            continue

        ece += (
            mask.mean()
            *
            abs(
                y[mask].mean()
                -
                p[mask].mean()
            )
        )

    return float(ece)


def group_metrics(frame):

    y = frame["y"].to_numpy()
    p = frame["p"].to_numpy()

    result = {
        "n": len(frame),
        "defaults": int(y.sum()),
        "nondefaults": int(len(y) - y.sum()),
        "default_rate": float(y.mean()),
        "mean_prediction": float(p.mean()),
        "brier": float(
            brier_score_loss(y, p)
        ),
        "logloss": float(
            log_loss(
                y,
                p,
                labels=[0, 1]
            )
        ),
        "ece": ece_equal_width_local(
            y,
            p
        )
    }

    if len(np.unique(y)) == 2:
        result["auc"] = float(
            roc_auc_score(y, p)
        )
    else:
        result["auc"] = np.nan

    return result


group_rows = []

for name, mask in [
    (
        "Other-state",
        ~test_meta["high_unemployment"]
    ),
    (
        "High-unemployment",
        test_meta["high_unemployment"]
    )
]:

    metrics = group_metrics(
        test_meta.loc[mask]
    )

    group_rows.append({
        "group": name,
        **metrics
    })


group_table = pd.DataFrame(
    group_rows
)


print("\n" + "=" * 110)
print("B. GROUP PERFORMANCE")
print("=" * 110)

print(
    group_table.to_string(
        index=False,
        formatters={
            "default_rate":
                "{:.6f}".format,
            "mean_prediction":
                "{:.6f}".format,
            "auc":
                "{:.6f}".format,
            "brier":
                "{:.6f}".format,
            "logloss":
                "{:.6f}".format,
            "ece":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# D. Stratified bootstrap CIs within each group
# ------------------------------------------------------------

BOOT = 2000
BOOT_SEED = 2026

rng = np.random.default_rng(
    BOOT_SEED
)


def bootstrap_group(frame, n_boot=2000):

    y = frame["y"].to_numpy()
    p = frame["p"].to_numpy()

    pos = np.flatnonzero(
        y == 1
    )

    neg = np.flatnonzero(
        y == 0
    )

    rows = []

    for b in range(n_boot):

        sampled_pos = rng.choice(
            pos,
            size=len(pos),
            replace=True
        )

        sampled_neg = rng.choice(
            neg,
            size=len(neg),
            replace=True
        )

        idx = np.concatenate(
            [
                sampled_pos,
                sampled_neg
            ]
        )

        yb = y[idx]
        pb = p[idx]

        rows.append([
            roc_auc_score(
                yb,
                pb
            ),
            brier_score_loss(
                yb,
                pb
            ),
            log_loss(
                yb,
                pb,
                labels=[0, 1]
            ),
            ece_equal_width_local(
                yb,
                pb
            )
        ])

    arr = np.asarray(
        rows
    )

    return {
        "auc_ci_low":
            np.quantile(arr[:, 0], .025),
        "auc_ci_high":
            np.quantile(arr[:, 0], .975),

        "brier_ci_low":
            np.quantile(arr[:, 1], .025),
        "brier_ci_high":
            np.quantile(arr[:, 1], .975),

        "logloss_ci_low":
            np.quantile(arr[:, 2], .025),
        "logloss_ci_high":
            np.quantile(arr[:, 2], .975),

        "ece_ci_low":
            np.quantile(arr[:, 3], .025),
        "ece_ci_high":
            np.quantile(arr[:, 3], .975)
    }


ci_rows = []

for name, mask in [
    (
        "Other-state",
        ~test_meta["high_unemployment"]
    ),
    (
        "High-unemployment",
        test_meta["high_unemployment"]
    )
]:

    print(
        f"\nBootstrapping {name}..."
    )

    ci = bootstrap_group(
        test_meta.loc[mask],
        n_boot=BOOT
    )

    ci_rows.append({
        "group": name,
        **ci
    })


ci_table = pd.DataFrame(
    ci_rows
)

group_table_ci = (
    group_table
    .merge(
        ci_table,
        on="group",
        how="left"
    )
)


print("\n" + "=" * 110)
print("C. GROUP PERFORMANCE WITH 95% BOOTSTRAP CIs")
print("=" * 110)

print(
    group_table_ci.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# E. Use validation-selected 10:1 threshold from Cell 30
# ------------------------------------------------------------

threshold_file = pd.read_csv(
    OUTPUT_DIR /
    "cell30_recalibrated_threshold_analysis.csv"
)

threshold_10 = float(
    threshold_file.loc[
        (
            threshold_file["fn_fp_ratio"]
            == "10:1"
        )
        &
        (
            threshold_file["cohort"]
            == "Test"
        ),
        "threshold"
    ].iloc[0]
)


print("\n" + "=" * 110)
print("D. FIXED DECISION THRESHOLD")
print("=" * 110)

print(
    "10:1 validation-selected threshold:",
    f"{threshold_10:.6f}"
)


# ------------------------------------------------------------
# F. Decision metrics by unemployment group
# ------------------------------------------------------------

def threshold_metrics(
    frame,
    threshold
):

    y = frame["y"].to_numpy()
    p = frame["p"].to_numpy()

    flag = (
        p >= threshold
    )

    tp = int(
        np.sum(
            flag & (y == 1)
        )
    )

    fp = int(
        np.sum(
            flag & (y == 0)
        )
    )

    tn = int(
        np.sum(
            (~flag) & (y == 0)
        )
    )

    fn = int(
        np.sum(
            (~flag) & (y == 1)
        )
    )

    defaults = (
        tp + fn
    )

    flagged = (
        tp + fp
    )

    fnr = (
        fn / defaults
        if defaults
        else np.nan
    )

    precision = (
        tp / flagged
        if flagged
        else np.nan
    )

    review_rate = (
        flagged / len(frame)
    )

    # Wilson CI for FNR
    if defaults > 0:

        fnr_low, fnr_high = (
            proportion_confint(
                fn,
                defaults,
                alpha=.05,
                method="wilson"
            )
        )

    else:

        fnr_low = np.nan
        fnr_high = np.nan

    return {
        "n": len(frame),
        "defaults": defaults,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "fnr": fnr,
        "fnr_ci_low": fnr_low,
        "fnr_ci_high": fnr_high,
        "review_rate": review_rate,
        "precision": precision
    }


decision_rows = []

for name, mask in [
    (
        "Other-state",
        ~test_meta["high_unemployment"]
    ),
    (
        "High-unemployment",
        test_meta["high_unemployment"]
    )
]:

    m = threshold_metrics(
        test_meta.loc[mask],
        threshold_10
    )

    decision_rows.append({
        "group": name,
        **m
    })


decision_table = pd.DataFrame(
    decision_rows
)


print("\n" + "=" * 110)
print("E. 10:1 THRESHOLD PERFORMANCE BY UNEMPLOYMENT GROUP")
print("=" * 110)

print(
    decision_table.to_string(
        index=False,
        formatters={
            "fnr":
                "{:.6f}".format,
            "fnr_ci_low":
                "{:.6f}".format,
            "fnr_ci_high":
                "{:.6f}".format,
            "review_rate":
                "{:.6f}".format,
            "precision":
                "{:.6f}".format
        }
    )
)


# ------------------------------------------------------------
# G. Grade x term slice metadata
# ------------------------------------------------------------

# Need grade and term from original fixed test rows.
slice_raw = pd.read_stata(
    DTA_PATH,
    columns=[
        "grade",
        "term"
    ],
    convert_categoricals=False
)

test_meta[
    "grade_slice"
] = (
    slice_raw.loc[
        test_idx,
        "grade"
    ]
    .astype(str)
    .str.strip()
    .to_numpy()
)

test_meta[
    "term_slice"
] = (
    slice_raw.loc[
        test_idx,
        "term"
    ]
    .astype(str)
    .str.strip()
    .to_numpy()
)


# ------------------------------------------------------------
# H. Cell support audit
# ------------------------------------------------------------

support_rows = []

for (
    grade,
    term
), cell in test_meta.groupby(
    [
        "grade_slice",
        "term_slice"
    ]
):

    for regime_name, regime_mask in [
        (
            "Other-state",
            ~cell["high_unemployment"]
        ),
        (
            "High-unemployment",
            cell["high_unemployment"]
        )
    ]:

        sub = cell.loc[
            regime_mask
        ]

        defaults = int(
            sub["y"].sum()
        )

        nondefaults = int(
            len(sub)
            -
            defaults
        )

        support_rows.append({
            "grade":
                grade,
            "term":
                term,
            "regime":
                regime_name,
            "n":
                len(sub),
            "defaults":
                defaults,
            "nondefaults":
                nondefaults
        })


support_table = pd.DataFrame(
    support_rows
)


# ------------------------------------------------------------
# I. Apply strengthened support rule
#
# Require in BOTH regimes:
#   >=10 defaults
#   >=10 non-defaults
# ------------------------------------------------------------

support_pivot = (
    support_table
    .pivot_table(
        index=[
            "grade",
            "term"
        ],
        columns="regime",
        values=[
            "n",
            "defaults",
            "nondefaults"
        ],
        aggfunc="first",
        fill_value=0
    )
)


eligible_cells = []

for idx, row in support_pivot.iterrows():

    try:

        ok = (
            row[
                ("defaults",
                 "High-unemployment")
            ] >= 10
            and
            row[
                ("nondefaults",
                 "High-unemployment")
            ] >= 10
            and
            row[
                ("defaults",
                 "Other-state")
            ] >= 10
            and
            row[
                ("nondefaults",
                 "Other-state")
            ] >= 10
        )

    except KeyError:

        ok = False

    if ok:
        eligible_cells.append(
            idx
        )


print("\n" + "=" * 110)
print("F. STRENGTHENED SLICE SUPPORT AUDIT")
print("=" * 110)

print(
    "Rule: >=10 defaults and >=10 non-defaults "
    "in BOTH unemployment regimes."
)

print(
    "Eligible grade x term cells:",
    len(eligible_cells)
)

if eligible_cells:
    print(
        "Eligible cells:",
        eligible_cells
    )
else:
    print(
        "No grade x term cell satisfies the strengthened "
        "minimum-event rule."
    )


# ------------------------------------------------------------
# J. Supported slice-level threshold diagnostics
# ------------------------------------------------------------

slice_rows = []

for grade, term in eligible_cells:

    cell = test_meta.loc[
        (
            test_meta["grade_slice"]
            == grade
        )
        &
        (
            test_meta["term_slice"]
            == term
        )
    ]

    for regime_name, regime_mask in [
        (
            "Other-state",
            ~cell["high_unemployment"]
        ),
        (
            "High-unemployment",
            cell["high_unemployment"]
        )
    ]:

        sub = cell.loc[
            regime_mask
        ]

        metrics = threshold_metrics(
            sub,
            threshold_10
        )

        slice_rows.append({
            "grade":
                grade,
            "term":
                term,
            "regime":
                regime_name,
            **metrics
        })


slice_results = pd.DataFrame(
    slice_rows
)


print("\n" + "=" * 110)
print("G. SUPPORTED GRADE x TERM SLICE RESULTS")
print("=" * 110)

if len(slice_results):

    print(
        slice_results.to_string(
            index=False,
            formatters={
                "fnr":
                    "{:.6f}".format,
                "fnr_ci_low":
                    "{:.6f}".format,
                "fnr_ci_high":
                    "{:.6f}".format,
                "review_rate":
                    "{:.6f}".format,
                "precision":
                    "{:.6f}".format
            }
        )
    )

else:

    print(
        "No supported slice-level estimates remain "
        "under the strengthened event-count rule."
    )


# ------------------------------------------------------------
# K. Diagnostic: show all high-unemployment cell counts
#
# This allows us to explain why cells were excluded.
# ------------------------------------------------------------

high_support = (
    support_table[
        support_table["regime"]
        == "High-unemployment"
    ]
    .sort_values(
        [
            "defaults",
            "n"
        ],
        ascending=False
    )
)


print("\n" + "=" * 110)
print("H. HIGH-UNEMPLOYMENT SUPPORT BY GRADE x TERM")
print("=" * 110)

print(
    high_support.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# L. Save final reviewer-facing outputs
# ------------------------------------------------------------

group_table_ci.to_csv(
    OUTPUT_DIR /
    "cell31B_unemployment_group_performance.csv",
    index=False
)

decision_table.to_csv(
    OUTPUT_DIR /
    "cell31B_unemployment_threshold_performance.csv",
    index=False
)

support_table.to_csv(
    OUTPUT_DIR /
    "cell31B_slice_support_counts.csv",
    index=False
)

slice_results.to_csv(
    OUTPUT_DIR /
    "cell31B_supported_slice_results.csv",
    index=False
)

high_support.to_csv(
    OUTPUT_DIR /
    "cell31B_high_unemployment_slice_support.csv",
    index=False
)


print("\n" + "=" * 110)
print("FINAL")
print("=" * 110)

print(
    "Cell 31B complete."
)

print(
    "No further reviewer-required analysis is scheduled."
)

print(
    "Paste the complete output for final scientific inspection."
)

A. VERIFIED HIGH-UNEMPLOYMENT SUBGROUP
Definition: state-level unemployment rate >= 4.9%
High-unemployment N: 641
Other-state N: 40,504
High-unemployment share: 0.015579

B. GROUP PERFORMANCE
            group     n  defaults  nondefaults default_rate mean_prediction    brier  logloss      ece      auc
      Other-state 40504      5951        34553     0.146924        0.088364 0.123521 0.413233 0.058650 0.675232
High-unemployment   641        74          567     0.115445        0.077638 0.100383 0.352064 0.037806 0.636136

Bootstrapping Other-state...

Bootstrapping High-unemployment...

C. GROUP PERFORMANCE WITH 95% BOOTSTRAP CIs
            group     n  defaults  nondefaults  default_rate  mean_prediction    brier  logloss      ece      auc  auc_ci_low  auc_ci_high  brier_ci_low  brier_ci_high  logloss_ci_low  logloss_ci_high  ece_ci_low  ece_ci_high
      Other-state 40504      5951        34553      0.146924         0.088364 0.123521 0.413233 0.058650 0.675232    0.668787     0.682